# EXP004 Objective Evaluation

In [ ]:
# ======================================================================
# CELL 2 — EXP004 RUNTIME AND DEPENDENCY INITIALIZATION
# ======================================================================

import sys
import subprocess
import importlib
import platform
from pathlib import Path


print("=" * 70)
print("EXP004 RUNTIME AND DEPENDENCY INITIALIZATION")
print("=" * 70)


# ----------------------------------------------------------------------
# PYTHON / SYSTEM INFORMATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PYTHON / SYSTEM")
print("-" * 70)

print(f"Python version       : {sys.version}")
print(f"Python executable    : {sys.executable}")
print(f"Platform             : {platform.platform()}")


# ----------------------------------------------------------------------
# REQUIRED PACKAGE CONFIGURATION
# ----------------------------------------------------------------------
#
# These are the packages required by the finalized EXP004 objective
# evaluation pipeline.
#
# museval:
#   BSSEval objective metrics, including SDR, SIR and SAR.
#
# torchmetrics:
#   Scale-invariant SI-SDR.
#
# musdb:
#   MUSDB-compatible evaluation support retained from EXP002.
#
# The runtime already contains the remaining core scientific/audio
# packages, so we do not install unnecessary dependencies such as
# librosa or seaborn.
# ----------------------------------------------------------------------

REQUIRED_PACKAGES = {
    "museval": "museval",
    "musdb": "musdb",
    "torchmetrics": "torchmetrics",
}


# ----------------------------------------------------------------------
# PACKAGE CHECK
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("DEPENDENCY CHECK")
print("-" * 70)

missing_packages = []

for module_name, package_name in REQUIRED_PACKAGES.items():

    try:
        module = importlib.import_module(module_name)

        version = getattr(
            module,
            "__version__",
            "version unavailable"
        )

        print(
            f"✓ {package_name:<15}: "
            f"{version}"
        )

    except Exception as exc:

        print(
            f"✗ {package_name:<15}: "
            f"NOT AVAILABLE"
        )

        print(
            f"  Reason: {exc}"
        )

        missing_packages.append(package_name)


# ----------------------------------------------------------------------
# INSTALL ONLY MISSING PACKAGES
# ----------------------------------------------------------------------

if missing_packages:

    print("\n" + "-" * 70)
    print("INSTALLING MISSING DEPENDENCIES")
    print("-" * 70)

    print(
        "The following packages are missing:"
    )

    for package in missing_packages:
        print(f"  - {package}")

    print("\nInstalling...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            *missing_packages,
        ]
    )

    print("\n✓ Missing dependency installation completed.")

else:

    print("\n✓ All required objective-evaluation dependencies are already installed.")


# ----------------------------------------------------------------------
# RE-IMPORT / POST-INSTALL VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("POST-INSTALL DEPENDENCY VALIDATION")
print("-" * 70)

for module_name, package_name in REQUIRED_PACKAGES.items():

    try:

        module = importlib.import_module(module_name)

        version = getattr(
            module,
            "__version__",
            "version unavailable"
        )

        print(
            f"✓ {package_name:<15}: "
            f"{version}"
        )

    except Exception as exc:

        raise RuntimeError(
            f"Required package '{package_name}' "
            f"could not be imported after installation.\n"
            f"Error: {exc}"
        )


# ----------------------------------------------------------------------
# CORE RUNTIME VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("CORE RUNTIME VALIDATION")
print("-" * 70)

import torch
import numpy as np
import pandas as pd
import scipy
import soundfile as sf
import matplotlib


print(f"✓ PyTorch       : {torch.__version__}")
print(f"✓ NumPy         : {np.__version__}")
print(f"✓ Pandas        : {pd.__version__}")
print(f"✓ SciPy         : {scipy.__version__}")
print(f"✓ SoundFile     : {sf.__version__}")
print(f"✓ Matplotlib    : {matplotlib.__version__}")


# ----------------------------------------------------------------------
# CUDA / GPU VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PYTORCH / CUDA / GPU")
print("-" * 70)

print(f"PyTorch version      : {torch.__version__}")
print(f"CUDA available       : {torch.cuda.is_available()}")
print(f"CUDA version         : {torch.version.cuda}")

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA is not available. "
        "EXP004 requires the configured local CUDA runtime."
    )


GPU_COUNT = torch.cuda.device_count()

print(f"GPU count            : {GPU_COUNT}")

if GPU_COUNT < 1:

    raise RuntimeError(
        "No CUDA GPU detected."
    )


for gpu_index in range(GPU_COUNT):

    properties = torch.cuda.get_device_properties(
        gpu_index
    )

    vram_gb = (
        properties.total_memory
        / (1024 ** 3)
    )

    print(f"\nGPU {gpu_index}")
    print(
        f"  Name               : "
        f"{torch.cuda.get_device_name(gpu_index)}"
    )
    print(
        f"  VRAM               : "
        f"{vram_gb:.2f} GB"
    )
    print(
        f"  Compute capability : "
        f"{properties.major}.{properties.minor}"
    )


ACTIVE_GPU = torch.cuda.current_device()

print(
    f"\nActive GPU           : "
    f"{torch.cuda.get_device_name(ACTIVE_GPU)}"
)

print(
    f"Allocated VRAM       : "
    f"{torch.cuda.memory_allocated(ACTIVE_GPU) / (1024 ** 3):.3f} GB"
)

print(
    f"Reserved VRAM        : "
    f"{torch.cuda.memory_reserved(ACTIVE_GPU) / (1024 ** 3):.3f} GB"
)


# ----------------------------------------------------------------------
# FFMPEG VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("FFMPEG")
print("-" * 70)

ffmpeg_result = subprocess.run(
    ["ffmpeg", "-version"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    check=False,
)

if ffmpeg_result.returncode != 0:

    raise RuntimeError(
        "FFmpeg is not accessible from the local runtime."
    )


ffmpeg_first_line = (
    ffmpeg_result.stdout
    .splitlines()[0]
    if ffmpeg_result.stdout
    else "version unavailable"
)

print(f"✓ FFmpeg found")
print(f"  {ffmpeg_first_line}")


# ----------------------------------------------------------------------
# METRIC API VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("OBJECTIVE METRIC API VALIDATION")
print("-" * 70)

import museval
import musdb
import torchmetrics


print(
    f"✓ museval       : "
    f"{getattr(museval, '__version__', 'version unavailable')}"
)

print(
    f"✓ musdb         : "
    f"{getattr(musdb, '__version__', 'version unavailable')}"
)

print(
    f"✓ torchmetrics  : "
    f"{getattr(torchmetrics, '__version__', 'version unavailable')}"
)


if not hasattr(museval, "metrics"):

    raise RuntimeError(
        "museval.metrics is unavailable."
    )


if not hasattr(museval.metrics, "bss_eval"):

    raise RuntimeError(
        "museval.metrics.bss_eval is unavailable."
    )


try:

    from torchmetrics.audio import (
        ScaleInvariantSignalDistortionRatio
    )

except Exception as exc:

    raise RuntimeError(
        "Required torchmetrics SI-SDR implementation "
        f"is unavailable: {exc}"
    )


print(
    "✓ museval.metrics.bss_eval available"
)

print(
    "✓ torchmetrics "
    "ScaleInvariantSignalDistortionRatio available"
)


# ----------------------------------------------------------------------
# PERSISTENT RUNTIME STATE
# ----------------------------------------------------------------------
#
# These variables are intentionally established here and will be reused
# by subsequent cells. Later cells should not silently redefine them.
# ----------------------------------------------------------------------

EXPERIMENT = "EXP004"

MODEL_NAME = "htdemucs"

OBJECTIVE_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]

EXCLUDED_METRICS = [
    "ISR",
]

EXPECTED_SOURCES = [
    "vocals",
    "drums",
    "bass",
    "other",
]

EXPECTED_TRACK_COUNT = 50
EXPECTED_SOURCE_COUNT = 4
EXPECTED_EVALUATION_PAIRS = (
    EXPECTED_TRACK_COUNT
    * EXPECTED_SOURCE_COUNT
)

CHUNK_DURATION = 60
CHUNK_OVERLAP = 5
CHUNK_STEP = 55

print("\n" + "-" * 70)
print("PERSISTENT EXP004 CONFIGURATION")
print("-" * 70)

print(f"Experiment             : {EXPERIMENT}")
print(f"HT-Demucs model        : {MODEL_NAME}")

print(
    f"Physical chunk duration: "
    f"{CHUNK_DURATION} s"
)

print(
    f"Physical overlap       : "
    f"{CHUNK_OVERLAP} s"
)

print(
    f"Physical step          : "
    f"{CHUNK_STEP} s"
)

print(
    f"Expected tracks        : "
    f"{EXPECTED_TRACK_COUNT}"
)

print(
    f"Sources per track      : "
    f"{EXPECTED_SOURCE_COUNT}"
)

print(
    f"Expected source pairs  : "
    f"{EXPECTED_EVALUATION_PAIRS}"
)

print(
    "Approved metrics       : "
    + ", ".join(OBJECTIVE_METRICS)
)

print(
    "Excluded metrics       : "
    + ", ".join(EXCLUDED_METRICS)
)


# ----------------------------------------------------------------------
# FINAL CELL RESULT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 2 RESULT")
print("=" * 70)

print("✓ Required dependencies available")
print("✓ CUDA runtime available")
print("✓ Local RTX 3050 detected")
print("✓ FFmpeg available")
print("✓ BSSEval API available")
print("✓ SI-SDR API available")
print("✓ EXP004 persistent configuration initialized")

print("\n" + "=" * 70)
print("✓ CELL 2 COMPLETE")
print("=" * 70)

EXP004 RUNTIME AND DEPENDENCY INITIALIZATION

----------------------------------------------------------------------
PYTHON / SYSTEM
----------------------------------------------------------------------
Python version       : 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Python executable    : C:\IKS_Local_Runtime\iks_eval\Scripts\python.exe
Platform             : Windows-11-10.0.26200-SP0

----------------------------------------------------------------------
DEPENDENCY CHECK
----------------------------------------------------------------------
✓ museval        : version unavailable
✓ musdb          : version unavailable
✓ torchmetrics   : 1.9.0

✓ All required objective-evaluation dependencies are already installed.

----------------------------------------------------------------------
POST-INSTALL DEPENDENCY VALIDATION
----------------------------------------------------------------------
✓ museval        : version unavailable
✓ musdb         

# Path Configuration

In [ ]:
# ======================================================================
# CELL 4 — EXP004 PATH CONFIGURATION
# ======================================================================

from pathlib import Path


print("=" * 70)
print("EXP004 PATH CONFIGURATION")
print("=" * 70)


# ----------------------------------------------------------------------
# PROJECT ROOT
# ----------------------------------------------------------------------

PROJECT_ROOT = Path(
    r"G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation"
)

print("\n" + "-" * 70)
print("PROJECT ROOT")
print("-" * 70)

print(f"Project root: {PROJECT_ROOT}")


# ----------------------------------------------------------------------
# MUSDB18 TEST DATASET
# ----------------------------------------------------------------------

MUSDB18_TEST_ROOT = (
    PROJECT_ROOT
    / "03_Datasets"
    / "Benchmarks"
    / "MUSDB18"
    / "test"
)

print("\n" + "-" * 70)
print("MUSDB18 TEST DATASET")
print("-" * 70)

print(f"Test dataset: {MUSDB18_TEST_ROOT}")


# ----------------------------------------------------------------------
# BASELINE EXP004 ROOT
# ----------------------------------------------------------------------

BASELINE_EXP004_ROOT = (
    PROJECT_ROOT
    / "05_Experiments"
    / "Baseline_HT_Demucs"
    / "EXP004"
)

BASELINE_EXP004_NOTEBOOK_DIR = (
    BASELINE_EXP004_ROOT
    / "notebook"
)

BASELINE_EXP004_OUTPUTS_DIR = (
    BASELINE_EXP004_ROOT
    / "outputs"
)

BASELINE_EXP004_LOGS_DIR = (
    BASELINE_EXP004_ROOT
    / "logs"
)

BASELINE_EXP004_CHUNKED_DIR = (
    BASELINE_EXP004_ROOT
    / "chunked_processing"
)


print("\n" + "-" * 70)
print("BASELINE EXP004")
print("-" * 70)

print(f"Baseline root       : {BASELINE_EXP004_ROOT}")
print(f"Baseline notebook   : {BASELINE_EXP004_NOTEBOOK_DIR}")
print(f"Baseline outputs    : {BASELINE_EXP004_OUTPUTS_DIR}")
print(f"Baseline logs       : {BASELINE_EXP004_LOGS_DIR}")
print(f"Baseline chunked    : {BASELINE_EXP004_CHUNKED_DIR}")


# ----------------------------------------------------------------------
# OBJECTIVE EXP004 ROOT
# ----------------------------------------------------------------------

OBJECTIVE_EXP004_ROOT = (
    PROJECT_ROOT
    / "05_Experiments"
    / "Evaluation"
    / "Objective"
    / "EXP004"
)

OBJECTIVE_EXP004_LOGS_DIR = (
    OBJECTIVE_EXP004_ROOT
    / "logs"
)

OBJECTIVE_EXP004_METRICS_DIR = (
    OBJECTIVE_EXP004_ROOT
    / "metrics"
)

OBJECTIVE_EXP004_NOTEBOOK_DIR = (
    OBJECTIVE_EXP004_ROOT
    / "notebook"
)

OBJECTIVE_EXP004_PLOTS_DIR = (
    OBJECTIVE_EXP004_ROOT
    / "plots"
)

OBJECTIVE_EXP004_PREDICTIONS_DIR = (
    OBJECTIVE_EXP004_ROOT
    / "predictions"
)

OBJECTIVE_EXP004_REPORTS_DIR = (
    OBJECTIVE_EXP004_ROOT
    / "reports"
)


print("\n" + "-" * 70)
print("OBJECTIVE EXP004")
print("-" * 70)

print(f"Objective root       : {OBJECTIVE_EXP004_ROOT}")
print(f"Objective logs       : {OBJECTIVE_EXP004_LOGS_DIR}")
print(f"Objective metrics    : {OBJECTIVE_EXP004_METRICS_DIR}")
print(f"Objective notebook   : {OBJECTIVE_EXP004_NOTEBOOK_DIR}")
print(f"Objective plots      : {OBJECTIVE_EXP004_PLOTS_DIR}")
print(f"Objective predictions: {OBJECTIVE_EXP004_PREDICTIONS_DIR}")
print(f"Objective reports    : {OBJECTIVE_EXP004_REPORTS_DIR}")


# ----------------------------------------------------------------------
# PATH EXISTENCE VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("INPUT PATH VALIDATION")
print("-" * 70)


INPUT_PATHS = {
    "Project root": PROJECT_ROOT,
    "MUSDB18 test dataset": MUSDB18_TEST_ROOT,
    "Baseline EXP004 root": BASELINE_EXP004_ROOT,
    "Baseline chunked processing": BASELINE_EXP004_CHUNKED_DIR,
}


for name, path in INPUT_PATHS.items():

    if path.exists():

        print(f"✓ {name:<32}: {path}")

    else:

        raise FileNotFoundError(
            f"Required input path does not exist:\n"
            f"{name}: {path}"
        )


# ----------------------------------------------------------------------
# OBJECTIVE OUTPUT DIRECTORY CREATION
# ----------------------------------------------------------------------
#
# EXP004 objective evaluation has not produced anything yet.
# These directories are therefore initialized here.
#
# Creating the directories is safe and does not create analytical
# outputs, predictions, metrics, plots, or reports.
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("OBJECTIVE OUTPUT DIRECTORY INITIALIZATION")
print("-" * 70)


OBJECTIVE_OUTPUT_DIRS = {
    "Objective root": OBJECTIVE_EXP004_ROOT,
    "Logs": OBJECTIVE_EXP004_LOGS_DIR,
    "Metrics": OBJECTIVE_EXP004_METRICS_DIR,
    "Notebook": OBJECTIVE_EXP004_NOTEBOOK_DIR,
    "Plots": OBJECTIVE_EXP004_PLOTS_DIR,
    "Predictions": OBJECTIVE_EXP004_PREDICTIONS_DIR,
    "Reports": OBJECTIVE_EXP004_REPORTS_DIR,
}


for name, path in OBJECTIVE_OUTPUT_DIRS.items():

    path.mkdir(
        parents=True,
        exist_ok=True
    )

    print(f"✓ {name:<32}: {path}")


# ----------------------------------------------------------------------
# PATH SEPARATION SAFETY CHECK
# ----------------------------------------------------------------------
#
# Objective outputs must never resolve inside the baseline experiment.
# This check protects the already completed baseline experiment from
# accidental writes.
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("BASELINE / OBJECTIVE PATH SEPARATION")
print("-" * 70)


baseline_resolved = (
    BASELINE_EXP004_ROOT
    .resolve()
)

objective_resolved = (
    OBJECTIVE_EXP004_ROOT
    .resolve()
)


if (
    objective_resolved == baseline_resolved
    or baseline_resolved in objective_resolved.parents
    or objective_resolved in baseline_resolved.parents
):

    raise RuntimeError(
        "Baseline and Objective EXP004 paths overlap. "
        "Aborting to protect baseline outputs."
    )


print("✓ Baseline and Objective EXP004 paths are separate.")


# ----------------------------------------------------------------------
# CHUNKING CONFIGURATION PATH STATE
# ----------------------------------------------------------------------
#
# These variables mirror the established physical chunking architecture
# used by Baseline EXP004.
# ----------------------------------------------------------------------

CHUNK_DURATION_SECONDS = CHUNK_DURATION
CHUNK_OVERLAP_SECONDS = CHUNK_OVERLAP
CHUNK_STEP_SECONDS = CHUNK_STEP


print("\n" + "-" * 70)
print("PHYSICAL CHUNKING CONFIGURATION")
print("-" * 70)

print(
    f"Chunk duration : "
    f"{CHUNK_DURATION_SECONDS} seconds"
)

print(
    f"Overlap        : "
    f"{CHUNK_OVERLAP_SECONDS} seconds"
)

print(
    f"Step           : "
    f"{CHUNK_STEP_SECONDS} seconds"
)

if (
    CHUNK_DURATION_SECONDS
    - CHUNK_OVERLAP_SECONDS
    != CHUNK_STEP_SECONDS
):

    raise ValueError(
        "Invalid chunking configuration: "
        "chunk_duration - overlap must equal step."
    )


print(
    "✓ Chunk duration - overlap = step"
)


# ----------------------------------------------------------------------
# PERSISTENT PATH REGISTRY
# ----------------------------------------------------------------------
#
# This registry gives later validation cells one stable source for the
# experiment's filesystem layout.
# ----------------------------------------------------------------------

EXP004_PATHS = {
    "project_root": PROJECT_ROOT,
    "musdb18_test_root": MUSDB18_TEST_ROOT,

    "baseline_root": BASELINE_EXP004_ROOT,
    "baseline_notebook": BASELINE_EXP004_NOTEBOOK_DIR,
    "baseline_outputs": BASELINE_EXP004_OUTPUTS_DIR,
    "baseline_logs": BASELINE_EXP004_LOGS_DIR,
    "baseline_chunked_processing": BASELINE_EXP004_CHUNKED_DIR,

    "objective_root": OBJECTIVE_EXP004_ROOT,
    "objective_logs": OBJECTIVE_EXP004_LOGS_DIR,
    "objective_metrics": OBJECTIVE_EXP004_METRICS_DIR,
    "objective_notebook": OBJECTIVE_EXP004_NOTEBOOK_DIR,
    "objective_plots": OBJECTIVE_EXP004_PLOTS_DIR,
    "objective_predictions": OBJECTIVE_EXP004_PREDICTIONS_DIR,
    "objective_reports": OBJECTIVE_EXP004_REPORTS_DIR,
}


# ----------------------------------------------------------------------
# FINAL CELL RESULT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 4 RESULT")
print("=" * 70)

print(f"Experiment: {EXPERIMENT}")

print(
    f"MUSDB18 test root validated: "
    f"{MUSDB18_TEST_ROOT}"
)

print(
    f"Baseline EXP004 root validated: "
    f"{BASELINE_EXP004_ROOT}"
)

print(
    f"Objective EXP004 root initialized: "
    f"{OBJECTIVE_EXP004_ROOT}"
)

print(
    f"Chunking: "
    f"{CHUNK_DURATION_SECONDS}s / "
    f"{CHUNK_OVERLAP_SECONDS}s overlap / "
    f"{CHUNK_STEP_SECONDS}s step"
)

print(
    "✓ Persistent EXP004 path registry initialized"
)

print(
    "✓ Baseline and objective paths safely separated"
)

print("\n" + "=" * 70)
print("✓ CELL 4 COMPLETE")
print("=" * 70)

EXP004 PATH CONFIGURATION

----------------------------------------------------------------------
PROJECT ROOT
----------------------------------------------------------------------
Project root: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation

----------------------------------------------------------------------
MUSDB18 TEST DATASET
----------------------------------------------------------------------
Test dataset: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\03_Datasets\Benchmarks\MUSDB18\test

----------------------------------------------------------------------
BASELINE EXP004
----------------------------------------------------------------------
Baseline root       : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Baseline_HT_Demucs\EXP004
Baseline notebook   : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Baseline_HT_Demucs\EXP004\notebook
Baseline outputs    : G:\My Drive\IK

# MUSDB18 Test Set Validation

In [ ]:
# ======================================================================
# CELL 6 — EXP004 MUSDB18 TEST SET VALIDATION
# ======================================================================

print("=" * 70)
print("EXP004 MUSDB18 TEST SET VALIDATION")
print("=" * 70)


# ----------------------------------------------------------------------
# EXPECTED MUSDB18 TEST TRACKS
# ----------------------------------------------------------------------
#
# These are the 50 track directories supplied for the EXP004 objective
# evaluation.
#
# The list is intentionally fixed so that the experiment cannot
# silently evaluate an incomplete, altered, or substituted test set.
# ----------------------------------------------------------------------

EXPECTED_MUSDB18_TRACKS = [
    "Buitraker - Revo X",
    "M.E.R.C. Music - Knockout",
    "The Sunshine Garcia Band - For I Am The Moon",
    "PR - Oh No",
    "Little Chicago's Finest - My Own",
    "Signe Jakobsen - What Have You Done To Me",
    "The Mountaineering Club - Mallory",
    "Al James - Schoolboy Facination",
    "Speak Softly - Like Horses",
    "Zeno - Signs",
    "Hollow Ground - Ill Fate",
    "Timboz - Pony",
    "Forkupines - Semantics",
    "Moosmusic - Big Dummy Shake",
    "Tom McKenzie - Directions",
    "Ben Carrigan - We'll Talk About It All Tonight",
    "Secretariat - Borderline",
    "AM Contra - Heart Peripheral",
    "Motor Tapes - Shore",
    "Arise - Run Run Run",
    "Skelpolu - Resurrection",
    "Louis Cressy Band - Good Time",
    "Punkdisco - Oral Hygiene",
    "We Fell From The Sky - Not You",
    "Raft Monk - Tiring",
    "BKS - Too Much",
    "Detsky Sad - Walkie Talkie",
    "Girls Under Glass - We Feel Alright",
    "The Long Wait - Dark Horses",
    "BKS - Bulldozer",
    "The Easton Ellises (Baumi) - SDRNR",
    "Mu - Too Bright",
    "Nerve 9 - Pray For The Rain",
    "PR - Happy Daze",
    "Carlos Gonzalez - A Place For Us",
    "Sambasevam Shanmugam - Kaathaadi",
    "The Doppler Shift - Atrophy",
    "The Easton Ellises - Falcon 69",
    "Side Effects Project - Sing With Me",
    "Lyndsey Ollard - Catching Up",
    "Georgia Wonder - Siren",
    "Enda Reilly - Cur An Long Ag Seol",
    "Bobby Nobody - Stitch Up",
    "Juliet's Rescue - Heartbeats",
    "Angels In Amplifiers - I'm Alright",
    "Speak Softly - Broken Man",
    "Triviul feat. The Fiend - Widow",
    "Cristina Vane - So Easy",
    "Secretariat - Over The Top",
    "James Elder & Mark M Thompson - The English Actor",
]


EXPECTED_SOURCE_FILES = [
    "mixture.wav",
    "vocals.wav",
    "drums.wav",
    "bass.wav",
    "other.wav",
]


# ----------------------------------------------------------------------
# EXPECTED TRACK COUNT VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("EXPECTED DATASET CONFIGURATION")
print("-" * 70)

print(
    f"Expected tracks       : "
    f"{len(EXPECTED_MUSDB18_TRACKS)}"
)

print(
    f"Expected source files : "
    f"{len(EXPECTED_SOURCE_FILES)}"
)

print(
    "Expected files/track  : "
    + ", ".join(EXPECTED_SOURCE_FILES)
)


if len(EXPECTED_MUSDB18_TRACKS) != EXPECTED_TRACK_COUNT:

    raise ValueError(
        "Internal configuration error: "
        f"expected track list contains "
        f"{len(EXPECTED_MUSDB18_TRACKS)} tracks, "
        f"but EXPECTED_TRACK_COUNT is "
        f"{EXPECTED_TRACK_COUNT}."
    )


if len(set(EXPECTED_MUSDB18_TRACKS)) != len(
    EXPECTED_MUSDB18_TRACKS
):

    raise ValueError(
        "Internal configuration error: "
        "duplicate track names exist in the expected track list."
    )


# ----------------------------------------------------------------------
# DATASET ROOT VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("DATASET ROOT VALIDATION")
print("-" * 70)

if not MUSDB18_TEST_ROOT.exists():

    raise FileNotFoundError(
        f"MUSDB18 test root does not exist:\n"
        f"{MUSDB18_TEST_ROOT}"
    )


if not MUSDB18_TEST_ROOT.is_dir():

    raise NotADirectoryError(
        f"MUSDB18 test root is not a directory:\n"
        f"{MUSDB18_TEST_ROOT}"
    )


print(
    f"✓ Dataset root exists:\n"
    f"  {MUSDB18_TEST_ROOT}"
)


# ----------------------------------------------------------------------
# ACTUAL TRACK DIRECTORY DISCOVERY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK DIRECTORY DISCOVERY")
print("-" * 70)

actual_track_directories = sorted(
    [
        path
        for path in MUSDB18_TEST_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda path: path.name.lower(),
)


actual_track_names = [
    path.name
    for path in actual_track_directories
]


print(
    f"Track directories discovered: "
    f"{len(actual_track_directories)}"
)

if len(actual_track_directories) != EXPECTED_TRACK_COUNT:

    raise ValueError(
        "Unexpected MUSDB18 test track count: "
        f"found {len(actual_track_directories)}, "
        f"expected {EXPECTED_TRACK_COUNT}."
    )


# ----------------------------------------------------------------------
# TRACK SET CONSISTENCY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK SET CONSISTENCY")
print("-" * 70)

expected_track_set = set(
    EXPECTED_MUSDB18_TRACKS
)

actual_track_set = set(
    actual_track_names
)


missing_tracks = sorted(
    expected_track_set - actual_track_set,
    key=str.lower,
)

unexpected_tracks = sorted(
    actual_track_set - expected_track_set,
    key=str.lower,
)


if missing_tracks:

    print("Missing expected tracks:")

    for track in missing_tracks:
        print(f"  ✗ {track}")

    raise ValueError(
        f"{len(missing_tracks)} expected MUSDB18 tracks are missing."
    )


if unexpected_tracks:

    print("Unexpected tracks discovered:")

    for track in unexpected_tracks:
        print(f"  ✗ {track}")

    raise ValueError(
        f"{len(unexpected_tracks)} unexpected track directories "
        "were found."
    )


print(
    "✓ Actual track set exactly matches the expected 50-track set."
)


# ----------------------------------------------------------------------
# SOURCE FILE VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE FILE VALIDATION")
print("-" * 70)


track_validation_records = []


for track_name in EXPECTED_MUSDB18_TRACKS:

    track_dir = (
        MUSDB18_TEST_ROOT
        / track_name
    )

    if not track_dir.exists():

        raise FileNotFoundError(
            f"Expected track directory disappeared during validation:\n"
            f"{track_dir}"
        )


    files_in_track = sorted(
        [
            path
            for path in track_dir.iterdir()
            if path.is_file()
        ],
        key=lambda path: path.name.lower(),
    )


    actual_file_names = [
        path.name
        for path in files_in_track
    ]


    expected_file_set = set(
        EXPECTED_SOURCE_FILES
    )

    actual_file_set = set(
        actual_file_names
    )


    missing_files = sorted(
        expected_file_set - actual_file_set,
        key=str.lower,
    )

    unexpected_files = sorted(
        actual_file_set - expected_file_set,
        key=str.lower,
    )


    if missing_files:

        print(f"\n✗ {track_name}")
        print("  Missing files:")

        for filename in missing_files:
            print(f"    - {filename}")

        raise ValueError(
            f"Track '{track_name}' is missing required source files."
        )


    if unexpected_files:

        print(f"\n✗ {track_name}")
        print("  Unexpected files:")

        for filename in unexpected_files:
            print(f"    - {filename}")

        raise ValueError(
            f"Track '{track_name}' contains unexpected files."
        )


    if len(files_in_track) != len(
        EXPECTED_SOURCE_FILES
    ):

        raise ValueError(
            f"Track '{track_name}' contains "
            f"{len(files_in_track)} files, "
            f"expected exactly {len(EXPECTED_SOURCE_FILES)}."
        )


    track_validation_records.append(
        {
            "track_name": track_name,
            "track_path": track_dir,
            "mixture": track_dir / "mixture.wav",
            "vocals": track_dir / "vocals.wav",
            "drums": track_dir / "drums.wav",
            "bass": track_dir / "bass.wav",
            "other": track_dir / "other.wav",
        }
    )


    print(
        f"✓ {track_name}"
    )


# ----------------------------------------------------------------------
# SOURCE COVERAGE VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE COVERAGE VALIDATION")
print("-" * 70)


for source_name in EXPECTED_SOURCE_FILES:

    source_count = sum(
        1
        for record in track_validation_records
        if (
            record["track_path"]
            / source_name
        ).is_file()
    )

    print(
        f"{source_name:<12}: "
        f"{source_count}/{EXPECTED_TRACK_COUNT}"
    )

    if source_count != EXPECTED_TRACK_COUNT:

        raise ValueError(
            f"Incomplete source coverage for {source_name}: "
            f"{source_count}/{EXPECTED_TRACK_COUNT}"
        )


print(
    "✓ All five required WAV files exist for all 50 tracks."
)


# ----------------------------------------------------------------------
# PERSISTENT TRACK REGISTRY
# ----------------------------------------------------------------------
#
# This registry is intentionally preserved for later cells.
# Subsequent cells should use MUSDB18_TRACKS instead of rediscovering
# the dataset independently.
# ----------------------------------------------------------------------

MUSDB18_TRACKS = track_validation_records


# ----------------------------------------------------------------------
# SOURCE REGISTRY
# ----------------------------------------------------------------------

SOURCE_FILE_MAP = {
    "mixture": "mixture.wav",
    "vocals": "vocals.wav",
    "drums": "drums.wav",
    "bass": "bass.wav",
    "other": "other.wav",
}


# ----------------------------------------------------------------------
# FINAL STRUCTURAL ASSERTIONS
# ----------------------------------------------------------------------

if len(MUSDB18_TRACKS) != EXPECTED_TRACK_COUNT:

    raise RuntimeError(
        "Persistent MUSDB18_TRACKS registry does not contain "
        f"{EXPECTED_TRACK_COUNT} tracks."
    )


for record in MUSDB18_TRACKS:

    for source_name, filename in SOURCE_FILE_MAP.items():

        path = (
            record["track_path"]
            / filename
        )

        if not path.is_file():

            raise RuntimeError(
                f"Persistent track registry contains an invalid "
                f"path for {record['track_name']} / {source_name}:\n"
                f"{path}"
            )


# ----------------------------------------------------------------------
# FINAL CELL RESULT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6 RESULT")
print("=" * 70)

print(
    f"Tracks validated        : "
    f"{len(MUSDB18_TRACKS)}"
)

print(
    f"Sources per track       : "
    f"{len(EXPECTED_SOURCE_FILES)}"
)

print(
    f"Expected source files   : "
    f"{', '.join(EXPECTED_SOURCE_FILES)}"
)

print(
    "Track set               : EXACT MATCH"
)

print(
    "Source file coverage    : COMPLETE"
)

print(
    "Audio loading performed : NO"
)

print(
    "Inference performed     : NO"
)

print(
    "Metrics computed        : NO"
)

print(
    "✓ Persistent MUSDB18_TRACKS registry established."
)

print("\n" + "=" * 70)
print("✓ CELL 6 COMPLETE")
print("=" * 70)

EXP004 MUSDB18 TEST SET VALIDATION

----------------------------------------------------------------------
EXPECTED DATASET CONFIGURATION
----------------------------------------------------------------------
Expected tracks       : 50
Expected source files : 5
Expected files/track  : mixture.wav, vocals.wav, drums.wav, bass.wav, other.wav

----------------------------------------------------------------------
DATASET ROOT VALIDATION
----------------------------------------------------------------------
✓ Dataset root exists:
  G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\03_Datasets\Benchmarks\MUSDB18\test

----------------------------------------------------------------------
TRACK DIRECTORY DISCOVERY
----------------------------------------------------------------------
Track directories discovered: 50

----------------------------------------------------------------------
TRACK SET CONSISTENCY
---------------------------------------------------------------------

# Audio Compatibility Check

In [ ]:
# ======================================================================
# CELL 8 — EXP004 AUDIO COMPATIBILITY CHECK
# ======================================================================

import math
import soundfile as sf


print("=" * 70)
print("EXP004 AUDIO COMPATIBILITY CHECK")
print("=" * 70)


# ----------------------------------------------------------------------
# AUDIO CONFIGURATION
# ----------------------------------------------------------------------

REQUIRED_AUDIO_FORMAT = {
    "sample_rate": 44100,
    "channels": 2,
}

AUDIO_TOLERANCE_SECONDS = 1.0 / 44100.0


# ----------------------------------------------------------------------
# PERSISTENT AUDIO REGISTRY
# ----------------------------------------------------------------------
#
# This dictionary is preserved for subsequent cells.
# It contains metadata only. No complete waveform arrays are retained.
# ----------------------------------------------------------------------

MUSDB18_AUDIO_METADATA = {}

audio_validation_failures = []


# ----------------------------------------------------------------------
# DATASET-LEVEL CONFIGURATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("AUDIO COMPATIBILITY CONFIGURATION")
print("-" * 70)

print(
    f"Expected sample rate : "
    f"{REQUIRED_AUDIO_FORMAT['sample_rate']} Hz"
)

print(
    f"Expected channels    : "
    f"{REQUIRED_AUDIO_FORMAT['channels']}"
)

print(
    f"Tracks to inspect    : "
    f"{len(MUSDB18_TRACKS)}"
)

print(
    f"Files per track      : "
    f"{len(SOURCE_FILE_MAP)}"
)


# ----------------------------------------------------------------------
# PER-TRACK AUDIO INSPECTION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PER-TRACK AUDIO INSPECTION")
print("-" * 70)


for track_index, record in enumerate(
    MUSDB18_TRACKS,
    start=1,
):

    track_name = record["track_name"]
    track_dir = record["track_path"]


    track_metadata = {
        "track_name": track_name,
        "track_path": str(track_dir),
        "sources": {},
    }


    source_metadata = {}


    # --------------------------------------------------------------
    # Inspect all five WAV files
    # --------------------------------------------------------------

    for source_name, filename in SOURCE_FILE_MAP.items():

        audio_path = (
            track_dir
            / filename
        )


        try:

            info = sf.info(
                str(audio_path)
            )

        except Exception as exc:

            audio_validation_failures.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": str(audio_path),
                    "error": (
                        f"Unable to read WAV metadata: "
                        f"{type(exc).__name__}: {exc}"
                    ),
                }
            )

            continue


        duration = (
            float(info.frames)
            / float(info.samplerate)
        )


        metadata = {
            "path": str(audio_path),
            "sample_rate": int(info.samplerate),
            "channels": int(info.channels),
            "frames": int(info.frames),
            "duration_seconds": duration,
            "format": str(info.format),
            "subtype": str(info.subtype),
        }


        source_metadata[source_name] = metadata


        # ----------------------------------------------------------
        # Basic format validation
        # ----------------------------------------------------------

        if int(info.samplerate) != (
            REQUIRED_AUDIO_FORMAT["sample_rate"]
        ):

            audio_validation_failures.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": str(audio_path),
                    "error": (
                        f"Unexpected sample rate: "
                        f"{info.samplerate} Hz"
                    ),
                }
            )


        if int(info.channels) != (
            REQUIRED_AUDIO_FORMAT["channels"]
        ):

            audio_validation_failures.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": str(audio_path),
                    "error": (
                        f"Unexpected channel count: "
                        f"{info.channels}"
                    ),
                }
            )


        if int(info.frames) <= 0:

            audio_validation_failures.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": str(audio_path),
                    "error": (
                        "Audio file contains no frames."
                    ),
                }
            )


    # --------------------------------------------------------------
    # Skip cross-source checks if metadata could not be read
    # --------------------------------------------------------------

    if len(source_metadata) != len(
        SOURCE_FILE_MAP
    ):

        track_metadata["sources"] = source_metadata

        MUSDB18_AUDIO_METADATA[
            track_name
        ] = track_metadata

        continue


    # --------------------------------------------------------------
    # Cross-source compatibility
    # --------------------------------------------------------------

    mixture = source_metadata["mixture"]


    mixture_frames = mixture["frames"]
    mixture_sample_rate = mixture["sample_rate"]
    mixture_channels = mixture["channels"]
    mixture_duration = mixture["duration_seconds"]


    for source_name in [
        "vocals",
        "drums",
        "bass",
        "other",
    ]:

        source = source_metadata[
            source_name
        ]


        if source["sample_rate"] != (
            mixture_sample_rate
        ):

            audio_validation_failures.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": source["path"],
                    "error": (
                        "Sample rate does not match mixture."
                    ),
                }
            )


        if source["channels"] != (
            mixture_channels
        ):

            audio_validation_failures.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": source["path"],
                    "error": (
                        "Channel count does not match mixture."
                    ),
                }
            )


        frame_difference = abs(
            source["frames"]
            - mixture_frames
        )


        if frame_difference > 1:

            audio_validation_failures.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": source["path"],
                    "error": (
                        "Frame count does not match mixture: "
                        f"difference={frame_difference}"
                    ),
                }
            )


        duration_difference = abs(
            source["duration_seconds"]
            - mixture_duration
        )


        if duration_difference > (
            AUDIO_TOLERANCE_SECONDS * 2
        ):

            audio_validation_failures.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": source["path"],
                    "error": (
                        "Duration does not match mixture: "
                        f"difference="
                        f"{duration_difference:.9f}s"
                    ),
                }
            )


    # --------------------------------------------------------------
    # Track-level metadata
    # --------------------------------------------------------------

    track_metadata.update(
        {
            "sample_rate": mixture_sample_rate,
            "channels": mixture_channels,
            "frames": mixture_frames,
            "duration_seconds": mixture_duration,
            "sources": source_metadata,
        }
    )


    MUSDB18_AUDIO_METADATA[
        track_name
    ] = track_metadata


    # --------------------------------------------------------------
    # Progress output
    # --------------------------------------------------------------

    if track_name not in {
        failure["track"]
        for failure in audio_validation_failures
    }:

        print(
            f"✓ [{track_index:02d}/"
            f"{len(MUSDB18_TRACKS):02d}] "
            f"{track_name} | "
            f"{mixture_sample_rate} Hz | "
            f"{mixture_channels} ch | "
            f"{mixture_duration:.2f} s"
        )

    else:

        print(
            f"✗ [{track_index:02d}/"
            f"{len(MUSDB18_TRACKS):02d}] "
            f"{track_name}"
        )


# ----------------------------------------------------------------------
# DATASET-WIDE SAMPLE RATE / CHANNEL CONSISTENCY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("DATASET-WIDE AUDIO FORMAT CONSISTENCY")
print("-" * 70)


sample_rates = set()
channel_counts = set()


for track_metadata in MUSDB18_AUDIO_METADATA.values():

    if "sample_rate" in track_metadata:

        sample_rates.add(
            track_metadata["sample_rate"]
        )

    if "channels" in track_metadata:

        channel_counts.add(
            track_metadata["channels"]
        )


print(
    f"Sample rates discovered : "
    f"{sorted(sample_rates)}"
)

print(
    f"Channel counts discovered: "
    f"{sorted(channel_counts)}"
)


if sample_rates != {
    REQUIRED_AUDIO_FORMAT["sample_rate"]
}:

    audio_validation_failures.append(
        {
            "track": "__DATASET__",
            "source": "__ALL__",
            "path": str(MUSDB18_TEST_ROOT),
            "error": (
                "Dataset contains unexpected sample-rate values: "
                f"{sorted(sample_rates)}"
            ),
        }
    )


if channel_counts != {
    REQUIRED_AUDIO_FORMAT["channels"]
}:

    audio_validation_failures.append(
        {
            "track": "__DATASET__",
            "source": "__ALL__",
            "path": str(MUSDB18_TEST_ROOT),
            "error": (
                "Dataset contains unexpected channel-count values: "
                f"{sorted(channel_counts)}"
            ),
        }
    )


# ----------------------------------------------------------------------
# TRACK DURATION SUMMARY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK DURATION SUMMARY")
print("-" * 70)


track_durations = [
    metadata["duration_seconds"]
    for metadata in MUSDB18_AUDIO_METADATA.values()
    if "duration_seconds" in metadata
]


if len(track_durations) != EXPECTED_TRACK_COUNT:

    audio_validation_failures.append(
        {
            "track": "__DATASET__",
            "source": "__ALL__",
            "path": str(MUSDB18_TEST_ROOT),
            "error": (
                "Could not obtain valid mixture duration "
                f"for all {EXPECTED_TRACK_COUNT} tracks."
            ),
        }
    )

else:

    minimum_duration = min(
        track_durations
    )

    maximum_duration = max(
        track_durations
    )

    mean_duration = (
        sum(track_durations)
        / len(track_durations)
    )


    print(
        f"Minimum duration : "
        f"{minimum_duration:.2f} s"
    )

    print(
        f"Maximum duration : "
        f"{maximum_duration:.2f} s"
    )

    print(
        f"Mean duration    : "
        f"{mean_duration:.2f} s"
    )


# ----------------------------------------------------------------------
# PHYSICAL CHUNKING COMPATIBILITY
# ----------------------------------------------------------------------
#
# A track does NOT need to be an exact multiple of 60 seconds.
#
# The chunking implementation will handle the final partial segment
# during physical chunk construction.
#
# Therefore this check only verifies that every track has positive
# duration and that the configured chunking parameters are valid.
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PHYSICAL CHUNKING COMPATIBILITY")
print("-" * 70)


for track_name, metadata in (
    MUSDB18_AUDIO_METADATA.items()
):

    duration = metadata.get(
        "duration_seconds"
    )


    if duration is None:
        continue


    if duration <= 0:

        audio_validation_failures.append(
            {
                "track": track_name,
                "source": "mixture",
                "path": metadata["track_path"],
                "error": (
                    "Track duration is not positive."
                ),
            }
        )

        continue


    # Verify that the configured chunk duration and step are
    # mathematically valid for physical chunking.
    if CHUNK_DURATION <= 0:

        raise ValueError(
            "CHUNK_DURATION must be greater than zero."
        )


    if CHUNK_OVERLAP < 0:

        raise ValueError(
            "CHUNK_OVERLAP cannot be negative."
        )


    if CHUNK_STEP <= 0:

        raise ValueError(
            "CHUNK_STEP must be greater than zero."
        )


    if CHUNK_STEP != (
        CHUNK_DURATION - CHUNK_OVERLAP
    ):

        raise ValueError(
            "Physical chunking configuration is inconsistent: "
            "CHUNK_STEP != CHUNK_DURATION - CHUNK_OVERLAP."
        )


print(
    f"✓ All tracks compatible with "
    f"{CHUNK_DURATION}s / "
    f"{CHUNK_OVERLAP}s / "
    f"{CHUNK_STEP}s physical chunking."
)


# ----------------------------------------------------------------------
# FINAL VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("FINAL AUDIO VALIDATION")
print("-" * 70)


if audio_validation_failures:

    print(
        f"✗ Audio validation failures: "
        f"{len(audio_validation_failures)}"
    )


    for failure in audio_validation_failures:

        print(
            f"\nTrack : {failure['track']}"
        )

        print(
            f"Source: {failure['source']}"
        )

        print(
            f"Path  : {failure['path']}"
        )

        print(
            f"Error : {failure['error']}"
        )


    raise ValueError(
        "EXP004 audio compatibility validation failed. "
        "Resolve the reported audio issues before proceeding."
    )


print(
    "✓ All 250 MUSDB18 WAV files passed compatibility checks."
)


# ----------------------------------------------------------------------
# FINAL CELL RESULT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 8 RESULT")
print("=" * 70)

print(
    f"Tracks inspected          : "
    f"{len(MUSDB18_AUDIO_METADATA)}"
)

print(
    f"Audio files inspected     : "
    f"{len(MUSDB18_AUDIO_METADATA) * len(SOURCE_FILE_MAP)}"
)

print(
    f"Sample rate               : "
    f"{REQUIRED_AUDIO_FORMAT['sample_rate']} Hz"
)

print(
    f"Channels                  : "
    f"{REQUIRED_AUDIO_FORMAT['channels']}"
)

print(
    "Within-track alignment    : VERIFIED"
)

print(
    "WAV readability           : VERIFIED"
)

print(
    "Physical chunking         : COMPATIBLE"
)

print(
    "Waveform arrays retained  : NO"
)

print(
    "HT-Demucs inference       : NO"
)

print(
    "Objective metrics         : NO"
)

print(
    "✓ Persistent "
    "MUSDB18_AUDIO_METADATA registry established."
)

print("\n" + "=" * 70)
print("✓ CELL 8 COMPLETE")
print("=" * 70)

EXP004 AUDIO COMPATIBILITY CHECK

----------------------------------------------------------------------
AUDIO COMPATIBILITY CONFIGURATION
----------------------------------------------------------------------
Expected sample rate : 44100 Hz
Expected channels    : 2
Tracks to inspect    : 50
Files per track      : 5

----------------------------------------------------------------------
PER-TRACK AUDIO INSPECTION
----------------------------------------------------------------------
✓ [01/50] Buitraker - Revo X | 44100 Hz | 2 ch | 275.83 s
✓ [02/50] M.E.R.C. Music - Knockout | 44100 Hz | 2 ch | 270.00 s
✓ [03/50] The Sunshine Garcia Band - For I Am The Moon | 44100 Hz | 2 ch | 320.30 s
✓ [04/50] PR - Oh No | 44100 Hz | 2 ch | 76.19 s
✓ [05/50] Little Chicago's Finest - My Own | 44100 Hz | 2 ch | 281.61 s
✓ [06/50] Signe Jakobsen - What Have You Done To Me | 44100 Hz | 2 ch | 177.40 s
✓ [07/50] The Mountaineering Club - Mallory | 44100 Hz | 2 ch | 244.46 s
✓ [08/50] Al James - Schoolboy

# EXP004 HT-Demucs Configuration Check

In [ ]:
# ================================================================
# EXP004 HT-DEMUCS CONFIGURATION CHECK
# ================================================================

print("=" * 70)
print("EXP004 HT-DEMUCS CONFIGURATION CHECK")
print("=" * 70)

# ----------------------------------------------------------------
# MODEL CONFIGURATION
# ----------------------------------------------------------------
print("\n" + "-" * 70)
print("MODEL CONFIGURATION")
print("-" * 70)

print(f"Model name              : {MODEL_NAME}")
print(f"Expected sources        : {EXPECTED_SOURCES}")
print(f"Expected sample rate    : 44100 Hz")
print(f"Expected channels       : 2")

# ----------------------------------------------------------------
# DEMUCS PACKAGE VALIDATION
# ----------------------------------------------------------------
print("\n" + "-" * 70)
print("DEMUCS PACKAGE VALIDATION")
print("-" * 70)

import demucs

print("Demucs package          : ✓ available")
print(f"Demucs version          : {getattr(demucs, '__version__', 'version unavailable')}")

# ----------------------------------------------------------------
# HT-DEMUCS MODEL LOADING
# ----------------------------------------------------------------
print("\n" + "-" * 70)
print("HT-DEMUCS MODEL LOADING")
print("-" * 70)

print("Loading standard pretrained HT-Demucs model...")

from demucs.pretrained import get_model

exp004_model = get_model(MODEL_NAME)

print("✓ Standard pretrained HT-Demucs model loaded")

# ----------------------------------------------------------------
# MODEL IDENTITY VALIDATION
# ----------------------------------------------------------------
print("\n" + "-" * 70)
print("MODEL IDENTITY VALIDATION")
print("-" * 70)

model_sources = list(exp004_model.sources)

print(f"Model class             : {exp004_model.__class__.__name__}")
print(f"Model sources           : {model_sources}")
print(f"Model sample rate       : {exp004_model.samplerate}")
print(f"Model audio channels    : {exp004_model.audio_channels}")

# Source order is model-defined, so compare sets rather than order.
if set(model_sources) != set(EXPECTED_SOURCES):
    raise RuntimeError(
        "HT-Demucs source configuration does not match the expected "
        f"EXP004 source set.\n"
        f"Expected: {EXPECTED_SOURCES}\n"
        f"Found:    {model_sources}"
    )

print("✓ Source set matches EXP004 configuration")
print("✓ Source ordering accepted as model-defined")

if exp004_model.samplerate != 44100:
    raise RuntimeError(
        f"HT-Demucs sample rate mismatch: "
        f"expected 44100, found {exp004_model.samplerate}"
    )

print("✓ Sample rate matches MUSDB18 configuration")

if exp004_model.audio_channels != 2:
    raise RuntimeError(
        f"HT-Demucs channel mismatch: "
        f"expected 2, found {exp004_model.audio_channels}"
    )

print("✓ Channel configuration matches MUSDB18 configuration")

# ----------------------------------------------------------------
# DEVICE CONFIGURATION
# ----------------------------------------------------------------
print("\n" + "-" * 70)
print("DEVICE CONFIGURATION")
print("-" * 70)

import torch

if torch.cuda.is_available():
    EXP004_DEVICE = torch.device("cuda")
else:
    raise RuntimeError(
        "CUDA is not available. EXP004 requires the configured local GPU runtime."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"Selected device         : {EXP004_DEVICE}")
print(f"GPU                     : {gpu_name}")
print(f"GPU VRAM                : {gpu_vram_gb:.2f} GB")

# ----------------------------------------------------------------
# MODEL DEVICE PREPARATION
# ----------------------------------------------------------------
print("\n" + "-" * 70)
print("MODEL DEVICE PREPARATION")
print("-" * 70)

exp004_model.to(EXP004_DEVICE)
exp004_model.eval()

print(f"✓ HT-Demucs moved to {EXP004_DEVICE}")
print("✓ Model set to evaluation mode")

# ----------------------------------------------------------------
# MODEL PARAMETER VALIDATION
# ----------------------------------------------------------------
print("\n" + "-" * 70)
print("MODEL PARAMETER VALIDATION")
print("-" * 70)

total_parameters = sum(
    parameter.numel()
    for parameter in exp004_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in exp004_model.parameters()
    if parameter.requires_grad
)

print(f"Total parameters        : {total_parameters:,}")
print(f"Trainable parameters    : {trainable_parameters:,}")

if total_parameters <= 0:
    raise RuntimeError("HT-Demucs contains no model parameters.")

if trainable_parameters != total_parameters:
    raise RuntimeError(
        "Unexpected parameter state: not all model parameters are present "
        "as expected for the standard pretrained model."
    )

print("✓ Model parameter set is valid")

# ----------------------------------------------------------------
# PHYSICAL CHUNKING COMPATIBILITY
# ----------------------------------------------------------------
print("\n" + "-" * 70)
print("PHYSICAL CHUNKING COMPATIBILITY")
print("-" * 70)

print(f"Chunk duration          : {CHUNK_DURATION}s")
print(f"Chunk overlap           : {CHUNK_OVERLAP}s")
print(f"Chunk step              : {CHUNK_STEP}s")

if CHUNK_DURATION <= 0:
    raise RuntimeError("CHUNK_DURATION must be greater than zero.")

if CHUNK_OVERLAP < 0:
    raise RuntimeError("CHUNK_OVERLAP cannot be negative.")

if CHUNK_OVERLAP >= CHUNK_DURATION:
    raise RuntimeError(
        "CHUNK_OVERLAP must be smaller than CHUNK_DURATION."
    )

if CHUNK_DURATION - CHUNK_OVERLAP != CHUNK_STEP:
    raise RuntimeError(
        "Physical chunking configuration is inconsistent: "
        "CHUNK_DURATION - CHUNK_OVERLAP must equal CHUNK_STEP."
    )

print("✓ Chunk duration is valid")
print("✓ Chunk overlap is valid")
print("✓ Chunk step is valid")
print("✓ 60s / 5s / 55s physical chunking configuration confirmed")

# ----------------------------------------------------------------
# EXP004 MODEL CONFIGURATION REGISTRY
# ----------------------------------------------------------------
print("\n" + "-" * 70)
print("EXP004 MODEL CONFIGURATION REGISTRY")
print("-" * 70)

EXP004_MODEL_CONFIG = {
    "experiment": EXPERIMENT,
    "model_name": MODEL_NAME,
    "model_sources": list(model_sources),
    "expected_sources": list(EXPECTED_SOURCES),
    "sample_rate": int(exp004_model.samplerate),
    "audio_channels": int(exp004_model.audio_channels),
    "device": str(EXP004_DEVICE),
    "gpu_name": gpu_name,
    "gpu_vram_gb": float(gpu_vram_gb),
    "total_parameters": int(total_parameters),
    "trainable_parameters": int(trainable_parameters),
    "chunk_duration_seconds": int(CHUNK_DURATION),
    "chunk_overlap_seconds": int(CHUNK_OVERLAP),
    "chunk_step_seconds": int(CHUNK_STEP),
}

print("✓ EXP004 model configuration registry created")

# ----------------------------------------------------------------
# PERSISTENT REGISTRY
# ----------------------------------------------------------------
EXP004_HTDEMUCS_CONFIG = {
    "experiment": EXPERIMENT,
    "model_name": MODEL_NAME,
    "expected_sources": list(EXPECTED_SOURCES),
    "model_sources": list(model_sources),
    "sample_rate": int(exp004_model.samplerate),
    "audio_channels": int(exp004_model.audio_channels),
    "device": str(EXP004_DEVICE),
    "chunk_duration": int(CHUNK_DURATION),
    "chunk_overlap": int(CHUNK_OVERLAP),
    "chunk_step": int(CHUNK_STEP),
}

print("✓ Persistent EXP004 HT-Demucs configuration established")

# ----------------------------------------------------------------
# FINAL RESULT
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("CELL 10 RESULT")
print("=" * 70)

print(f"Experiment              : {EXPERIMENT}")
print(f"Model                   : {MODEL_NAME}")
print(f"Sources                 : {model_sources}")
print(f"Sample rate             : {exp004_model.samplerate} Hz")
print(f"Channels                : {exp004_model.audio_channels}")
print(f"Device                  : {EXP004_DEVICE}")
print(f"Chunking                : {CHUNK_DURATION}s / "
      f"{CHUNK_OVERLAP}s overlap / {CHUNK_STEP}s step")

print("✓ Standard pretrained HT-Demucs validated")
print("✓ Source configuration validated")
print("✓ Audio configuration validated")
print("✓ CUDA device configuration validated")
print("✓ Physical chunking configuration validated")
print("✓ EXP004 HT-Demucs configuration registry established")

print("\n" + "=" * 70)
print("✓ CELL 10 COMPLETE")
print("=" * 70)

print(
    "No MUSDB18 waveform was loaded, no chunk was created, "
    "no prediction was generated, no reconstruction was performed, "
    "and no objective metric was computed during this cell."
)

EXP004 HT-DEMUCS CONFIGURATION CHECK

----------------------------------------------------------------------
MODEL CONFIGURATION
----------------------------------------------------------------------
Model name              : htdemucs
Expected sources        : ['vocals', 'drums', 'bass', 'other']
Expected sample rate    : 44100 Hz
Expected channels       : 2

----------------------------------------------------------------------
DEMUCS PACKAGE VALIDATION
----------------------------------------------------------------------
Demucs package          : ✓ available
Demucs version          : 4.1.0

----------------------------------------------------------------------
HT-DEMUCS MODEL LOADING
----------------------------------------------------------------------
Loading standard pretrained HT-Demucs model...


C:\IKS_Local_Runtime\iks_eval\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Standard pretrained HT-Demucs model loaded

----------------------------------------------------------------------
MODEL IDENTITY VALIDATION
----------------------------------------------------------------------
Model class             : BagOfModels
Model sources           : ['drums', 'bass', 'other', 'vocals']
Model sample rate       : 44100
Model audio channels    : 2
✓ Source set matches EXP004 configuration
✓ Source ordering accepted as model-defined
✓ Sample rate matches MUSDB18 configuration
✓ Channel configuration matches MUSDB18 configuration

----------------------------------------------------------------------
DEVICE CONFIGURATION
----------------------------------------------------------------------
Selected device         : cuda
GPU                     : NVIDIA GeForce RTX 3050 Laptop GPU
GPU VRAM                : 4.00 GB

----------------------------------------------------------------------
MODEL DEVICE PREPARATION
------------------------------------------------------

# EXP004 Prediction Audit

In [ ]:
print("=" * 70)
print("EXP004_PATHS REGISTRY DIAGNOSTIC")
print("=" * 70)

print("\nType:")
print(type(EXP004_PATHS))

print("\nKeys:")
for key in EXP004_PATHS:
    print(f"  {key!r}")

print("\nFull registry:")
print(EXP004_PATHS)

EXP004_PATHS REGISTRY DIAGNOSTIC

Type:
<class 'dict'>

Keys:
  'project_root'
  'musdb18_test_root'
  'baseline_root'
  'baseline_notebook'
  'baseline_outputs'
  'baseline_logs'
  'baseline_chunked_processing'
  'objective_root'
  'objective_logs'
  'objective_metrics'
  'objective_notebook'
  'objective_plots'
  'objective_predictions'
  'objective_reports'

Full registry:
{'project_root': WindowsPath('G:/My Drive/IKS_Music_Source_Separation_and_Instrument_Isolation'), 'musdb18_test_root': WindowsPath('G:/My Drive/IKS_Music_Source_Separation_and_Instrument_Isolation/03_Datasets/Benchmarks/MUSDB18/test'), 'baseline_root': WindowsPath('G:/My Drive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP004'), 'baseline_notebook': WindowsPath('G:/My Drive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP004/notebook'), 'baseline_outputs': WindowsPath('G:/My Drive/IKS_Music_Source_Separation_and_Instrument_Isolat

In [ ]:
# ======================================================================
# CELL 11 - EXP004 PREDICTION AUDIT
# ======================================================================

from pathlib import Path


print("=" * 70)
print("EXP004 PREDICTION AUDIT")
print("=" * 70)


# ----------------------------------------------------------------------
# PERSISTENT REGISTRY VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PERSISTENT REGISTRY VALIDATION")
print("-" * 70)

required_registries = [
    "EXP004_PATHS",
    "MUSDB18_TRACKS",
    "MUSDB18_AUDIO_METADATA",
    "EXPECTED_TRACK_COUNT",
    "EXPECTED_SOURCE_COUNT",
    "EXPECTED_SOURCES",
]

missing_registries = [
    name
    for name in required_registries
    if name not in globals()
]

if missing_registries:
    raise RuntimeError(
        "Required persistent EXP004 variables are missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_registries
        )
    )

print("✓ EXP004_PATHS registry available")
print("✓ MUSDB18_TRACKS registry available")
print("✓ MUSDB18_AUDIO_METADATA registry available")
print("✓ EXP004 persistent dataset configuration available")


# ----------------------------------------------------------------------
# DATASET REGISTRY CONSISTENCY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("DATASET REGISTRY CONSISTENCY")
print("-" * 70)

if len(MUSDB18_TRACKS) != EXPECTED_TRACK_COUNT:
    raise RuntimeError(
        "MUSDB18 track registry contains "
        f"{len(MUSDB18_TRACKS)} tracks, "
        f"expected {EXPECTED_TRACK_COUNT}."
    )

if len(MUSDB18_AUDIO_METADATA) != EXPECTED_TRACK_COUNT:
    raise RuntimeError(
        "MUSDB18 audio metadata registry contains "
        f"{len(MUSDB18_AUDIO_METADATA)} tracks, "
        f"expected {EXPECTED_TRACK_COUNT}."
    )

print(
    f"✓ MUSDB18_TRACKS contains "
    f"{len(MUSDB18_TRACKS)} tracks"
)

print(
    f"✓ MUSDB18_AUDIO_METADATA contains "
    f"{len(MUSDB18_AUDIO_METADATA)} tracks"
)


# ----------------------------------------------------------------------
# OBJECTIVE PREDICTION DIRECTORY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("OBJECTIVE PREDICTION DIRECTORY")
print("-" * 70)

PREDICTIONS_DIR = Path(
    EXP004_PATHS["objective_predictions"]
)

print("Prediction directory:")
print(f"  {PREDICTIONS_DIR}")

if not PREDICTIONS_DIR.exists():
    PREDICTIONS_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    print("✓ Objective prediction directory created")

elif not PREDICTIONS_DIR.is_dir():
    raise RuntimeError(
        "Configured objective prediction path exists "
        "but is not a directory:\n"
        f"{PREDICTIONS_DIR}"
    )

else:
    print("✓ Objective prediction directory exists")


# ----------------------------------------------------------------------
# EXPECTED PREDICTION STRUCTURE
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("EXPECTED PREDICTION STRUCTURE")
print("-" * 70)

EXPECTED_PREDICTION_SOURCES = list(
    EXPECTED_SOURCES
)

EXPECTED_PREDICTION_FILES = {
    source_name: f"{source_name}.wav"
    for source_name in EXPECTED_PREDICTION_SOURCES
}

EXPECTED_PREDICTION_FILE_COUNT = (
    EXPECTED_TRACK_COUNT
    * EXPECTED_SOURCE_COUNT
)

print(
    f"Expected test tracks       : "
    f"{EXPECTED_TRACK_COUNT}"
)

print(
    f"Expected sources per track : "
    f"{EXPECTED_SOURCE_COUNT}"
)

print(
    "Expected source files      : "
    + ", ".join(
        EXPECTED_PREDICTION_FILES.values()
    )
)

print(
    f"Expected prediction WAVs   : "
    f"{EXPECTED_PREDICTION_FILE_COUNT}"
)

if EXPECTED_SOURCE_COUNT != len(
    EXPECTED_PREDICTION_SOURCES
):
    raise RuntimeError(
        "EXPECTED_SOURCE_COUNT does not match "
        "the configured HT-Demucs source count."
    )


# ----------------------------------------------------------------------
# TRACK-BY-TRACK PREDICTION AUDIT
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK-BY-TRACK PREDICTION AUDIT")
print("-" * 70)

complete_tracks = []
incomplete_tracks = []
missing_prediction_directories = []

prediction_file_count = 0

for index, record in enumerate(
    MUSDB18_TRACKS,
    start=1
):

    track_name = record["track_name"]

    track_prediction_dir = (
        PREDICTIONS_DIR / track_name
    )

    if not track_prediction_dir.is_dir():

        missing_prediction_directories.append(
            track_name
        )

        incomplete_tracks.append(
            {
                "track": track_name,
                "missing": list(
                    EXPECTED_PREDICTION_FILES.values()
                ),
                "directory_missing": True,
            }
        )

        print(
            f"✗ [{index:02d}/{EXPECTED_TRACK_COUNT}] "
            f"{track_name} | "
            f"prediction directory missing"
        )

        continue

    files_found = sorted(
        path.name
        for path in track_prediction_dir.glob("*.wav")
    )

    prediction_file_count += len(files_found)

    missing_files = [
        filename
        for filename in EXPECTED_PREDICTION_FILES.values()
        if not (
            track_prediction_dir / filename
        ).is_file()
    ]

    unexpected_files = [
        filename
        for filename in files_found
        if filename not in
        EXPECTED_PREDICTION_FILES.values()
    ]

    if missing_files:

        incomplete_tracks.append(
            {
                "track": track_name,
                "missing": missing_files,
                "unexpected": unexpected_files,
                "directory_missing": False,
            }
        )

        print(
            f"✗ [{index:02d}/{EXPECTED_TRACK_COUNT}] "
            f"{track_name} | missing: "
            f"{', '.join(missing_files)}"
        )

    else:

        complete_tracks.append(
            track_name
        )

        if unexpected_files:

            print(
                f"✓ [{index:02d}/{EXPECTED_TRACK_COUNT}] "
                f"{track_name} | all required predictions "
                f"present | unexpected WAVs: "
                f"{', '.join(unexpected_files)}"
            )

        else:

            print(
                f"✓ [{index:02d}/{EXPECTED_TRACK_COUNT}] "
                f"{track_name}"
            )


# ----------------------------------------------------------------------
# PREDICTION AUDIT SUMMARY
# ----------------------------------------------------------------------

complete_count = len(
    complete_tracks
)

incomplete_count = len(
    incomplete_tracks
)

expected_prediction_files = (
    EXPECTED_TRACK_COUNT
    * EXPECTED_SOURCE_COUNT
)

print("\n" + "=" * 70)
print("PREDICTION AUDIT SUMMARY")
print("=" * 70)

print(
    f"\nTotal MUSDB18 test tracks:"
    f"\n  {EXPECTED_TRACK_COUNT}"
)

print(
    f"\nComplete prediction tracks:"
    f"\n  {complete_count} / "
    f"{EXPECTED_TRACK_COUNT}"
)

print(
    f"\nIncomplete prediction tracks:"
    f"\n  {incomplete_count} / "
    f"{EXPECTED_TRACK_COUNT}"
)

print(
    f"\nMissing prediction directories:"
    f"\n  {len(missing_prediction_directories)}"
)

print(
    f"\nPrediction WAV files found:"
    f"\n  {prediction_file_count} / "
    f"{expected_prediction_files}"
)


# ----------------------------------------------------------------------
# INFERENCE DECISION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("INFERENCE DECISION")
print("=" * 70)

if complete_count == EXPECTED_TRACK_COUNT:

    INFERENCE_REQUIRED = False
    INFERENCE_MODE = "SKIP_ALL"

    print("\n✓ INFERENCE NOT REQUIRED")

    print(
        "  All expected tracks already have "
        "complete predictions."
    )

elif complete_count == 0:

    INFERENCE_REQUIRED = True
    INFERENCE_MODE = "RUN_ALL"

    print(
        "\n⚠ INFERENCE REQUIRED FOR ALL TRACKS"
    )

else:

    INFERENCE_REQUIRED = True
    INFERENCE_MODE = "RUN_MISSING"

    print(
        "\n⚠ PARTIAL INFERENCE REQUIRED"
    )

    print(
        f"  {incomplete_count} track(s) "
        "are incomplete."
    )


# ----------------------------------------------------------------------
# PERSISTENT PREDICTION AUDIT REGISTRY
# ----------------------------------------------------------------------

PREDICTION_AUDIT = {
    "prediction_directory": str(
        PREDICTIONS_DIR
    ),
    "expected_tracks": EXPECTED_TRACK_COUNT,
    "expected_sources_per_track":
        EXPECTED_SOURCE_COUNT,
    "expected_prediction_files":
        expected_prediction_files,
    "complete_tracks":
        complete_tracks,
    "incomplete_tracks":
        incomplete_tracks,
    "missing_prediction_directories":
        missing_prediction_directories,
    "prediction_file_count":
        prediction_file_count,
    "inference_required":
        INFERENCE_REQUIRED,
    "inference_mode":
        INFERENCE_MODE,
}

print(
    "\n✓ Persistent PREDICTION_AUDIT "
    "registry established."
)


# ----------------------------------------------------------------------
# FINAL MACHINE-READABLE STATE
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("AUDIT STATE")
print("=" * 70)

print(
    f"\nINFERENCE_REQUIRED = "
    f"{INFERENCE_REQUIRED}"
)

print(
    f"INFERENCE_MODE     = "
    f"'{INFERENCE_MODE}'"
)

print(
    f"COMPLETE_TRACKS    = "
    f"{complete_count}"
)

print(
    f"INCOMPLETE_TRACKS  = "
    f"{incomplete_count}"
)

print(
    f"PREDICTION_FILES   = "
    f"{prediction_file_count}"
)

print(
    f"EXPECTED_FILES     = "
    f"{expected_prediction_files}"
)


# ----------------------------------------------------------------------
# CELL 11 RESULT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 11 RESULT")
print("=" * 70)

print(
    f"Tracks audited          : "
    f"{EXPECTED_TRACK_COUNT}"
)

print(
    f"Sources per track       : "
    f"{EXPECTED_SOURCE_COUNT}"
)

print(
    f"Expected prediction WAVs: "
    f"{expected_prediction_files}"
)

print(
    f"Complete tracks         : "
    f"{complete_count}"
)

print(
    f"Incomplete tracks       : "
    f"{incomplete_count}"
)

print(
    f"Inference required      : "
    f"{INFERENCE_REQUIRED}"
)

print(
    f"Inference mode          : "
    f"{INFERENCE_MODE}"
)

print(
    "\nNo MUSDB18 waveform was loaded, "
    "no physical chunk was created, "
    "no inference was performed, "
    "no reconstruction was performed, "
    "and no objective metric was computed."
)

print("\n" + "=" * 70)
print("✓ CELL 11 COMPLETE")
print("=" * 70)

EXP004 PREDICTION AUDIT

----------------------------------------------------------------------
PERSISTENT REGISTRY VALIDATION
----------------------------------------------------------------------
✓ EXP004_PATHS registry available
✓ MUSDB18_TRACKS registry available
✓ MUSDB18_AUDIO_METADATA registry available
✓ EXP004 persistent dataset configuration available

----------------------------------------------------------------------
DATASET REGISTRY CONSISTENCY
----------------------------------------------------------------------
✓ MUSDB18_TRACKS contains 50 tracks
✓ MUSDB18_AUDIO_METADATA contains 50 tracks

----------------------------------------------------------------------
OBJECTIVE PREDICTION DIRECTORY
----------------------------------------------------------------------
Prediction directory:
  G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\predictions
✓ Objective prediction directory exists

-------------------------

# Physical Chunking and Reconstruction Engine

In [ ]:
# ======================================================================
# CELL 12 - PHYSICAL CHUNKING AND RECONSTRUCTION ENGINE
# ======================================================================

from pathlib import Path
import gc
import math
import time

import numpy as np
import soundfile as sf
import torch


print("=" * 70)
print("EXP004 PHYSICAL CHUNKING AND RECONSTRUCTION ENGINE")
print("=" * 70)


# ----------------------------------------------------------------------
# PERSISTENT REGISTRY VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PERSISTENT REGISTRY VALIDATION")
print("-" * 70)

required_registries = [
    "EXP004_PATHS",
    "MUSDB18_TRACKS",
    "MUSDB18_AUDIO_METADATA",
    "EXP004_MODEL_CONFIG",
    "PREDICTION_AUDIT",
]

missing_registries = [
    name
    for name in required_registries
    if name not in globals()
]

if missing_registries:
    raise RuntimeError(
        "Required persistent EXP004 registries are unavailable:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_registries
        )
    )

for registry_name in required_registries:
    print(f"✓ {registry_name} registry available")


# ----------------------------------------------------------------------
# EXP004 MODEL CONFIGURATION REGISTRY VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("EXP004 MODEL CONFIGURATION REGISTRY VALIDATION")
print("-" * 70)

required_model_config_keys = [
    "experiment",
    "model_name",
    "model_sources",
    "expected_sources",
    "sample_rate",
    "audio_channels",
    "device",
    "gpu_name",
    "gpu_vram_gb",
    "total_parameters",
    "trainable_parameters",
    "chunk_duration_seconds",
    "chunk_overlap_seconds",
    "chunk_step_seconds",
]

missing_model_config_keys = [
    key
    for key in required_model_config_keys
    if key not in EXP004_MODEL_CONFIG
]

if missing_model_config_keys:
    raise RuntimeError(
        "EXP004_MODEL_CONFIG is incomplete. Missing keys:\n"
        + "\n".join(
            f"  - {key}"
            for key in missing_model_config_keys
        )
    )

print("✓ EXP004_MODEL_CONFIG contains all required entries")


# ----------------------------------------------------------------------
# MODEL STATE VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("MODEL STATE VALIDATION")
print("-" * 70)

if "exp004_model" not in globals():
    raise RuntimeError(
        "The validated HT-Demucs model from Cell 10 is unavailable."
    )

if "EXP004_DEVICE" not in globals():
    raise RuntimeError(
        "EXP004_DEVICE from the validated Cell 10 configuration "
        "is unavailable."
    )

print(f"Model              : {EXP004_MODEL_CONFIG['model_name']}")
print(f"Device             : {EXP004_DEVICE}")
print(f"Model class        : {type(exp004_model).__name__}")
print(f"Model sources      : {list(exp004_model.sources)}")
print(f"Model sample rate  : {exp004_model.samplerate}")
print(f"Model channels     : {exp004_model.audio_channels}")


# ----------------------------------------------------------------------
# MODEL IDENTITY CONSISTENCY VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("MODEL IDENTITY CONSISTENCY VALIDATION")
print("-" * 70)

configured_model_name = str(
    EXP004_MODEL_CONFIG["model_name"]
)

configured_sources = list(
    EXP004_MODEL_CONFIG["model_sources"]
)

configured_expected_sources = list(
    EXP004_MODEL_CONFIG["expected_sources"]
)

configured_sample_rate = int(
    EXP004_MODEL_CONFIG["sample_rate"]
)

configured_channels = int(
    EXP004_MODEL_CONFIG["audio_channels"]
)

configured_device = str(
    EXP004_MODEL_CONFIG["device"]
)


if "MODEL_NAME" in globals():

    if str(MODEL_NAME) != configured_model_name:
        raise RuntimeError(
            "Model identity mismatch between the canonical EXP004 "
            f"configuration and MODEL_NAME.\n"
            f"EXP004_MODEL_CONFIG : {configured_model_name}\n"
            f"MODEL_NAME          : {MODEL_NAME}"
        )

    print("✓ MODEL_NAME matches EXP004_MODEL_CONFIG")


if configured_model_name != "htdemucs":
    raise RuntimeError(
        "Unexpected EXP004 model name.\n"
        f"Expected: htdemucs\n"
        f"Found:    {configured_model_name}"
    )

if list(exp004_model.sources) != configured_sources:
    raise RuntimeError(
        "Loaded HT-Demucs source ordering does not match the "
        "persistent EXP004 model configuration.\n"
        f"Registry: {configured_sources}\n"
        f"Model:    {list(exp004_model.sources)}"
    )

if set(configured_sources) != set(
    configured_expected_sources
):
    raise RuntimeError(
        "EXP004 model source configuration is internally inconsistent.\n"
        f"Model sources:    {configured_sources}\n"
        f"Expected sources: {configured_expected_sources}"
    )

if int(exp004_model.samplerate) != configured_sample_rate:
    raise RuntimeError(
        "Loaded HT-Demucs sample rate does not match "
        "EXP004_MODEL_CONFIG.\n"
        f"Registry: {configured_sample_rate}\n"
        f"Model:    {exp004_model.samplerate}"
    )

if int(exp004_model.audio_channels) != configured_channels:
    raise RuntimeError(
        "Loaded HT-Demucs channel configuration does not match "
        "EXP004_MODEL_CONFIG.\n"
        f"Registry: {configured_channels}\n"
        f"Model:    {exp004_model.audio_channels}"
    )

if str(EXP004_DEVICE) != configured_device:
    raise RuntimeError(
        "EXP004 device does not match EXP004_MODEL_CONFIG.\n"
        f"Registry: {configured_device}\n"
        f"Runtime:  {EXP004_DEVICE}"
    )

print("✓ Model identity matches persistent configuration")
print("✓ Model source configuration matches")
print("✓ Model sample rate matches")
print("✓ Model channel configuration matches")
print("✓ Model device configuration matches")


# ----------------------------------------------------------------------
# CUDA VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("CUDA DEVICE VALIDATION")
print("-" * 70)

if EXP004_DEVICE.type != "cuda":
    raise RuntimeError(
        "EXP004 requires CUDA execution, but the configured device is "
        f"{EXP004_DEVICE}."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is no longer available in the current runtime."
    )

print("✓ EXP004 CUDA device configuration validated")
print(f"CUDA device          : {torch.cuda.get_device_name(0)}")


# ----------------------------------------------------------------------
# PHYSICAL CHUNKING CONFIGURATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PHYSICAL CHUNKING CONFIGURATION")
print("-" * 70)

# These values are read ONLY from the persistent Cell 10 registry.
# No second independent EXP004 configuration is created here.

chunk_duration = float(
    EXP004_MODEL_CONFIG[
        "chunk_duration_seconds"
    ]
)

chunk_overlap = float(
    EXP004_MODEL_CONFIG[
        "chunk_overlap_seconds"
    ]
)

chunk_step = float(
    EXP004_MODEL_CONFIG[
        "chunk_step_seconds"
    ]
)

sample_rate = int(
    EXP004_MODEL_CONFIG[
        "sample_rate"
    ]
)

channels = int(
    EXP004_MODEL_CONFIG[
        "audio_channels"
    ]
)

print(
    f"Chunk duration : {chunk_duration:g} s"
)

print(
    f"Chunk overlap  : {chunk_overlap:g} s"
)

print(
    f"Chunk step     : {chunk_step:g} s"
)

print(
    f"Sample rate    : {sample_rate} Hz"
)

print(
    f"Channels       : {channels}"
)


# ----------------------------------------------------------------------
# PHYSICAL CHUNKING VALIDATION
# ----------------------------------------------------------------------

if chunk_duration <= 0:
    raise RuntimeError(
        "Chunk duration must be greater than zero."
    )

if chunk_overlap < 0:
    raise RuntimeError(
        "Chunk overlap cannot be negative."
    )

if chunk_overlap >= chunk_duration:
    raise RuntimeError(
        "Chunk overlap must be smaller than chunk duration."
    )

if not math.isclose(
    chunk_duration - chunk_overlap,
    chunk_step,
    rel_tol=0.0,
    abs_tol=1e-9,
):
    raise RuntimeError(
        "Invalid physical chunking configuration: "
        "chunk_duration - overlap != step.\n"
        f"Duration : {chunk_duration}\n"
        f"Overlap  : {chunk_overlap}\n"
        f"Step     : {chunk_step}"
    )

print("✓ Chunk duration is valid")
print("✓ Chunk overlap is valid")
print("✓ Chunk step is valid")
print(
    "✓ 60s / 5s / 55s physical chunking configuration confirmed"
)


# ----------------------------------------------------------------------
# SAMPLE-BASED CHUNK PARAMETERS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("SAMPLE-BASED CHUNK PARAMETERS")
print("-" * 70)

chunk_size_samples = int(
    round(
        chunk_duration * sample_rate
    )
)

overlap_samples = int(
    round(
        chunk_overlap * sample_rate
    )
)

step_samples = int(
    round(
        chunk_step * sample_rate
    )
)

if chunk_size_samples <= 0:
    raise RuntimeError(
        "Computed chunk size is invalid."
    )

if overlap_samples < 0:
    raise RuntimeError(
        "Computed overlap size is invalid."
    )

if step_samples <= 0:
    raise RuntimeError(
        "Computed step size is invalid."
    )

if (
    chunk_size_samples
    - overlap_samples
    != step_samples
):
    raise RuntimeError(
        "Sample-level chunking arithmetic is inconsistent."
    )

print(
    f"Chunk size      : {chunk_size_samples:,} samples"
)

print(
    f"Overlap         : {overlap_samples:,} samples"
)

print(
    f"Step            : {step_samples:,} samples"
)

print("✓ Sample-level chunk parameters validated")


# ----------------------------------------------------------------------
# CHUNK INDEX GENERATOR
# ----------------------------------------------------------------------

def generate_chunk_boundaries(
    total_samples,
    chunk_size,
    step,
):
    """
    Generate deterministic physical chunk boundaries.

    Each returned tuple is:

        (chunk_index, start_sample, end_sample)

    The final chunk may be shorter than the nominal
    chunk duration.
    """

    total_samples = int(total_samples)
    chunk_size = int(chunk_size)
    step = int(step)

    if total_samples <= 0:
        raise ValueError(
            "total_samples must be greater than zero."
        )

    if chunk_size <= 0:
        raise ValueError(
            "chunk_size must be greater than zero."
        )

    if step <= 0:
        raise ValueError(
            "step must be greater than zero."
        )

    boundaries = []

    start = 0
    chunk_index = 0

    while start < total_samples:

        end = min(
            start + chunk_size,
            total_samples,
        )

        boundaries.append(
            (
                chunk_index,
                start,
                end,
            )
        )

        chunk_index += 1

        if end >= total_samples:
            break

        start += step

    return boundaries


# ----------------------------------------------------------------------
# LINEAR CROSSFADE WEIGHTS
# ----------------------------------------------------------------------

def linear_crossfade_weights(length):
    """
    Generate linear overlap weights.

    Returns:
        previous_weight
        current_weight

    Previous chunk fades from 1 -> 0.
    Current chunk rises from 0 -> 1.
    """

    length = int(length)

    if length <= 0:
        return (
            np.empty(
                0,
                dtype=np.float32,
            ),
            np.empty(
                0,
                dtype=np.float32,
            ),
        )

    current_weight = np.linspace(
        0.0,
        1.0,
        num=length,
        endpoint=False,
        dtype=np.float32,
    )

    previous_weight = (
        1.0 - current_weight
    )

    return (
        previous_weight,
        current_weight,
    )


# ----------------------------------------------------------------------
# OVERLAP RECONSTRUCTION ENGINE
# ----------------------------------------------------------------------

def reconstruct_overlapping_chunks(
    chunks,
    boundaries,
    total_samples,
    overlap_samples,
):
    """
    Reconstruct full-length audio from sequential physical chunks
    using linear crossfade over overlapping regions.

    Parameters
    ----------
    chunks:
        List of arrays with shape:
            (channels, samples)

    boundaries:
        List of:
            (chunk_index, start_sample, end_sample)

    total_samples:
        Required final waveform length.

    overlap_samples:
        Physical overlap in samples.

    Returns
    -------
    np.ndarray
        Shape:
            (channels, total_samples)
    """

    if len(chunks) != len(boundaries):
        raise ValueError(
            "Number of chunks does not match number of boundaries."
        )

    total_samples = int(total_samples)
    overlap_samples = int(overlap_samples)

    if total_samples <= 0:
        raise ValueError(
            "total_samples must be greater than zero."
        )

    if not chunks:
        raise ValueError(
            "At least one chunk is required."
        )

    first_chunk = np.asarray(
        chunks[0]
    )

    if first_chunk.ndim != 2:
        raise ValueError(
            "Chunks must have shape (channels, samples)."
        )

    channel_count = (
        first_chunk.shape[0]
    )

    reconstruction = np.zeros(
        (
            channel_count,
            total_samples,
        ),
        dtype=np.float32,
    )

    weights = np.zeros(
        total_samples,
        dtype=np.float32,
    )

    for chunk, boundary in zip(
        chunks,
        boundaries,
    ):

        chunk = np.asarray(
            chunk,
            dtype=np.float32,
        )

        (
            chunk_index,
            start,
            end,
        ) = boundary

        expected_length = (
            end - start
        )

        if chunk.ndim != 2:
            raise ValueError(
                f"Chunk {chunk_index} is not 2-dimensional."
            )

        if chunk.shape[0] != channel_count:
            raise ValueError(
                f"Chunk {chunk_index} channel count "
                "does not match the first chunk."
            )

        if chunk.shape[1] != expected_length:
            raise ValueError(
                f"Chunk {chunk_index} contains "
                f"{chunk.shape[1]} samples but the boundary "
                f"requires {expected_length}."
            )

        # --------------------------------------------------------------
        # Determine overlap with already reconstructed audio.
        # --------------------------------------------------------------

        overlap_start = start

        overlap_end = min(
            end,
            start + overlap_samples,
        )

        # First chunk has no previous chunk.
        if chunk_index == 0:
            overlap_start = end
            overlap_end = end

        if overlap_start < overlap_end:

            overlap_length = (
                overlap_end
                - overlap_start
            )

            (
                previous_weight,
                current_weight,
            ) = linear_crossfade_weights(
                overlap_length
            )

            local_start = (
                overlap_start
                - start
            )

            local_end = (
                overlap_end
                - start
            )

            reconstruction[
                :,
                overlap_start:overlap_end,
            ] *= (
                previous_weight[
                    np.newaxis,
                    :
                ]
            )

            reconstruction[
                :,
                overlap_start:overlap_end,
            ] += (
                chunk[
                    :,
                    local_start:local_end,
                ]
                * current_weight[
                    np.newaxis,
                    :
                ]
            )

            weights[
                overlap_start:overlap_end
            ] = 1.0

            non_overlap_start = (
                overlap_end
            )

        else:

            non_overlap_start = (
                start
            )

        # --------------------------------------------------------------
        # Add non-overlapping portion directly.
        # --------------------------------------------------------------

        if non_overlap_start < end:

            local_start = (
                non_overlap_start
                - start
            )

            local_end = (
                local_start
                + (
                    end
                    - non_overlap_start
                )
            )

            reconstruction[
                :,
                non_overlap_start:end,
            ] = chunk[
                :,
                local_start:local_end,
            ]

            weights[
                non_overlap_start:end
            ] = 1.0

    # ------------------------------------------------------------------
    # Reconstruction coverage validation.
    # ------------------------------------------------------------------

    uncovered = np.where(
        weights == 0.0
    )[0]

    if uncovered.size > 0:
        raise RuntimeError(
            "Reconstruction contains uncovered samples. "
            f"First uncovered sample: "
            f"{int(uncovered[0])}"
        )

    if not np.isfinite(
        reconstruction
    ).all():
        raise RuntimeError(
            "Reconstructed waveform contains non-finite values."
        )

    return reconstruction


# ----------------------------------------------------------------------
# DETERMINISTIC CHUNKING SELF-TEST
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("CHUNKING / RECONSTRUCTION ENGINE SELF-TEST")
print("-" * 70)

test_duration = 135.0

test_total_samples = int(
    round(
        test_duration
        * sample_rate
    )
)

test_boundaries = (
    generate_chunk_boundaries(
        total_samples=test_total_samples,
        chunk_size=chunk_size_samples,
        step=step_samples,
    )
)

print(
    f"Test duration       : {test_duration:g} s"
)

print(
    f"Test total samples  : {test_total_samples:,}"
)

print(
    f"Test chunk count    : {len(test_boundaries)}"
)

for (
    chunk_index,
    start_sample,
    end_sample,
) in test_boundaries:

    duration = (
        end_sample
        - start_sample
    ) / sample_rate

    print(
        f"  Chunk {chunk_index:03d} | "
        f"{start_sample:,} -> "
        f"{end_sample:,} | "
        f"{duration:.3f} s"
    )


# ----------------------------------------------------------------------
# CREATE DETERMINISTIC SYNTHETIC CHUNKS
# ----------------------------------------------------------------------

test_chunks = []

for (
    chunk_index,
    start_sample,
    end_sample,
) in test_boundaries:

    length = (
        end_sample
        - start_sample
    )

    synthetic_chunk = np.full(
        (
            channels,
            length,
        ),
        fill_value=float(
            chunk_index + 1
        ),
        dtype=np.float32,
    )

    test_chunks.append(
        synthetic_chunk
    )


# ----------------------------------------------------------------------
# RECONSTRUCT SYNTHETIC TEST AUDIO
# ----------------------------------------------------------------------

test_reconstruction = (
    reconstruct_overlapping_chunks(
        chunks=test_chunks,
        boundaries=test_boundaries,
        total_samples=test_total_samples,
        overlap_samples=overlap_samples,
    )
)


# ----------------------------------------------------------------------
# SELF-TEST VALIDATION
# ----------------------------------------------------------------------

expected_test_shape = (
    channels,
    test_total_samples,
)

if test_reconstruction.shape != (
    expected_test_shape
):
    raise RuntimeError(
        "Self-test reconstruction shape is incorrect.\n"
        f"Expected: {expected_test_shape}\n"
        f"Found:    {test_reconstruction.shape}"
    )

if not np.isfinite(
    test_reconstruction
).all():
    raise RuntimeError(
        "Self-test reconstruction contains non-finite values."
    )

print("✓ Chunk boundaries generated successfully")
print("✓ Synthetic chunk shapes validated")
print("✓ Linear crossfade reconstruction executed")
print("✓ Reconstructed sample count preserved")
print("✓ Reconstruction contains only finite values")


# ----------------------------------------------------------------------
# PERSISTENT CHUNKING ENGINE REGISTRY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PERSISTENT CHUNKING ENGINE REGISTRY")
print("-" * 70)

EXP004_CHUNKING_ENGINE = {
    "experiment": str(
        EXP004_MODEL_CONFIG["experiment"]
    ),
    "model_name": str(
        EXP004_MODEL_CONFIG["model_name"]
    ),
    "chunk_duration_seconds": float(
        chunk_duration
    ),
    "overlap_seconds": float(
        chunk_overlap
    ),
    "step_seconds": float(
        chunk_step
    ),
    "sample_rate": int(
        sample_rate
    ),
    "channels": int(
        channels
    ),
    "chunk_size_samples": int(
        chunk_size_samples
    ),
    "overlap_samples": int(
        overlap_samples
    ),
    "step_samples": int(
        step_samples
    ),
    "chunk_boundary_generator": (
        generate_chunk_boundaries
    ),
    "reconstruction_function": (
        reconstruct_overlapping_chunks
    ),
    "crossfade_function": (
        linear_crossfade_weights
    ),
}

print(
    "✓ EXP004_CHUNKING_ENGINE registry created"
)

print(
    "✓ Chunk boundary generator registered"
)

print(
    "✓ Reconstruction function registered"
)

print(
    "✓ Crossfade function registered"
)


# ----------------------------------------------------------------------
# FINAL REGISTRY VALIDATION
# ----------------------------------------------------------------------

required_engine_keys = [
    "experiment",
    "model_name",
    "chunk_duration_seconds",
    "overlap_seconds",
    "step_seconds",
    "sample_rate",
    "channels",
    "chunk_size_samples",
    "overlap_samples",
    "step_samples",
    "chunk_boundary_generator",
    "reconstruction_function",
    "crossfade_function",
]

missing_engine_keys = [
    key
    for key in required_engine_keys
    if key not in EXP004_CHUNKING_ENGINE
]

if missing_engine_keys:
    raise RuntimeError(
        "EXP004_CHUNKING_ENGINE was created but is incomplete:\n"
        + "\n".join(
            f"  - {key}"
            for key in missing_engine_keys
        )
    )

print(
    "✓ EXP004_CHUNKING_ENGINE registry fully validated"
)


# ----------------------------------------------------------------------
# FINAL CELL RESULT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 12 RESULT")
print("=" * 70)

print(
    f"Experiment              : "
    f"{EXP004_MODEL_CONFIG['experiment']}"
)

print(
    f"Model                   : "
    f"{EXP004_MODEL_CONFIG['model_name']}"
)

print(
    f"Physical chunk duration : "
    f"{chunk_duration:g} s"
)

print(
    f"Physical overlap        : "
    f"{chunk_overlap:g} s"
)

print(
    f"Physical step           : "
    f"{chunk_step:g} s"
)

print(
    f"Sample rate             : "
    f"{sample_rate} Hz"
)

print(
    f"Channels                : "
    f"{channels}"
)

print(
    "Chunking engine         : READY"
)

print(
    "Reconstruction engine   : READY"
)

print(
    "Crossfade method        : LINEAR"
)

print(
    "MUSDB18 waveform loaded : NO"
)

print(
    "MUSDB18 chunk created   : NO"
)

print(
    "HT-Demucs inference     : NO"
)

print(
    "Prediction generated   : NO"
)

print(
    "Objective metrics       : NO"
)

print(
    "\n✓ Persistent EXP004_CHUNKING_ENGINE registry established"
)

print(
    "✓ Cell 12 completed without processing "
    "any MUSDB18 track"
)

print("=" * 70)

EXP004 PHYSICAL CHUNKING AND RECONSTRUCTION ENGINE

----------------------------------------------------------------------
PERSISTENT REGISTRY VALIDATION
----------------------------------------------------------------------
✓ EXP004_PATHS registry available
✓ MUSDB18_TRACKS registry available
✓ MUSDB18_AUDIO_METADATA registry available
✓ EXP004_MODEL_CONFIG registry available
✓ PREDICTION_AUDIT registry available

----------------------------------------------------------------------
EXP004 MODEL CONFIGURATION REGISTRY VALIDATION
----------------------------------------------------------------------
✓ EXP004_MODEL_CONFIG contains all required entries

----------------------------------------------------------------------
MODEL STATE VALIDATION
----------------------------------------------------------------------
Model              : htdemucs
Device             : cuda
Model class        : BagOfModels
Model sources      : ['drums', 'bass', 'other', 'vocals']
Model sample rate  : 44100


# EXP004 Full-Track Physical Chunk Inference and Reconstruction

In [ ]:
print("=" * 70)
print("EXP004_MODEL_CONFIG REGISTRY DIAGNOSTIC")
print("=" * 70)

print("\nType:")
print(type(EXP004_MODEL_CONFIG))

print("\nKeys:")
for key in EXP004_MODEL_CONFIG.keys():
    print(f"  {key!r}")

print("\nFull registry:")
print(EXP004_MODEL_CONFIG)

EXP004_MODEL_CONFIG REGISTRY DIAGNOSTIC

Type:
<class 'dict'>

Keys:
  'experiment'
  'model_name'
  'model_sources'
  'expected_sources'
  'sample_rate'
  'audio_channels'
  'device'
  'gpu_name'
  'gpu_vram_gb'
  'total_parameters'
  'trainable_parameters'
  'chunk_duration_seconds'
  'chunk_overlap_seconds'
  'chunk_step_seconds'

Full registry:
{'experiment': 'EXP004', 'model_name': 'htdemucs', 'model_sources': ['drums', 'bass', 'other', 'vocals'], 'expected_sources': ['vocals', 'drums', 'bass', 'other'], 'sample_rate': 44100, 'audio_channels': 2, 'device': 'cuda', 'gpu_name': 'NVIDIA GeForce RTX 3050 Laptop GPU', 'gpu_vram_gb': 3.99951171875, 'total_parameters': 41984456, 'trainable_parameters': 41984456, 'chunk_duration_seconds': 60, 'chunk_overlap_seconds': 5, 'chunk_step_seconds': 55}


In [ ]:
# ============================================================================
# CELL 14 — EXP004 FULL-TRACK PHYSICAL CHUNK INFERENCE
# ============================================================================
#
# Purpose
# -------
# Execute the actual EXP004 objective-evaluation prediction stage.
#
# IMPORTANT DESIGN RULES
# ----------------------
# 1. The MUSDB18 test directory is the authoritative source of track discovery.
# 2. No track names are hard-coded.
# 3. No track_id field is assumed.
# 4. The existing MUSDB18_TRACKS registry is used only for cross-validation,
#    not as the source of the inference track list.
# 5. HT-Demucs is invoked through demucs.apply.apply_model().
# 6. The model-defined source ordering is preserved.
# 7. Physical chunking remains exactly:
#       60 s chunk
#       5 s overlap
#       55 s step
# 8. Overlapping predictions are reconstructed using linear crossfade.
# 9. No objective metrics are computed here.
#
# Output:
#   predictions/<actual_track_directory_name>/
#       vocals.wav
#       drums.wav
#       bass.wav
#       other.wav
#
# ============================================================================


from pathlib import Path
import gc
import math
import time
import traceback

import numpy as np
import soundfile as sf
import torch

from demucs.apply import apply_model


print("=" * 70)
print("EXP004 FULL-TRACK PHYSICAL CHUNK INFERENCE")
print("=" * 70)


# ============================================================================
# 1. PERSISTENT REGISTRY VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("PERSISTENT REGISTRY VALIDATION")
print("-" * 70)


required_registries = [
    "EXP004_PATHS",
    "MUSDB18_TRACKS",
    "MUSDB18_AUDIO_METADATA",
    "EXP004_MODEL_CONFIG",
    "PREDICTION_AUDIT",
    "EXP004_CHUNKING_ENGINE",
]


missing_registries = [
    registry_name
    for registry_name in required_registries
    if registry_name not in globals()
]


if missing_registries:

    raise RuntimeError(
        "Required persistent EXP004 registries are unavailable:\n"
        + "\n".join(
            f"  - {registry_name}"
            for registry_name in missing_registries
        )
    )


for registry_name in required_registries:

    print(f"✓ {registry_name} registry available")


# ============================================================================
# 2. MODEL STATE VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("MODEL STATE VALIDATION")
print("-" * 70)


if "exp004_model" not in globals():

    raise RuntimeError(
        "The validated HT-Demucs model `exp004_model` is unavailable."
    )


if "EXP004_DEVICE" not in globals():

    raise RuntimeError(
        "The validated EXP004_DEVICE variable is unavailable."
    )


# ----------------------------------------------------------------------
# Resolve model identity from the persistent configuration registry.
#
# Cell 12 established EXP004_MODEL_CONFIG as the persistent authority
# for the experiment configuration. EXP004_MODEL_NAME is therefore
# reconstructed here instead of requiring a transient notebook variable.
# ----------------------------------------------------------------------

if "model_name" not in EXP004_MODEL_CONFIG:

    raise RuntimeError(
        "EXP004_MODEL_CONFIG does not contain the required "
        "'model_name' entry."
    )


EXP004_MODEL_NAME = str(
    EXP004_MODEL_CONFIG["model_name"]
).strip()


if not EXP004_MODEL_NAME:

    raise RuntimeError(
        "EXP004_MODEL_CONFIG contains an empty model_name."
    )


MODEL_DEVICE = EXP004_DEVICE


if MODEL_DEVICE.type != "cuda":

    raise RuntimeError(
        "EXP004 requires CUDA execution. "
        f"Configured device is {MODEL_DEVICE}."
    )


if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA is not available in the current runtime."
    )


print(f"Model              : {EXP004_MODEL_NAME}")
print(f"Device             : {MODEL_DEVICE}")
print(f"Model class        : {type(exp004_model).__name__}")
print(f"Model sources      : {list(exp004_model.sources)}")
print(f"Model sample rate  : {exp004_model.samplerate}")
print(f"Model channels     : {exp004_model.audio_channels}")


# ----------------------------------------------------------------------
# Validate the model's actual source set against the persistent
# EXP004 configuration.
# ----------------------------------------------------------------------

MODEL_SOURCES = list(exp004_model.sources)

EXPECTED_MODEL_SOURCES = list(
    EXP004_MODEL_CONFIG["model_sources"]
)

EXPECTED_SOURCES = list(
    EXP004_MODEL_CONFIG["expected_sources"]
)


if set(MODEL_SOURCES) != set(EXPECTED_MODEL_SOURCES):

    raise RuntimeError(
        "HT-Demucs source set does not match EXP004_MODEL_CONFIG.\n"
        f"Configured: {EXPECTED_MODEL_SOURCES}\n"
        f"Model:      {MODEL_SOURCES}"
    )


if set(MODEL_SOURCES) != set(EXPECTED_SOURCES):

    raise RuntimeError(
        "HT-Demucs source set does not match the expected EXP004 "
        "source set.\n"
        f"Expected: {EXPECTED_SOURCES}\n"
        f"Model:    {MODEL_SOURCES}"
    )


MODEL_SAMPLE_RATE = int(
    EXP004_MODEL_CONFIG["sample_rate"]
)

MODEL_CHANNELS = int(
    EXP004_MODEL_CONFIG["audio_channels"]
)


if int(exp004_model.samplerate) != MODEL_SAMPLE_RATE:

    raise RuntimeError(
        "HT-Demucs sample-rate mismatch.\n"
        f"Expected: {MODEL_SAMPLE_RATE}\n"
        f"Model:    {exp004_model.samplerate}"
    )


if int(exp004_model.audio_channels) != MODEL_CHANNELS:

    raise RuntimeError(
        "HT-Demucs channel-count mismatch.\n"
        f"Expected: {MODEL_CHANNELS}\n"
        f"Model:    {exp004_model.audio_channels}"
    )


exp004_model.to(MODEL_DEVICE)
exp004_model.eval()


print("✓ Validated HT-Demucs model is available")
print("✓ CUDA device is available")
print("✓ Model source set validated")
print("✓ Model moved to configured device")
print("✓ Model set to evaluation mode")


# ============================================================================
# 3. MODEL SOURCE INDEX MAPPING
# ============================================================================

print("\n" + "-" * 70)
print("MODEL SOURCE INDEX MAPPING")
print("-" * 70)


SOURCE_INDEX = {
    source_name: index
    for index, source_name in enumerate(MODEL_SOURCES)
}


for source_name in EXPECTED_SOURCES:

    if source_name not in SOURCE_INDEX:

        raise RuntimeError(
            f"Required source '{source_name}' is missing "
            "from the loaded HT-Demucs model."
        )


    print(
        f"{source_name:<10} -> "
        f"output index {SOURCE_INDEX[source_name]}"
    )


print("✓ Model-defined source ordering preserved")


# ============================================================================
# 4. PHYSICAL CHUNKING CONFIGURATION
# ============================================================================

print("\n" + "-" * 70)
print("PHYSICAL CHUNKING CONFIGURATION")
print("-" * 70)


CHUNK_DURATION_SECONDS = float(
    EXP004_MODEL_CONFIG["chunk_duration_seconds"]
)

CHUNK_OVERLAP_SECONDS = float(
    EXP004_MODEL_CONFIG["chunk_overlap_seconds"]
)

CHUNK_STEP_SECONDS = float(
    EXP004_MODEL_CONFIG["chunk_step_seconds"]
)


if (
    CHUNK_DURATION_SECONDS
    - CHUNK_OVERLAP_SECONDS
    != CHUNK_STEP_SECONDS
):

    raise RuntimeError(
        "Invalid EXP004 chunking configuration:\n"
        f"duration={CHUNK_DURATION_SECONDS}\n"
        f"overlap={CHUNK_OVERLAP_SECONDS}\n"
        f"step={CHUNK_STEP_SECONDS}"
    )


CHUNK_SIZE = int(
    round(
        CHUNK_DURATION_SECONDS
        * MODEL_SAMPLE_RATE
    )
)

OVERLAP_SIZE = int(
    round(
        CHUNK_OVERLAP_SECONDS
        * MODEL_SAMPLE_RATE
    )
)

STEP_SIZE = int(
    round(
        CHUNK_STEP_SECONDS
        * MODEL_SAMPLE_RATE
    )
)


if STEP_SIZE != CHUNK_SIZE - OVERLAP_SIZE:

    raise RuntimeError(
        "Sample-based chunking configuration is inconsistent."
    )


print(
    f"Chunk duration         : "
    f"{CHUNK_DURATION_SECONDS:g} s"
)

print(
    f"Chunk overlap          : "
    f"{CHUNK_OVERLAP_SECONDS:g} s"
)

print(
    f"Chunk step             : "
    f"{CHUNK_STEP_SECONDS:g} s"
)

print(
    f"Chunk size             : "
    f"{CHUNK_SIZE:,} samples"
)

print(
    f"Chunk overlap          : "
    f"{OVERLAP_SIZE:,} samples"
)

print(
    f"Chunk step             : "
    f"{STEP_SIZE:,} samples"
)

print("✓ Physical chunking configuration validated")


# ============================================================================
# 5. AUTHORITATIVE FILESYSTEM TRACK DISCOVERY
# ============================================================================
#
# This is intentionally independent of MUSDB18_TRACKS.
#
# The actual directory structure is inspected here.
#
# A valid MUSDB18 test track is a directory directly under the test root
# containing mixture.wav.
#
# No track name is assumed.
# No track_id is assumed.
# No ordering is assumed.
#
# ============================================================================

print("\n" + "-" * 70)
print("AUTHORITATIVE MUSDB18 FILESYSTEM DISCOVERY")
print("-" * 70)


TEST_ROOT = Path(
    EXP004_PATHS["musdb18_test_root"]
)


if not TEST_ROOT.exists():

    raise RuntimeError(
        "MUSDB18 test directory does not exist:\n"
        f"{TEST_ROOT}"
    )


if not TEST_ROOT.is_dir():

    raise RuntimeError(
        "MUSDB18 test path is not a directory:\n"
        f"{TEST_ROOT}"
    )


print(f"Test root:\n  {TEST_ROOT}")


# Discover only immediate child directories.
candidate_track_directories = sorted(
    [
        path
        for path in TEST_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda path: path.name.casefold()
)


print(
    f"Track directories discovered : "
    f"{len(candidate_track_directories)}"
)


# ----------------------------------------------------------------------
# A directory qualifies as an inference track if it contains mixture.wav.
# ----------------------------------------------------------------------

filesystem_tracks = []

for track_directory in candidate_track_directories:

    mixture_path = (
        track_directory
        / "mixture.wav"
    )

    if mixture_path.is_file():

        filesystem_tracks.append(
            {
                "track_name": track_directory.name,
                "track_path": track_directory,
                "mixture_path": mixture_path,
            }
        )


EXPECTED_TRACK_COUNT = int(
    EXP004_MODEL_CONFIG.get(
        "expected_tracks",
        globals().get(
            "EXPECTED_TRACK_COUNT",
            50,
        ),
    )
)


if len(filesystem_tracks) != EXPECTED_TRACK_COUNT:

    raise RuntimeError(
        "Filesystem track discovery does not match "
        "the established EXP004 track count.\n"
        f"Expected tracks : {EXPECTED_TRACK_COUNT}\n"
        f"Discovered      : {len(filesystem_tracks)}\n"
        f"Test root       : {TEST_ROOT}"
    )


print(
    f"✓ Exactly {len(filesystem_tracks)} "
    "MUSDB18 inference tracks discovered from filesystem"
)


# ============================================================================
# 6. FILESYSTEM TRACK VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("FILESYSTEM TRACK VALIDATION")
print("-" * 70)


required_source_files = [
    "mixture.wav",
    "vocals.wav",
    "drums.wav",
    "bass.wav",
    "other.wav",
]


for index, track_record in enumerate(
    filesystem_tracks,
    start=1,
):

    track_name = track_record["track_name"]
    track_path = track_record["track_path"]


    missing_files = [
        filename
        for filename in required_source_files
        if not (
            track_path / filename
        ).is_file()
    ]


    if missing_files:

        raise RuntimeError(
            f"Track {index} is missing required MUSDB18 files.\n"
            f"Track: {track_name}\n"
            f"Directory: {track_path}\n"
            f"Missing: {missing_files}"
        )


print(
    f"✓ All {len(filesystem_tracks)} discovered tracks "
    "contain the required five MUSDB18 WAV files"
)


# ============================================================================
# 7. CROSS-VALIDATE AGAINST MUSDB18_TRACKS
# ============================================================================
#
# The filesystem remains authoritative.
#
# The persistent registry is checked only to detect divergence between
# Cell 6 and the actual dataset.
#
# No registry field other than the fields actually established by Cell 6
# is assumed.
#
# ============================================================================

print("\n" + "-" * 70)
print("PERSISTENT REGISTRY CROSS-VALIDATION")
print("-" * 70)


if not isinstance(MUSDB18_TRACKS, list):

    raise RuntimeError(
        "MUSDB18_TRACKS is not a list."
    )


registry_paths = []

for record in MUSDB18_TRACKS:

    if not isinstance(record, dict):

        raise RuntimeError(
            "MUSDB18_TRACKS contains a non-dictionary record."
        )


    if "track_path" not in record:

        raise RuntimeError(
            "A MUSDB18_TRACKS record is missing "
            "the established Cell 6 key 'track_path'."
        )


    registry_paths.append(
        Path(record["track_path"]).resolve()
    )


filesystem_paths = [
    record["track_path"].resolve()
    for record in filesystem_tracks
]


if set(registry_paths) != set(filesystem_paths):

    registry_only = sorted(
        str(path)
        for path in (
            set(registry_paths)
            - set(filesystem_paths)
        )
    )

    filesystem_only = sorted(
        str(path)
        for path in (
            set(filesystem_paths)
            - set(registry_paths)
        )
    )

    raise RuntimeError(
        "MUSDB18_TRACKS and the actual filesystem "
        "do not describe the same 50 tracks.\n\n"
        "Registry-only tracks:\n"
        + "\n".join(
            f"  {path}"
            for path in registry_only
        )
        + "\n\nFilesystem-only tracks:\n"
        + "\n".join(
            f"  {path}"
            for path in filesystem_only
        )
    )


print(
    "✓ Filesystem track set matches the persistent "
    "MUSDB18_TRACKS registry"
)


# ============================================================================
# 8. OBJECTIVE PREDICTION PATH
# ============================================================================

print("\n" + "-" * 70)
print("OBJECTIVE PREDICTION PATH")
print("-" * 70)


PREDICTIONS_ROOT = Path(
    EXP004_PATHS["objective_predictions"]
)


PREDICTIONS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print("Prediction root:")
print(f"  {PREDICTIONS_ROOT}")

print("✓ Objective prediction directory ready")


# ============================================================================
# 9. CHUNK BOUNDARY GENERATOR
# ============================================================================
#
# This implementation follows the validated Cell 12 physical architecture:
#
# 60 s chunks
# 5 s overlap
# 55 s step
#
# The final chunk is shortened to the remaining track length.
#
# ============================================================================

def generate_exp004_chunk_boundaries(
    total_samples,
):

    if total_samples <= 0:

        raise ValueError(
            "Track contains no samples."
        )


    boundaries = []

    start = 0

    while start < total_samples:

        end = min(
            start + CHUNK_SIZE,
            total_samples,
        )

        boundaries.append(
            (start, end)
        )


        if end >= total_samples:

            break


        start += STEP_SIZE


    return boundaries


# ============================================================================
# 10. LINEAR CROSSFADE RECONSTRUCTION
# ============================================================================

def reconstruct_exp004_chunks(
    chunk_predictions,
    chunk_boundaries,
    total_samples,
):
    """
    Reconstruct source predictions from overlapping physical chunks.

    Parameters
    ----------
    chunk_predictions:
        List of arrays with shape [sources, channels, chunk_length].

    chunk_boundaries:
        List of (start_sample, end_sample).

    total_samples:
        Original full-track sample count.

    Returns
    -------
    np.ndarray
        Shape [sources, channels, total_samples].
    """

    if len(chunk_predictions) != len(
        chunk_boundaries
    ):

        raise ValueError(
            "Prediction/boundary count mismatch."
        )


    source_count = len(
        EXPECTED_SOURCES
    )


    reconstructed = np.zeros(
        (
            source_count,
            MODEL_CHANNELS,
            total_samples,
        ),
        dtype=np.float32,
    )


    weights = np.zeros(
        total_samples,
        dtype=np.float32,
    )


    for chunk_index, (
        prediction,
        boundary,
    ) in enumerate(
        zip(
            chunk_predictions,
            chunk_boundaries,
        )
    ):

        start, end = boundary

        expected_length = end - start


        if prediction.shape != (
            source_count,
            MODEL_CHANNELS,
            expected_length,
        ):

            raise RuntimeError(
                "Chunk prediction shape mismatch.\n"
                f"Chunk index: {chunk_index}\n"
                f"Expected: "
                f"{(source_count, MODEL_CHANNELS, expected_length)}\n"
                f"Actual:   {prediction.shape}"
            )


        chunk_length = expected_length


        # --------------------------------------------------------------
        # Construct linear crossfade weights.
        #
        # For non-overlapping boundaries the weights remain 1.
        # For overlapping boundaries, neighbouring chunks are blended.
        # --------------------------------------------------------------

        left_overlap = 0
        right_overlap = 0


        if chunk_index > 0:

            previous_start, previous_end = (
                chunk_boundaries[
                    chunk_index - 1
                ]
            )

            left_overlap = max(
                0,
                previous_end - start,
            )


        if chunk_index < (
            len(chunk_boundaries) - 1
        ):

            next_start, next_end = (
                chunk_boundaries[
                    chunk_index + 1
                ]
            )

            right_overlap = max(
                0,
                end - next_start,
            )


        crossfade = np.ones(
            chunk_length,
            dtype=np.float32,
        )


        # --------------------------------------------------------------
        # Left linear fade-in.
        # --------------------------------------------------------------

        if left_overlap > 0:

            left_overlap = min(
                left_overlap,
                chunk_length,
            )


            crossfade[
                :left_overlap
            ] = np.linspace(
                0.0,
                1.0,
                left_overlap,
                endpoint=False,
                dtype=np.float32,
            )


        # --------------------------------------------------------------
        # Right linear fade-out.
        # --------------------------------------------------------------

        if right_overlap > 0:

            right_overlap = min(
                right_overlap,
                chunk_length,
            )


            crossfade[
                -right_overlap:
            ] = np.linspace(
                1.0,
                0.0,
                right_overlap,
                endpoint=False,
                dtype=np.float32,
            )


        reconstructed[
            :,
            :,
            start:end,
        ] += (
            prediction
            * crossfade[
                None,
                None,
                :,
            ]
        )


        weights[start:end] += crossfade


    if np.any(weights <= 0):

        raise RuntimeError(
            "Chunk reconstruction left uncovered "
            "samples in the full track."
        )


    reconstructed /= weights[
        None,
        None,
        :,
    ]


    if not np.all(
        np.isfinite(reconstructed)
    ):

        raise RuntimeError(
            "Reconstructed prediction contains "
            "non-finite values."
        )


    return reconstructed


# ============================================================================
# 11. HT-DEMUCS SINGLE-CHUNK INFERENCE
# ============================================================================
#
# IMPORTANT:
# ----------------
# DO NOT call:
#
#     exp004_model(chunk)
#
# HT-Demucs loaded by get_model("htdemucs") is a BagOfModels and its forward()
# intentionally raises:
#
#     NotImplementedError: Call `apply_model` on this.
#
# Therefore all inference goes through:
#
#     apply_model(...)
#
# ============================================================================

def infer_exp004_single_chunk(
    chunk_audio,
):

    if not isinstance(
        chunk_audio,
        np.ndarray,
    ):

        raise TypeError(
            "chunk_audio must be a numpy array."
        )


    if chunk_audio.ndim != 2:

        raise ValueError(
            "Expected chunk_audio shape "
            "[channels, samples].\n"
            f"Received: {chunk_audio.shape}"
        )


    if chunk_audio.shape[0] != MODEL_CHANNELS:

        raise ValueError(
            "Unexpected channel count.\n"
            f"Expected: {MODEL_CHANNELS}\n"
            f"Received: {chunk_audio.shape[0]}"
        )


    if not np.all(
        np.isfinite(chunk_audio)
    ):

        raise RuntimeError(
            "Input chunk contains non-finite values."
        )


    # --------------------------------------------------------------
    # Convert:
    #
    # [channels, samples]
    #
    # into:
    #
    # [batch=1, channels, samples]
    # --------------------------------------------------------------

    mixture_tensor = torch.from_numpy(
        np.ascontiguousarray(
            chunk_audio,
            dtype=np.float32,
        )
    ).unsqueeze(0)


    mixture_tensor = mixture_tensor.to(
        MODEL_DEVICE
    )


    # --------------------------------------------------------------
    # ACTUAL DEMUCS INFERENCE
    # --------------------------------------------------------------

    with torch.inference_mode():

        output = apply_model(
            exp004_model,
            mixture_tensor,
            shifts=1,
            split=True,
            overlap=0.25,
            progress=False,
            device=MODEL_DEVICE,
        )


    # --------------------------------------------------------------
    # Expected output:
    #
    # [batch, sources, channels, samples]
    # --------------------------------------------------------------

    if not isinstance(
        output,
        torch.Tensor,
    ):

        raise RuntimeError(
            "Demucs apply_model returned an "
            f"unexpected object: {type(output)}"
        )


    if output.ndim != 4:

        raise RuntimeError(
            "Unexpected HT-Demucs output dimensions.\n"
            f"Expected 4 dimensions, received "
            f"{output.ndim}.\n"
            f"Shape: {tuple(output.shape)}"
        )


    if output.shape[0] != 1:

        raise RuntimeError(
            "Unexpected HT-Demucs batch dimension.\n"
            f"Shape: {tuple(output.shape)}"
        )


    if output.shape[1] != len(
        MODEL_SOURCES
    ):

        raise RuntimeError(
            "Unexpected HT-Demucs source dimension.\n"
            f"Model sources: {MODEL_SOURCES}\n"
            f"Output shape:  {tuple(output.shape)}"
        )


    if output.shape[2] != MODEL_CHANNELS:

        raise RuntimeError(
            "Unexpected HT-Demucs channel dimension.\n"
            f"Expected: {MODEL_CHANNELS}\n"
            f"Output shape: {tuple(output.shape)}"
        )


    output = output.squeeze(
        0
    ).detach().cpu().numpy()


    output = np.asarray(
        output,
        dtype=np.float32,
    )


    if not np.all(
        np.isfinite(output)
    ):

        raise RuntimeError(
            "HT-Demucs prediction contains "
            "non-finite values."
        )


    return output


# ============================================================================
# 12. SINGLE-TRACK INFERENCE
# ============================================================================

def infer_exp004_track(
    track_number,
    track_record,
):

    track_name = track_record[
        "track_name"
    ]

    track_path = track_record[
        "track_path"
    ]

    mixture_path = track_record[
        "mixture_path"
    ]


    track_start_time = time.time()


    print("\n" + "-" * 70)
    print(
        f"[{track_number:02d}/{len(filesystem_tracks):02d}] "
        f"{track_name}"
    )
    print("-" * 70)


    # ------------------------------------------------------------------
    # Load mixture
    # ------------------------------------------------------------------

    mixture_audio, sample_rate = sf.read(
        mixture_path,
        dtype="float32",
        always_2d=True,
    )


    if sample_rate != MODEL_SAMPLE_RATE:

        raise RuntimeError(
            "MUSDB18 mixture sample rate mismatch.\n"
            f"Track:    {track_name}\n"
            f"Expected: {MODEL_SAMPLE_RATE}\n"
            f"Actual:   {sample_rate}"
        )


    # soundfile returns [samples, channels].
    if mixture_audio.shape[1] != MODEL_CHANNELS:

        raise RuntimeError(
            "MUSDB18 mixture channel mismatch.\n"
            f"Track:    {track_name}\n"
            f"Expected: {MODEL_CHANNELS}\n"
            f"Actual:   {mixture_audio.shape[1]}"
        )


    mixture_audio = np.ascontiguousarray(
        mixture_audio.T,
        dtype=np.float32,
    )


    total_samples = mixture_audio.shape[1]


    if total_samples <= 0:

        raise RuntimeError(
            f"Track '{track_name}' contains no audio samples."
        )


    duration_seconds = (
        total_samples
        / MODEL_SAMPLE_RATE
    )


    # ------------------------------------------------------------------
    # Generate physical chunk boundaries.
    # ------------------------------------------------------------------

    chunk_boundaries = (
        generate_exp004_chunk_boundaries(
            total_samples
        )
    )


    print(
        f"  Duration              : "
        f"{duration_seconds:.2f} s"
    )

    print(
        f"  Samples               : "
        f"{total_samples:,}"
    )

    print(
        f"  Physical chunks       : "
        f"{len(chunk_boundaries)}"
    )


    chunk_predictions = []


    # ------------------------------------------------------------------
    # Infer every physical chunk.
    # ------------------------------------------------------------------

    for chunk_index, (
        start,
        end,
    ) in enumerate(
        chunk_boundaries,
        start=1,
    ):

        chunk_start_time = time.time()


        chunk_audio = (
            mixture_audio[
                :,
                start:end,
            ]
        )


        chunk_duration = (
            (end - start)
            / MODEL_SAMPLE_RATE
        )


        print(
            f"    Chunk "
            f"{chunk_index:03d}/"
            f"{len(chunk_boundaries):03d}"
            f" | samples "
            f"{start:,} -> {end:,}"
            f" | {chunk_duration:.2f} s",
            end="",
        )


        try:

            prediction = (
                infer_exp004_single_chunk(
                    chunk_audio
                )
            )


            expected_shape = (
                len(MODEL_SOURCES),
                MODEL_CHANNELS,
                end - start,
            )


            if prediction.shape != (
                expected_shape
            ):

                raise RuntimeError(
                    "Unexpected chunk prediction shape.\n"
                    f"Expected: {expected_shape}\n"
                    f"Actual:   {prediction.shape}"
                )


            chunk_predictions.append(
                prediction
            )


            elapsed = (
                time.time()
                - chunk_start_time
            )


            print(
                f" ✓ "
                f"{elapsed:.2f} s"
            )


        except Exception:

            print(" ✗ FAILED")

            raise


        finally:

            del chunk_audio

            gc.collect()

            if MODEL_DEVICE.type == "cuda":

                torch.cuda.empty_cache()


    # ------------------------------------------------------------------
    # Reconstruct complete track.
    # ------------------------------------------------------------------

    print(
        "  Reconstructing overlapping chunks..."
    )


    reconstructed = (
        reconstruct_exp004_chunks(
            chunk_predictions,
            chunk_boundaries,
            total_samples,
        )
    )


    if reconstructed.shape != (
        len(MODEL_SOURCES),
        MODEL_CHANNELS,
        total_samples,
    ):

        raise RuntimeError(
            "Full-track reconstruction shape mismatch.\n"
            f"Expected: "
            f"{(len(MODEL_SOURCES), MODEL_CHANNELS, total_samples)}\n"
            f"Actual:   {reconstructed.shape}"
        )


    print(
        "  ✓ Linear crossfade reconstruction complete"
    )


    # ------------------------------------------------------------------
    # Track prediction directory.
    #
    # IMPORTANT:
    # The actual filesystem directory name is used directly.
    # Nothing is hard-coded.
    # ------------------------------------------------------------------

    track_output_dir = (
        PREDICTIONS_ROOT
        / track_name
    )


    track_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    expected_output_files = {}


    # ------------------------------------------------------------------
    # Write predictions according to model-defined source order.
    # ------------------------------------------------------------------

    for source_name in EXPECTED_SOURCES:

        source_index = SOURCE_INDEX[
            source_name
        ]


        output_path = (
            track_output_dir
            / f"{source_name}.wav"
        )


        source_audio = (
            reconstructed[
                source_index
            ]
        )


        # soundfile expects [samples, channels].
        source_audio = (
            np.ascontiguousarray(
                source_audio.T,
                dtype=np.float32,
            )
        )


        if source_audio.shape != (
            total_samples,
            MODEL_CHANNELS,
        ):

            raise RuntimeError(
                "Output source shape mismatch.\n"
                f"Source: {source_name}\n"
                f"Expected: "
                f"{(total_samples, MODEL_CHANNELS)}\n"
                f"Actual:   {source_audio.shape}"
            )


        if not np.all(
            np.isfinite(source_audio)
        ):

            raise RuntimeError(
                f"Output '{source_name}' "
                "contains non-finite values."
            )


        sf.write(
            output_path,
            source_audio,
            MODEL_SAMPLE_RATE,
            subtype="FLOAT",
        )


        expected_output_files[
            source_name
        ] = output_path


        print(
            f"  ✓ {source_name:<7} -> "
            f"{output_path.name}"
        )


    # ------------------------------------------------------------------
    # Validate written files immediately.
    # ------------------------------------------------------------------

    print(
        "  Validating written prediction files..."
    )


    for source_name, output_path in (
        expected_output_files.items()
    ):

        if not output_path.is_file():

            raise RuntimeError(
                "Expected prediction file was not created:\n"
                f"{output_path}"
            )


        with sf.SoundFile(
            output_path,
            mode="r",
        ) as info:

            if info.samplerate != (
                MODEL_SAMPLE_RATE
            ):

                raise RuntimeError(
                    "Prediction sample-rate mismatch.\n"
                    f"{output_path}\n"
                    f"Expected: {MODEL_SAMPLE_RATE}\n"
                    f"Actual:   {info.samplerate}"
                )


            if info.channels != (
                MODEL_CHANNELS
            ):

                raise RuntimeError(
                    "Prediction channel mismatch.\n"
                    f"{output_path}\n"
                    f"Expected: {MODEL_CHANNELS}\n"
                    f"Actual:   {info.channels}"
                )


            if info.frames != total_samples:

                raise RuntimeError(
                    "Prediction duration mismatch.\n"
                    f"{output_path}\n"
                    f"Expected frames: {total_samples:,}\n"
                    f"Actual frames:   {info.frames:,}"
                )


    print(
        "  ✓ All four prediction WAVs validated"
    )


    # ------------------------------------------------------------------
    # Cleanup
    # ------------------------------------------------------------------

    del chunk_predictions
    del reconstructed
    del mixture_audio

    gc.collect()

    if MODEL_DEVICE.type == "cuda":

        torch.cuda.empty_cache()


    elapsed_total = (
        time.time()
        - track_start_time
    )


    print(
        f"  ✓ Track completed in "
        f"{elapsed_total / 60:.2f} min"
    )


    return {
        "track_name": track_name,
        "track_path": track_path,
        "track_number": track_number,
        "chunk_count": len(
            chunk_boundaries
        ),
        "total_samples": total_samples,
        "duration_seconds": duration_seconds,
        "prediction_directory": track_output_dir,
        "prediction_files": expected_output_files,
        "status": "COMPLETED",
        "elapsed_seconds": elapsed_total,
    }


# ============================================================================
# 13. INFERENCE SCOPE
# ============================================================================

print("\n" + "-" * 70)
print("INFERENCE SCOPE")
print("-" * 70)


print(
    "Filesystem is authoritative for "
    "track discovery."
)

print(
    f"Tracks scheduled for inference : "
    f"{len(filesystem_tracks)}"
)

print(
    f"Expected track count           : "
    f"{EXPECTED_TRACK_COUNT}"
)


if len(filesystem_tracks) != (
    EXPECTED_TRACK_COUNT
):

    raise RuntimeError(
        "Inference scope is not exactly "
        f"{EXPECTED_TRACK_COUNT} tracks."
    )


print(
    "✓ Exact 50-track filesystem inference "
    "scope confirmed"
)


# ============================================================================
# 14. FULL 50-TRACK INFERENCE
# ============================================================================

print("\n")
print("=" * 70)
print("STARTING EXP004 FULL-TRACK INFERENCE")
print("=" * 70)


overall_start_time = time.time()


successful_tracks = []
inference_failures = []


for track_number, track_record in enumerate(
    filesystem_tracks,
    start=1,
):

    try:

        result = infer_exp004_track(
            track_number,
            track_record,
        )


        successful_tracks.append(
            result
        )


    except Exception as exc:

        track_name = track_record[
            "track_name"
        ]


        failure_record = {
            "track_name": track_name,
            "track_path": track_record[
                "track_path"
            ],
            "error_type": type(
                exc
            ).__name__,
            "error_message": str(
                exc
            ),
            "traceback": traceback.format_exc(),
        }


        inference_failures.append(
            failure_record
        )


        print(
            "\n"
            + "!" * 70
        )

        print(
            f"TRACK INFERENCE FAILED: "
            f"{track_name}"
        )

        print(
            "!" * 70
        )

        print(
            f"Error: "
            f"{type(exc).__name__}: {exc}"
        )


        # --------------------------------------------------------------
        # Continue with remaining tracks.
        #
        # This prevents one corrupt/problematic track from terminating
        # the entire 50-track run.
        # --------------------------------------------------------------

        continue


overall_elapsed = (
    time.time()
    - overall_start_time
)


# ============================================================================
# 15. POST-INFERENCE PREDICTION VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("POST-INFERENCE PREDICTION VALIDATION")
print("-" * 70)


expected_prediction_count = (
    EXPECTED_TRACK_COUNT
    * len(EXPECTED_SOURCES)
)


actual_prediction_count = 0
incomplete_prediction_tracks = []


for track_record in filesystem_tracks:

    track_name = track_record[
        "track_name"
    ]


    track_output_dir = (
        PREDICTIONS_ROOT
        / track_name
    )


    track_complete = True


    for source_name in EXPECTED_SOURCES:

        output_path = (
            track_output_dir
            / f"{source_name}.wav"
        )


        if not output_path.is_file():

            track_complete = False

            continue


        actual_prediction_count += 1


        # Validate file metadata again.
        with sf.SoundFile(
            output_path,
            mode="r",
        ) as info:

            mixture_path = (
                track_record[
                    "mixture_path"
                ]
            )


            with sf.SoundFile(
                mixture_path,
                mode="r",
            ) as mixture_info:

                if (
                    info.samplerate
                    != MODEL_SAMPLE_RATE
                    or info.channels
                    != MODEL_CHANNELS
                    or info.frames
                    != mixture_info.frames
                ):

                    track_complete = False


    if not track_complete:

        incomplete_prediction_tracks.append(
            track_name
        )


print(
    f"Tracks successfully inferred : "
    f"{len(successful_tracks)}"
)

print(
    f"Tracks with inference errors  : "
    f"{len(inference_failures)}"
)

print(
    f"Tracks with incomplete output : "
    f"{len(incomplete_prediction_tracks)}"
)

print(
    f"Prediction WAVs found         : "
    f"{actual_prediction_count} / "
    f"{expected_prediction_count}"
)


# ============================================================================
# 16. UPDATE PREDICTION AUDIT REGISTRY
# ============================================================================

complete_prediction_tracks = (
    EXPECTED_TRACK_COUNT
    - len(incomplete_prediction_tracks)
)


PREDICTION_AUDIT = {
    **PREDICTION_AUDIT,

    "prediction_directory": str(
        PREDICTIONS_ROOT
    ),

    "expected_tracks":
        EXPECTED_TRACK_COUNT,

    "expected_sources_per_track":
        len(EXPECTED_SOURCES),

    "expected_prediction_files":
        expected_prediction_count,

    "complete_tracks":
        complete_prediction_tracks,

    "incomplete_tracks":
        len(incomplete_prediction_tracks),

    "missing_prediction_directories": [
        str(
            PREDICTIONS_ROOT
            / track_name
        )
        for track_name in (
            incomplete_prediction_tracks
        )
        if not (
            PREDICTIONS_ROOT
            / track_name
        ).is_dir()
    ],

    "prediction_file_count":
        actual_prediction_count,

    "inference_required":
        (
            complete_prediction_tracks
            != EXPECTED_TRACK_COUNT
        ),

    "inference_mode":
        (
            "SKIP_ALL"
            if complete_prediction_tracks
            == EXPECTED_TRACK_COUNT
            else "RUN_MISSING"
        ),

    "filesystem_track_discovery":
        True,

    "filesystem_test_root":
        str(TEST_ROOT),

    "successful_tracks":
        [
            result["track_name"]
            for result in successful_tracks
        ],

    "inference_failures":
        inference_failures,

    "incomplete_prediction_tracks":
        incomplete_prediction_tracks,

    "total_inference_seconds":
        overall_elapsed,
}


# ============================================================================
# 17. PERSISTENT EXP004_INFERENCE_RESULTS REGISTRY
# ============================================================================

EXP004_INFERENCE_RESULTS = {
    "experiment": "EXP004",

    "model_name":
        EXP004_MODEL_NAME,

    "device":
        str(MODEL_DEVICE),

    "test_root":
        TEST_ROOT,

    "prediction_root":
        PREDICTIONS_ROOT,

    "track_count":
        EXPECTED_TRACK_COUNT,

    "tracks_discovered_from_filesystem":
        len(filesystem_tracks),

    "sources":
        list(EXPECTED_SOURCES),

    "model_source_order":
        list(MODEL_SOURCES),

    "source_index_mapping":
        dict(SOURCE_INDEX),

    "sample_rate":
        MODEL_SAMPLE_RATE,

    "channels":
        MODEL_CHANNELS,

    "chunk_duration_seconds":
        CHUNK_DURATION_SECONDS,

    "chunk_overlap_seconds":
        CHUNK_OVERLAP_SECONDS,

    "chunk_step_seconds":
        CHUNK_STEP_SECONDS,

    "chunk_size_samples":
        CHUNK_SIZE,

    "overlap_samples":
        OVERLAP_SIZE,

    "step_samples":
        STEP_SIZE,

    "reconstruction":
        "LINEAR_CROSSFADE",

    "inference_api":
        "demucs.apply.apply_model",

    "successful_track_count":
        len(successful_tracks),

    "failed_track_count":
        len(inference_failures),

    "complete_prediction_track_count":
        complete_prediction_tracks,

    "incomplete_prediction_track_count":
        len(incomplete_prediction_tracks),

    "prediction_file_count":
        actual_prediction_count,

    "expected_prediction_file_count":
        expected_prediction_count,

    "inference_failures":
        inference_failures,

    "incomplete_prediction_tracks":
        incomplete_prediction_tracks,

    "total_inference_seconds":
        overall_elapsed,

    "status":
        (
            "COMPLETE"
            if (
                len(successful_tracks)
                == EXPECTED_TRACK_COUNT
                and actual_prediction_count
                == expected_prediction_count
                and not inference_failures
                and not incomplete_prediction_tracks
            )
            else "INCOMPLETE"
        ),
}


# ============================================================================
# 18. FINAL RESULT
# ============================================================================

print("\n")
print("=" * 70)
print("EXP004 INFERENCE RESULT")
print("=" * 70)


print(
    f"Test root                 : "
    f"{TEST_ROOT}"
)

print(
    f"Tracks discovered         : "
    f"{len(filesystem_tracks)}"
)

print(
    f"Tracks completed          : "
    f"{len(successful_tracks)}"
)

print(
    f"Tracks failed             : "
    f"{len(inference_failures)}"
)

print(
    f"Complete prediction sets  : "
    f"{complete_prediction_tracks} / "
    f"{EXPECTED_TRACK_COUNT}"
)

print(
    f"Prediction WAVs           : "
    f"{actual_prediction_count} / "
    f"{expected_prediction_count}"
)

print(
    f"Chunking                  : "
    f"{CHUNK_DURATION_SECONDS:g}s / "
    f"{CHUNK_OVERLAP_SECONDS:g}s overlap / "
    f"{CHUNK_STEP_SECONDS:g}s step"
)

print(
    "Inference API             : "
    "demucs.apply.apply_model"
)

print(
    "Reconstruction            : "
    "LINEAR CROSSFADE"
)

print(
    f"Total inference time      : "
    f"{overall_elapsed / 60:.2f} min"
)


if (
    len(successful_tracks)
    == EXPECTED_TRACK_COUNT
    and actual_prediction_count
    == expected_prediction_count
    and not inference_failures
    and not incomplete_prediction_tracks
):

    print(
        "\n✓ ALL 50 TRACKS COMPLETED"
    )

    print(
        "✓ ALL 200 PREDICTION WAV FILES "
        "VALIDATED"
    )

    print(
        "✓ EXP004 FULL-TRACK INFERENCE "
        "COMPLETE"
    )


else:

    print(
        "\n⚠ EXP004 inference is incomplete."
    )

    if inference_failures:

        print(
            "\nFailed tracks:"
        )

        for failure in inference_failures:

            print(
                f"  - "
                f"{failure['track_name']}: "
                f"{failure['error_type']}: "
                f"{failure['error_message']}"
            )


    if incomplete_prediction_tracks:

        print(
            "\nTracks with incomplete predictions:"
        )

        for track_name in (
            incomplete_prediction_tracks
        ):

            print(
                f"  - {track_name}"
            )


    print(
        "\n⚠ Do not proceed to objective "
        "metrics until prediction validation "
        "is complete."
    )


print(
    "\n✓ Persistent PREDICTION_AUDIT "
    "registry updated."
)

print(
    "✓ Persistent EXP004_INFERENCE_RESULTS "
    "registry established."
)

print(
    "\nNo objective metrics were computed "
    "during this cell."
)

print("=" * 70)
print("✓ CELL 14 COMPLETE")
print("=" * 70)

EXP004 FULL-TRACK PHYSICAL CHUNK INFERENCE

----------------------------------------------------------------------
PERSISTENT REGISTRY VALIDATION
----------------------------------------------------------------------
✓ EXP004_PATHS registry available
✓ MUSDB18_TRACKS registry available
✓ MUSDB18_AUDIO_METADATA registry available
✓ EXP004_MODEL_CONFIG registry available
✓ PREDICTION_AUDIT registry available
✓ EXP004_CHUNKING_ENGINE registry available

----------------------------------------------------------------------
MODEL STATE VALIDATION
----------------------------------------------------------------------
Model              : htdemucs
Device             : cuda
Model class        : BagOfModels
Model sources      : ['drums', 'bass', 'other', 'vocals']
Model sample rate  : 44100
Model channels     : 2
✓ Validated HT-Demucs model is available
✓ CUDA device is available
✓ Model source set validated
✓ Model moved to configured device
✓ Model set to evaluation mode

-------------------

KeyboardInterrupt: 

# EXP004 OBJECTIVE METRIC COMPUTATION

In [ ]:
# ============================================================================
# CELL 15 — EXP004 RESUMABLE OBJECTIVE METRIC EVALUATION
# ============================================================================
#
# Purpose
# -------
# Compute objective metrics for complete EXP004 prediction sets only.
#
# This cell is deliberately independent of the expensive HT-Demucs
# inference stage. It reads prediction WAV files already present on disk.
#
# APPROVED METRICS
# ----------------
#   1. SDR
#   2. SIR
#   3. SAR
#   4. SI-SDR
#
# EXCLUDED
# --------
#   ISR
#
# BSSEVAL CONFIGURATION
# ---------------------
#   BSSEval v4
#   Whole-track evaluation
#   window = np.inf
#   hop = np.inf
#   filters_len = 512
#   framewise_filters = False
#   compute_permutation = False
#   Fixed source ordering
#
# SI-SDR CONFIGURATION
# --------------------
#   TorchMetrics ScaleInvariantSignalDistortionRatio
#   Scale-invariant
#
# RESUMABILITY
# ------------
# Results are persisted after EVERY successfully completed track.
#
# If the notebook/kernel/laptop is closed:
#
#   1. Reopen the notebook.
#   2. Run only the inexpensive validation/configuration cells.
#   3. Run this Cell 15 again.
#
# Previously completed tracks are detected from the persistent JSONL file
# and are NOT recomputed.
#
# INCOMPLETE PREDICTIONS
# ----------------------
# A prediction track is eligible only if ALL FOUR files exist and can be
# decoded:
#
#   vocals.wav
#   drums.wav
#   bass.wav
#   other.wav
#
# Incomplete prediction sets are SKIPPED, not treated as metric failures.
#
# IMPORTANT
# ---------
# This cell does NOT invoke HT-Demucs.
# This cell does NOT perform inference.
#
# ============================================================================


from pathlib import Path
from datetime import datetime
import csv
import gc
import json
import math
import os
import tempfile
import time
import traceback

import numpy as np
import soundfile as sf
import torch

import museval

from museval.metrics import bss_eval

from torchmetrics.audio import (
    ScaleInvariantSignalDistortionRatio,
)


print("=" * 70)
print("EXP004 RESUMABLE OBJECTIVE METRIC EVALUATION")
print("=" * 70)


# ============================================================================
# 1. PERSISTENT REGISTRY VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("PERSISTENT REGISTRY VALIDATION")
print("-" * 70)


required_registries = [
    "EXP004_PATHS",
    "MUSDB18_TRACKS",
    "MUSDB18_AUDIO_METADATA",
    "EXP004_MODEL_CONFIG",
    "PREDICTION_AUDIT",
]


missing_registries = [
    name
    for name in required_registries
    if name not in globals()
]


if missing_registries:

    raise RuntimeError(
        "Required persistent EXP004 registries are unavailable:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_registries
        )
    )


for registry_name in required_registries:

    print(
        f"✓ {registry_name} registry available"
    )


# ============================================================================
# 2. EXP004 EVALUATION CONFIGURATION
# ============================================================================

print("\n" + "-" * 70)
print("EXP004 EVALUATION CONFIGURATION")
print("-" * 70)


EXPERIMENT_NAME = "EXP004"

EXPECTED_TRACK_COUNT = int(
    EXP004_MODEL_CONFIG.get(
        "expected_tracks",
        50,
    )
)


SOURCE_NAMES = [
    "vocals",
    "drums",
    "bass",
    "other",
]


APPROVED_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]


EXCLUDED_METRICS = [
    "ISR",
]


EXPECTED_SOURCES_PER_TRACK = len(
    SOURCE_NAMES
)


EXPECTED_SOURCE_EVALUATIONS = (
    EXPECTED_TRACK_COUNT
    * EXPECTED_SOURCES_PER_TRACK
)


MODEL_SAMPLE_RATE = int(
    EXP004_MODEL_CONFIG["sample_rate"]
)


MODEL_CHANNELS = int(
    EXP004_MODEL_CONFIG["audio_channels"]
)


BSSEVAL_FILTERS_LEN = 512
BSSEVAL_WINDOW = "whole_track"
BSSEVAL_FRAMEWISE_FILTERS = False
BSSEVAL_COMPUTE_PERMUTATION = False


print(
    f"Experiment                  : "
    f"{EXPERIMENT_NAME}"
)

print(
    f"Expected tracks             : "
    f"{EXPECTED_TRACK_COUNT}"
)

print(
    f"Sources                     : "
    f"{SOURCE_NAMES}"
)

print(
    f"Expected source evaluations : "
    f"{EXPECTED_SOURCE_EVALUATIONS}"
)

print(
    f"Sample rate                 : "
    f"{MODEL_SAMPLE_RATE}"
)

print(
    f"Channels                    : "
    f"{MODEL_CHANNELS}"
)

print(
    f"BSSEval window              : "
    f"{BSSEVAL_WINDOW}"
)

print(
    f"BSSEval filter length      : "
    f"{BSSEVAL_FILTERS_LEN}"
)

print(
    f"Framewise filters           : "
    f"{BSSEVAL_FRAMEWISE_FILTERS}"
)

print(
    f"Permutation search          : "
    f"{BSSEVAL_COMPUTE_PERMUTATION}"
)

print(
    f"Approved metrics            : "
    f"{APPROVED_METRICS}"
)

print(
    f"Excluded metrics            : "
    f"{EXCLUDED_METRICS}"
)


if BSSEVAL_FILTERS_LEN != 512:

    raise RuntimeError(
        "EXP004 requires BSSEval filter length 512."
    )


if BSSEVAL_FRAMEWISE_FILTERS:

    raise RuntimeError(
        "EXP004 requires time-invariant BSSEval filters."
    )


if BSSEVAL_COMPUTE_PERMUTATION:

    raise RuntimeError(
        "EXP004 requires fixed source ordering. "
        "Permutation search must remain disabled."
    )


# ============================================================================
# 3. PATH CONFIGURATION
# ============================================================================

print("\n" + "-" * 70)
print("PATH CONFIGURATION")
print("-" * 70)


TEST_ROOT = Path(
    EXP004_PATHS["musdb18_test_root"]
)


PREDICTIONS_ROOT = Path(
    r"G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation"
    r"\05_Experiments\Evaluation\Objective\EXP004\predictions"
)


# ---------------------------------------------------------------------------
# Prefer the persistent EXP004 metric directory if already established.
# Otherwise derive it from the prediction directory.
# ---------------------------------------------------------------------------

if "objective_metrics" in EXP004_PATHS:

    METRICS_ROOT = Path(
        EXP004_PATHS["objective_metrics"]
    )

else:

    METRICS_ROOT = (
        PREDICTIONS_ROOT.parent
        / "metrics"
    )


METRICS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


PER_SOURCE_CSV = (
    METRICS_ROOT
    / "EXP004_per_source_metrics.csv"
)


PER_TRACK_JSONL = (
    METRICS_ROOT
    / "EXP004_per_track_metrics.jsonl"
)


EVALUATION_AUDIT_JSON = (
    METRICS_ROOT
    / "EXP004_metric_integrity_audit.json"
)


EVALUATION_LOG = (
    METRICS_ROOT
    / "EXP004_metric_evaluation_log.jsonl"
)


print(
    f"Ground truth root :\n  {TEST_ROOT}"
)

print(
    f"Prediction root   :\n  {PREDICTIONS_ROOT}"
)

print(
    f"Metrics root      :\n  {METRICS_ROOT}"
)

print(
    f"Per-source CSV    :\n  {PER_SOURCE_CSV}"
)

print(
    f"Per-track JSONL   :\n  {PER_TRACK_JSONL}"
)

print(
    f"Audit file        :\n  {EVALUATION_AUDIT_JSON}"
)


if not TEST_ROOT.is_dir():

    raise RuntimeError(
        f"MUSDB18 test root does not exist:\n{TEST_ROOT}"
    )


if not PREDICTIONS_ROOT.is_dir():

    raise RuntimeError(
        f"EXP004 prediction root does not exist:\n"
        f"{PREDICTIONS_ROOT}"
    )


# ============================================================================
# 4. AUDIO NORMALIZATION
# ============================================================================

print("\n" + "-" * 70)
print("AUDIO NORMALIZATION ENGINE")
print("-" * 70)


def normalize_audio_array(
    audio,
    *,
    expected_channels,
    context,
):
    """
    Normalize SoundFile output to:

        [samples, channels]

    with dtype:

        float32

    This function intentionally does NOT add a batch dimension.

    SoundFile with always_2d=True already returns:

        [samples, channels]

    The returned array is contiguous and finite.
    """

    array = np.asarray(
        audio
    )


    # --------------------------------------------------------------
    # Explicitly reject impossible dimensions.
    # --------------------------------------------------------------

    if array.ndim != 2:

        raise RuntimeError(
            f"{context}: expected 2-D audio array "
            f"[samples, channels], received "
            f"shape={array.shape}"
        )


    if array.shape[1] != expected_channels:

        raise RuntimeError(
            f"{context}: expected "
            f"{expected_channels} channels, "
            f"received shape={array.shape}"
        )


    # --------------------------------------------------------------
    # Convert only after dimensionality validation.
    # --------------------------------------------------------------

    array = np.asarray(
        array,
        dtype=np.float32,
    )


    array = np.ascontiguousarray(
        array,
        dtype=np.float32,
    )


    if not np.all(
        np.isfinite(array)
    ):

        raise RuntimeError(
            f"{context}: audio contains "
            "non-finite values."
        )


    if array.shape[0] <= 0:

        raise RuntimeError(
            f"{context}: audio contains no samples."
        )


    return array


def load_audio_for_evaluation(
    path,
    *,
    expected_sample_rate,
    expected_channels,
    context,
):
    """
    Load one WAV file using SoundFile and return:

        audio   -> [samples, channels], float32
        rate    -> int
        frames  -> int
    """

    try:

        audio, sample_rate = sf.read(
            path,
            dtype="float32",
            always_2d=True,
        )

    except Exception as exc:

        raise RuntimeError(
            f"{context}: could not decode:\n"
            f"{path}\n"
            f"{type(exc).__name__}: {exc}"
        ) from exc


    sample_rate = int(
        sample_rate
    )


    if sample_rate != expected_sample_rate:

        raise RuntimeError(
            f"{context}: sample-rate mismatch.\n"
            f"Path:     {path}\n"
            f"Expected: {expected_sample_rate}\n"
            f"Actual:   {sample_rate}"
        )


    audio = normalize_audio_array(
        audio,
        expected_channels=expected_channels,
        context=context,
    )


    return (
        audio,
        sample_rate,
        int(audio.shape[0]),
    )


print(
    "✓ Audio normalization explicitly produces "
    "[samples, channels] float32 arrays"
)

print(
    "✓ No accidental batch dimension is introduced"
)


# ============================================================================
# 5. METRIC SCALAR NORMALIZATION
# ============================================================================

def metric_scalar(
    value,
):
    """
    Convert a scalar / NumPy scalar / one-element array into
    a true Python float.

    Explicitly handles NumPy 2.x scalar conversion behavior.
    """

    array = np.asarray(
        value,
        dtype=np.float64,
    )


    if array.size != 1:

        raise RuntimeError(
            "Expected exactly one metric value, "
            f"received shape={array.shape}, "
            f"size={array.size}."
        )


    scalar = float(
        array.reshape(-1)[0]
    )


    if not math.isfinite(
        scalar
    ):

        raise RuntimeError(
            f"Metric result is non-finite: {scalar}"
        )


    return scalar


# ============================================================================
# 6. SI-SDR ENGINE
# ============================================================================

print("\n" + "-" * 70)
print("SI-SDR ENGINE")
print("-" * 70)


def calculate_si_sdr(
    reference,
    estimate,
):
    """
    Calculate scale-invariant SI-SDR for one source.

    Input:
        reference: [samples, channels]
        estimate : [samples, channels]

    TorchMetrics input:
        [batch, channels, time]

    Therefore:

        [samples, channels]
            .T
        [channels, samples]
            unsqueeze(0)
        [1, channels, samples]
    """

    reference = np.asarray(
        reference,
        dtype=np.float32,
    )


    estimate = np.asarray(
        estimate,
        dtype=np.float32,
    )


    if reference.ndim != 2:

        raise RuntimeError(
            "SI-SDR reference must be 2-D "
            "[samples, channels]. "
            f"Received {reference.shape}"
        )


    if estimate.ndim != 2:

        raise RuntimeError(
            "SI-SDR estimate must be 2-D "
            "[samples, channels]. "
            f"Received {estimate.shape}"
        )


    if reference.shape != estimate.shape:

        raise RuntimeError(
            "SI-SDR reference and estimate "
            "shapes must match.\n"
            f"Reference: {reference.shape}\n"
            f"Estimate:  {estimate.shape}"
        )


    reference_tensor = torch.from_numpy(
        np.ascontiguousarray(
            reference.T,
            dtype=np.float32,
        )
    ).unsqueeze(0)


    estimate_tensor = torch.from_numpy(
        np.ascontiguousarray(
            estimate.T,
            dtype=np.float32,
        )
    ).unsqueeze(0)


    # --------------------------------------------------------------
    # Explicit shape assertion before TorchMetrics.
    # --------------------------------------------------------------

    expected_tensor_shape = (
        1,
        reference.shape[1],
        reference.shape[0],
    )


    if tuple(
        reference_tensor.shape
    ) != expected_tensor_shape:

        raise RuntimeError(
            "SI-SDR reference tensor dimensionality error.\n"
            f"Expected: {expected_tensor_shape}\n"
            f"Actual:   {tuple(reference_tensor.shape)}"
        )


    if tuple(
        estimate_tensor.shape
    ) != expected_tensor_shape:

        raise RuntimeError(
            "SI-SDR estimate tensor dimensionality error.\n"
            f"Expected: {expected_tensor_shape}\n"
            f"Actual:   {tuple(estimate_tensor.shape)}"
        )


    metric = (
        ScaleInvariantSignalDistortionRatio()
    )


    with torch.no_grad():

        value = metric(
            estimate_tensor,
            reference_tensor,
        )


    return metric_scalar(
        value.detach().cpu().numpy()
    )


print(
    "✓ TorchMetrics ScaleInvariantSignalDistortionRatio configured"
)

print(
    "✓ SI-SDR is scale-invariant"
)


# ============================================================================
# 7. BSSEVAL V4 ENGINE
# ============================================================================

print("\n" + "-" * 70)
print("BSSEVAL V4 ENGINE")
print("-" * 70)


def calculate_bsseval_metrics(
    references,
    estimates,
):
    """
    Calculate SDR / SIR / SAR using the same BSSEval path
    established in EXP002 / EXP003.

    Required input:

        references.shape == (4, samples, channels)
        estimates.shape  == (4, samples, channels)

    Configuration:

        window=np.inf
        hop=np.inf
        compute_permutation=False
        filters_len=512
        framewise_filters=False
        bsseval_sources_version=False
    """

    references = np.asarray(
        references,
        dtype=np.float32,
    )


    estimates = np.asarray(
        estimates,
        dtype=np.float32,
    )


    if references.ndim != 3:

        raise RuntimeError(
            "BSSEval references must have shape "
            "(sources, samples, channels).\n"
            f"Received: {references.shape}"
        )


    if estimates.ndim != 3:

        raise RuntimeError(
            "BSSEval estimates must have shape "
            "(sources, samples, channels).\n"
            f"Received: {estimates.shape}"
        )


    if references.shape != estimates.shape:

        raise RuntimeError(
            "BSSEval reference and estimate arrays "
            "must have identical shapes.\n"
            f"Reference: {references.shape}\n"
            f"Estimate:  {estimates.shape}"
        )


    expected_shape = (
        EXPECTED_SOURCES_PER_TRACK,
        references.shape[1],
        MODEL_CHANNELS,
    )


    if references.shape != expected_shape:

        raise RuntimeError(
            "Unexpected BSSEval input shape.\n"
            f"Expected: {expected_shape}\n"
            f"Actual:   {references.shape}"
        )


    if not np.all(
        np.isfinite(references)
    ):

        raise RuntimeError(
            "BSSEval references contain non-finite values."
        )


    if not np.all(
        np.isfinite(estimates)
    ):

        raise RuntimeError(
            "BSSEval estimates contain non-finite values."
        )


    # ------------------------------------------------------------------
    # Canonical EXP002 / EXP003 BSSEval v4 call.
    # ------------------------------------------------------------------

    sdr, _, sir, sar, _ = bss_eval(
        references,
        estimates,
        window=np.inf,
        hop=np.inf,
        compute_permutation=False,
        filters_len=BSSEVAL_FILTERS_LEN,
        framewise_filters=False,
        bsseval_sources_version=False,
    )


    sdr = np.asarray(
        sdr,
        dtype=np.float64,
    ).reshape(-1)


    sir = np.asarray(
        sir,
        dtype=np.float64,
    ).reshape(-1)


    sar = np.asarray(
        sar,
        dtype=np.float64,
    ).reshape(-1)


    if sdr.size != EXPECTED_SOURCES_PER_TRACK:

        raise RuntimeError(
            "Unexpected SDR result length.\n"
            f"Shape: {sdr.shape}\n"
            f"Size:  {sdr.size}"
        )


    if sir.size != EXPECTED_SOURCES_PER_TRACK:

        raise RuntimeError(
            "Unexpected SIR result length.\n"
            f"Shape: {sir.shape}\n"
            f"Size:  {sir.size}"
        )


    if sar.size != EXPECTED_SOURCES_PER_TRACK:

        raise RuntimeError(
            "Unexpected SAR result length.\n"
            f"Shape: {sar.shape}\n"
            f"Size:  {sar.size}"
        )


    for metric_name, values in (
        ("SDR", sdr),
        ("SIR", sir),
        ("SAR", sar),
    ):

        if not np.all(
            np.isfinite(values)
        ):

            raise RuntimeError(
                f"BSSEval {metric_name} "
                "contains non-finite values."
            )


    return (
        sdr,
        sir,
        sar,
    )


print(
    "✓ BSSEval v4 engine configured"
)

print(
    "✓ Whole-track evaluation"
)

print(
    "✓ 512-tap filters"
)

print(
    "✓ Time-invariant filters"
)

print(
    "✓ Fixed source ordering"
)

print(
    "✓ Permutation search disabled"
)

print(
    "✓ ISR is not exposed to the EXP004 metric pipeline"
)


# ============================================================================
# 8. AUTHORITATIVE MUSDB18 TRACK DISCOVERY
# ============================================================================

print("\n" + "-" * 70)
print("AUTHORITATIVE MUSDB18 TRACK DISCOVERY")
print("-" * 70)


candidate_ground_truth_tracks = sorted(
    [
        path
        for path in TEST_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda path: path.name.casefold(),
)


ground_truth_tracks = []


for track_directory in (
    candidate_ground_truth_tracks
):

    mixture_path = (
        track_directory
        / "mixture.wav"
    )


    if mixture_path.is_file():

        ground_truth_tracks.append(
            track_directory
        )


if len(ground_truth_tracks) != (
    EXPECTED_TRACK_COUNT
):

    raise RuntimeError(
        "MUSDB18 ground-truth discovery does not "
        "match the expected 50-track evaluation set.\n"
        f"Expected: {EXPECTED_TRACK_COUNT}\n"
        f"Found:    {len(ground_truth_tracks)}"
    )


print(
    f"✓ {len(ground_truth_tracks)} "
    "ground-truth tracks discovered"
)


# ============================================================================
# 9. PREDICTION SET DISCOVERY
# ============================================================================

print("\n" + "-" * 70)
print("PREDICTION SET DISCOVERY")
print("-" * 70)


prediction_track_directories = {
    path.name: path
    for path in PREDICTIONS_ROOT.iterdir()
    if path.is_dir()
}


print(
    f"Prediction directories found: "
    f"{len(prediction_track_directories)}"
)


def inspect_prediction_set(
    prediction_directory,
):
    """
    Inspect a prediction directory without loading audio.

    Returns:

        {
            "complete": bool,
            "missing": [...],
            "paths": {...}
        }
    """

    paths = {
        source: (
            prediction_directory
            / f"{source}.wav"
        )
        for source in SOURCE_NAMES
    }


    missing = [
        source
        for source, path in paths.items()
        if not path.is_file()
    ]


    return {
        "complete": (
            len(missing) == 0
        ),
        "missing": missing,
        "paths": paths,
    }


complete_prediction_tracks = []
incomplete_prediction_tracks = []


for track_name, prediction_directory in sorted(
    prediction_track_directories.items(),
    key=lambda item: item[0].casefold(),
):

    inspection = inspect_prediction_set(
        prediction_directory
    )


    if inspection["complete"]:

        complete_prediction_tracks.append(
            track_name
        )

    else:

        incomplete_prediction_tracks.append(
            {
                "track": track_name,
                "missing": inspection["missing"],
            }
        )


print(
    f"Complete prediction sets   : "
    f"{len(complete_prediction_tracks)}"
)

print(
    f"Incomplete prediction sets : "
    f"{len(incomplete_prediction_tracks)}"
)


# ============================================================================
# 10. EXISTING RESULT RECOVERY
# ============================================================================

print("\n" + "-" * 70)
print("PERSISTENT RESULT RECOVERY")
print("-" * 70)


def atomic_write_text(
    path,
    text,
):
    """
    Atomically replace a text file.

    This prevents an interrupted write from leaving a partially
    written final JSONL/CSV file.
    """

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    fd, temporary_path = tempfile.mkstemp(
        prefix=f".{path.name}.",
        suffix=".tmp",
        dir=str(path.parent),
        text=True,
    )


    try:

        with os.fdopen(
            fd,
            "w",
            encoding="utf-8",
            newline="",
        ) as handle:

            handle.write(
                text
            )

            handle.flush()

            os.fsync(
                handle.fileno()
            )


        os.replace(
            temporary_path,
            path,
        )


    finally:

        if os.path.exists(
            temporary_path
        ):

            os.remove(
                temporary_path
            )


def load_existing_jsonl(
    path,
):
    """
    Recover valid completed track records.

    A malformed final line is ignored rather than allowing a
    previous interrupted write to destroy resumability.
    """

    records = {}


    if not path.is_file():

        return records


    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:

        for line_number, line in enumerate(
            handle,
            start=1,
        ):

            line = line.strip()


            if not line:

                continue


            try:

                record = json.loads(
                    line
                )

            except json.JSONDecodeError:

                print(
                    f"⚠ Ignoring malformed JSONL "
                    f"line {line_number} during recovery."
                )

                continue


            if (
                record.get("experiment")
                != EXPERIMENT_NAME
            ):

                continue


            track_name = record.get(
                "track"
            )


            if not track_name:

                continue


            if record.get(
                "status"
            ) != "success":

                continue


            if not (
                isinstance(
                    record.get("sources"),
                    dict,
                )
            ):

                continue


            if set(
                record["sources"].keys()
            ) != set(
                SOURCE_NAMES
            ):

                continue


            track_average = record.get(
                "track_average"
            )


            if not (
                isinstance(
                    track_average,
                    dict,
                )
            ):

                continue


            if set(
                track_average.keys()
            ) != set(
                APPROVED_METRICS
            ):

                continue


            # ----------------------------------------------------------
            # Validate every persisted metric as a finite scalar.
            # ----------------------------------------------------------

            valid_record = True


            for source in SOURCE_NAMES:

                source_record = record[
                    "sources"
                ].get(
                    source
                )


                if not isinstance(
                    source_record,
                    dict,
                ):

                    valid_record = False

                    break


                expected_metric_keys = {
                    "SDR_dB",
                    "SIR_dB",
                    "SAR_dB",
                    "SI_SDR_dB",
                }


                if set(
                    source_record.keys()
                ) != expected_metric_keys:

                    valid_record = False

                    break


                for value in (
                    source_record.values()
                ):

                    try:

                        numeric_value = float(
                            value
                        )

                    except Exception:

                        valid_record = False

                        break


                    if not math.isfinite(
                        numeric_value
                    ):

                        valid_record = False

                        break


                if not valid_record:

                    break


            if not valid_record:

                continue


            records[
                track_name
            ] = record


    return records


existing_records = load_existing_jsonl(
    PER_TRACK_JSONL
)


print(
    f"Recovered completed tracks: "
    f"{len(existing_records)}"
)


if existing_records:

    for track_name in sorted(
        existing_records,
        key=str.casefold,
    ):

        print(
            f"  ✓ already evaluated: "
            f"{track_name}"
        )

else:

    print(
        "No previous completed metric records found."
    )


# ============================================================================
# 11. REBUILD PERSISTENT CSV FROM RECOVERED RESULTS
# ============================================================================

def build_csv_rows(
    records,
):
    """
    Convert nested per-track records into the canonical
    per-source metric table used by EXP003-style analysis.
    """

    rows = []


    for track_name in sorted(
        records,
        key=str.casefold,
    ):

        record = records[
            track_name
        ]


        for source in SOURCE_NAMES:

            source_metrics = record[
                "sources"
            ][source]


            rows.append(
                {
                    "experiment":
                        EXPERIMENT_NAME,

                    "track":
                        track_name,

                    "source":
                        source,

                    "sample_rate":
                        int(
                            record[
                                "sample_rate"
                            ]
                        ),

                    "num_samples":
                        int(
                            record[
                                "num_samples"
                            ]
                        ),

                    "num_channels":
                        int(
                            record[
                                "num_channels"
                            ]
                        ),

                    "SDR":
                        float(
                            source_metrics[
                                "SDR_dB"
                            ]
                        ),

                    "SIR":
                        float(
                            source_metrics[
                                "SIR_dB"
                            ]
                        ),

                    "SAR":
                        float(
                            source_metrics[
                                "SAR_dB"
                            ]
                        ),

                    "SI-SDR":
                        float(
                            source_metrics[
                                "SI_SDR_dB"
                            ]
                        ),
                }
            )


    return rows


def write_per_source_csv(
    records,
):
    """
    Rewrite the canonical per-source CSV atomically from
    the complete persistent JSONL result set.

    This is inexpensive compared with metric computation.
    """

    fieldnames = [
        "experiment",
        "track",
        "source",
        "sample_rate",
        "num_samples",
        "num_channels",
        "SDR",
        "SIR",
        "SAR",
        "SI-SDR",
    ]


    rows = build_csv_rows(
        records
    )


    temporary_lines = []

    output = (
        __import__(
            "io"
        ).StringIO()
    )


    writer = csv.DictWriter(
        output,
        fieldnames=fieldnames,
    )


    writer.writeheader()


    for row in rows:

        writer.writerow(
            row
        )


    atomic_write_text(
        PER_SOURCE_CSV,
        output.getvalue(),
    )


    return len(rows)


def write_jsonl_records(
    records,
):
    """
    Rewrite the persistent JSONL file atomically.

    The JSONL contains one complete record per successfully
    evaluated track.
    """

    lines = []


    for track_name in sorted(
        records,
        key=str.casefold,
    ):

        record = records[
            track_name
        ]


        lines.append(
            json.dumps(
                record,
                ensure_ascii=False,
                allow_nan=False,
                separators=(
                    ",",
                    ":",
                ),
            )
            + "\n"
        )


    atomic_write_text(
        PER_TRACK_JSONL,
        "".join(
            lines
        ),
    )


# ============================================================================
# 12. TRACK METRIC COMPUTATION
# ============================================================================

def evaluate_single_track(
    track_index,
    ground_truth_dir,
    prediction_dir,
):
    """
    Evaluate one complete track.

    Returns one complete persistent track record.

    No result is returned until ALL four source metrics have been
    successfully calculated.
    """

    track_name = (
        ground_truth_dir.name
    )


    track_start = time.time()


    print("\n" + "-" * 70)

    print(
        f"TRACK {track_index:02d}/"
        f"{EXPECTED_TRACK_COUNT:02d}: "
        f"{track_name}"
    )

    print("-" * 70)


    # ------------------------------------------------------------------
    # Load all four reference and prediction stems.
    # ------------------------------------------------------------------

    reference_sources = []
    estimate_sources = []

    sample_rates = []
    sample_counts = []


    for source_name in SOURCE_NAMES:

        reference_path = (
            ground_truth_dir
            / f"{source_name}.wav"
        )

        prediction_path = (
            prediction_dir
            / f"{source_name}.wav"
        )


        reference_audio, reference_rate, reference_frames = (
            load_audio_for_evaluation(
                reference_path,
                expected_sample_rate=MODEL_SAMPLE_RATE,
                expected_channels=MODEL_CHANNELS,
                context=(
                    f"{track_name} reference "
                    f"{source_name}"
                ),
            )
        )


        prediction_audio, prediction_rate, prediction_frames = (
            load_audio_for_evaluation(
                prediction_path,
                expected_sample_rate=MODEL_SAMPLE_RATE,
                expected_channels=MODEL_CHANNELS,
                context=(
                    f"{track_name} prediction "
                    f"{source_name}"
                ),
            )
        )


        if reference_rate != prediction_rate:

            raise RuntimeError(
                f"{track_name} / {source_name}: "
                "reference and prediction "
                "sample rates differ."
            )


        if reference_frames != prediction_frames:

            raise RuntimeError(
                f"{track_name} / {source_name}: "
                "reference and prediction "
                "sample counts differ.\n"
                f"Reference: {reference_frames}\n"
                f"Prediction: {prediction_frames}"
            )


        if reference_audio.shape != (
            prediction_audio.shape
        ):

            raise RuntimeError(
                f"{track_name} / {source_name}: "
                "reference and prediction "
                "array shapes differ.\n"
                f"Reference: {reference_audio.shape}\n"
                f"Prediction: {prediction_audio.shape}"
            )


        reference_sources.append(
            reference_audio
        )

        estimate_sources.append(
            prediction_audio
        )

        sample_rates.append(
            reference_rate
        )

        sample_counts.append(
            reference_frames
        )


    if len(
        set(sample_rates)
    ) != 1:

        raise RuntimeError(
            f"{track_name}: source sample rates "
            "are not internally consistent."
        )


    if len(
        set(sample_counts)
    ) != 1:

        raise RuntimeError(
            f"{track_name}: source sample counts "
            "are not internally consistent."
        )


    track_sample_rate = int(
        sample_rates[0]
    )

    track_sample_count = int(
        sample_counts[0]
    )


    # ------------------------------------------------------------------
    # Stack into canonical BSSEval representation:
    #
    #     [sources, samples, channels]
    # ------------------------------------------------------------------

    references = np.stack(
        reference_sources,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    estimates = np.stack(
        estimate_sources,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    expected_stack_shape = (
        EXPECTED_SOURCES_PER_TRACK,
        track_sample_count,
        MODEL_CHANNELS,
    )


    if references.shape != (
        expected_stack_shape
    ):

        raise RuntimeError(
            f"{track_name}: reference stack "
            "shape mismatch.\n"
            f"Expected: {expected_stack_shape}\n"
            f"Actual:   {references.shape}"
        )


    if estimates.shape != (
        expected_stack_shape
    ):

        raise RuntimeError(
            f"{track_name}: estimate stack "
            "shape mismatch.\n"
            f"Expected: {expected_stack_shape}\n"
            f"Actual:   {estimates.shape}"
        )


    print(
        f"Reference array shape: "
        f"{references.shape}"
    )

    print(
        f"Prediction array shape: "
        f"{estimates.shape}"
    )


    # ------------------------------------------------------------------
    # BSSEval
    # ------------------------------------------------------------------

    print(
        "\nCalculating BSSEval v4 "
        "(SDR / SIR / SAR)..."
    )


    sdr, sir, sar = (
        calculate_bsseval_metrics(
            references,
            estimates,
        )
    )


    print(
        "✓ BSSEval calculation complete"
    )


    # ------------------------------------------------------------------
    # SI-SDR
    # ------------------------------------------------------------------

    print(
        "Calculating SI-SDR..."
    )


    si_sdr = np.zeros(
        EXPECTED_SOURCES_PER_TRACK,
        dtype=np.float64,
    )


    for source_index, source_name in enumerate(
        SOURCE_NAMES
    ):

        si_sdr[
            source_index
        ] = calculate_si_sdr(
            reference_sources[
                source_index
            ],
            estimate_sources[
                source_index
            ],
        )


    if not np.all(
        np.isfinite(si_sdr)
    ):

        raise RuntimeError(
            f"{track_name}: SI-SDR produced "
            "non-finite values."
        )


    print(
        "✓ SI-SDR calculation complete"
    )


    # ------------------------------------------------------------------
    # Construct persistent source records.
    # ------------------------------------------------------------------

    sources_object = {}


    for source_index, source_name in enumerate(
        SOURCE_NAMES
    ):

        sources_object[
            source_name
        ] = {
            "SDR_dB":
                float(
                    sdr[
                        source_index
                    ]
                ),

            "SIR_dB":
                float(
                    sir[
                        source_index
                    ]
                ),

            "SAR_dB":
                float(
                    sar[
                        source_index
                    ]
                ),

            "SI_SDR_dB":
                float(
                    si_sdr[
                        source_index
                    ]
                ),
        }


    # ------------------------------------------------------------------
    # Track-level averages.
    #
    # This follows the EXP003-compatible structure.
    # ------------------------------------------------------------------

    track_average = {
        "SDR":
            float(
                np.mean(
                    sdr
                )
            ),

        "SIR":
            float(
                np.mean(
                    sir
                )
            ),

        "SAR":
            float(
                np.mean(
                    sar
                )
            ),

        "SI-SDR":
            float(
                np.mean(
                    si_sdr
                )
            ),
    }


    # ------------------------------------------------------------------
    # Final finite-value validation BEFORE persistence.
    # ------------------------------------------------------------------

    for metric_name, values in (
        ("SDR", sdr),
        ("SIR", sir),
        ("SAR", sar),
        ("SI-SDR", si_sdr),
    ):

        if not np.all(
            np.isfinite(values)
        ):

            raise RuntimeError(
                f"{track_name}: "
                f"{metric_name} contains "
                "non-finite values."
            )


    elapsed_seconds = (
        time.time()
        - track_start
    )


    record = {
        "experiment":
            EXPERIMENT_NAME,

        "track":
            track_name,

        "status":
            "success",

        "sample_rate":
            track_sample_rate,

        "num_samples":
            track_sample_count,

        "num_channels":
            MODEL_CHANNELS,

        "sources":
            sources_object,

        "track_average":
            track_average,

        "evaluation_configuration": {
            "bsseval":
                "BSSEval v4",

            "window":
                "whole_track",

            "hop":
                "whole_track",

            "filters_len":
                BSSEVAL_FILTERS_LEN,

            "framewise_filters":
                False,

            "compute_permutation":
                False,

            "si_sdr":
                "scale_invariant",

            "excluded_metrics":
                list(
                    EXCLUDED_METRICS
                ),
        },

        "elapsed_seconds":
            float(
                elapsed_seconds
            ),
    }


    # ------------------------------------------------------------------
    # Release large arrays before returning.
    # ------------------------------------------------------------------

    del references
    del estimates
    del reference_sources
    del estimate_sources


    gc.collect()


    print(
        "\nMetric results:"
    )


    for source_name in SOURCE_NAMES:

        source_metrics = sources_object[
            source_name
        ]

        print(
            f"  {source_name:<7} "
            f"SDR={source_metrics['SDR_dB']:8.3f} "
            f"SIR={source_metrics['SIR_dB']:8.3f} "
            f"SAR={source_metrics['SAR_dB']:8.3f} "
            f"SI-SDR={source_metrics['SI_SDR_dB']:8.3f}"
        )


    print(
        f"\n✓ TRACK EVALUATION COMPLETE"
    )

    print(
        f"  Time: "
        f"{elapsed_seconds / 60:.2f} minutes"
    )


    return record


# ============================================================================
# 13. INITIAL RESUMABILITY PLAN
# ============================================================================

print("\n" + "-" * 70)
print("RESUMABILITY PLAN")
print("-" * 70)


ground_truth_names = {
    path.name
    for path in ground_truth_tracks
}


# ---------------------------------------------------------------------------
# Remove stale persisted records for tracks no longer present in the
# authoritative MUSDB18 test root.
# ---------------------------------------------------------------------------

stale_records = [
    track_name
    for track_name in existing_records
    if track_name not in ground_truth_names
]


if stale_records:

    print(
        "⚠ Removing stale persisted metric records:"
    )

    for track_name in stale_records:

        print(
            f"  - {track_name}"
        )

        del existing_records[
            track_name
        ]


# ---------------------------------------------------------------------------
# Existing records are only reusable if the prediction set currently
# exists completely.
#
# This is important if a previously evaluated prediction directory
# has subsequently become incomplete.
# ---------------------------------------------------------------------------

invalidated_existing_records = []


for track_name in list(
    existing_records.keys()
):

    prediction_directory = (
        PREDICTIONS_ROOT
        / track_name
    )


    inspection = inspect_prediction_set(
        prediction_directory
    )


    if not inspection["complete"]:

        invalidated_existing_records.append(
            {
                "track":
                    track_name,

                "missing":
                    inspection["missing"],
            }
        )


        del existing_records[
            track_name
        ]


if invalidated_existing_records:

    print(
        "\n⚠ Existing metric records invalidated "
        "because their prediction sets are now incomplete:"
    )


    for item in (
        invalidated_existing_records
    ):

        print(
            f"  - {item['track']} "
            f"(missing: "
            f"{', '.join(item['missing'])})"
        )


# ---------------------------------------------------------------------------
# Persist the recovered/cleaned state immediately.
# ---------------------------------------------------------------------------

write_jsonl_records(
    existing_records
)


write_per_source_csv(
    existing_records
)


print(
    "\n✓ Persistent metric state synchronized."
)


# ============================================================================
# 14. BUILD INFERENCE-FREE EVALUATION SCOPE
# ============================================================================

print("\n" + "-" * 70)
print("EVALUATION SCOPE")
print("-" * 70)


tracks_to_skip_as_completed = set(
    existing_records.keys()
)


tracks_to_evaluate = []
tracks_to_skip_incomplete = []


for ground_truth_dir in (
    ground_truth_tracks
):

    track_name = (
        ground_truth_dir.name
    )


    prediction_directory = (
        PREDICTIONS_ROOT
        / track_name
    )


    inspection = inspect_prediction_set(
        prediction_directory
    )


    if track_name in (
        tracks_to_skip_as_completed
    ):

        print(
            f"SKIP | already evaluated | "
            f"{track_name}"
        )

        continue


    if not inspection["complete"]:

        tracks_to_skip_incomplete.append(
            {
                "track":
                    track_name,

                "missing":
                    inspection["missing"],
            }
        )


        print(
            f"SKIP | incomplete predictions | "
            f"{track_name} | "
            f"missing: "
            f"{', '.join(inspection['missing'])}"
        )

        continue


    tracks_to_evaluate.append(
        (
            ground_truth_dir,
            prediction_directory,
        )
    )


print(
    "\nEvaluation scope summary:"
)

print(
    f"  Already completed : "
    f"{len(tracks_to_skip_as_completed)}"
)

print(
    f"  Incomplete/skipped: "
    f"{len(tracks_to_skip_incomplete)}"
)

print(
    f"  New evaluation    : "
    f"{len(tracks_to_evaluate)}"
)


# ============================================================================
# 15. FULL RESUMABLE OBJECTIVE EVALUATION
# ============================================================================

print("\n")
print("=" * 70)
print("STARTING EXP004 RESUMABLE OBJECTIVE EVALUATION")
print("=" * 70)


overall_start_time = time.time()


evaluation_failures = []


for evaluation_number, (
    ground_truth_dir,
    prediction_directory,
) in enumerate(
    tracks_to_evaluate,
    start=1,
):

    track_name = (
        ground_truth_dir.name
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        f"NEW EVALUATION "
        f"{evaluation_number}/"
        f"{len(tracks_to_evaluate)}"
    )

    print(
        f"Track: {track_name}"
    )

    print(
        "=" * 70
    )


    try:

        result = evaluate_single_track(
            evaluation_number,
            ground_truth_dir,
            prediction_directory,
        )


        # --------------------------------------------------------------
        # CRITICAL RESUMABILITY POINT
        #
        # The result is inserted into the persistent registry BEFORE
        # moving to the next track.
        # --------------------------------------------------------------

        existing_records[
            track_name
        ] = result


        # --------------------------------------------------------------
        # Persist JSONL and CSV immediately after EVERY track.
        # --------------------------------------------------------------

        write_jsonl_records(
            existing_records
        )


        write_per_source_csv(
            existing_records
        )


        print(
            "\n✓ Persistent metric checkpoint written."
        )

        print(
            f"  Track: {track_name}"
        )

        print(
            f"  Completed tracks persisted: "
            f"{len(existing_records)}"
        )


        # --------------------------------------------------------------
        # Explicit memory cleanup.
        # --------------------------------------------------------------

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    except Exception as exc:

        failure_record = {
            "experiment":
                EXPERIMENT_NAME,

            "track":
                track_name,

            "status":
                "failed",

            "error_type":
                type(
                    exc
                ).__name__,

            "error_message":
                str(
                    exc
                ),

            "traceback":
                traceback.format_exc(),

            "timestamp_utc":
                datetime.utcnow().isoformat()
                + "Z",
        }


        evaluation_failures.append(
            failure_record
        )


        # --------------------------------------------------------------
        # IMPORTANT:
        # A metric failure does NOT corrupt previously persisted results.
        #
        # The track remains unevaluated and will be retried when Cell 15
        # is run again.
        # --------------------------------------------------------------

        print(
            "\n"
            + "!" * 70
        )

        print(
            f"TRACK METRIC EVALUATION FAILED: "
            f"{track_name}"
        )

        print(
            "!" * 70
        )

        print(
            f"Error: "
            f"{type(exc).__name__}: {exc}"
        )

        print(
            "\nThe track was NOT marked as completed."
        )

        print(
            "Previously persisted tracks remain safe."
        )

        print(
            "This track will be retried on the next "
            "Cell 15 execution."
        )


# ============================================================================
# 16. FINAL PERSISTENT STATE SYNCHRONIZATION
# ============================================================================

write_jsonl_records(
    existing_records
)


actual_source_rows = (
    write_per_source_csv(
        existing_records
    )
)


overall_elapsed = (
    time.time()
    - overall_start_time
)


# ============================================================================
# 17. POST-EVALUATION FILESYSTEM VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("POST-EVALUATION METRIC VALIDATION")
print("-" * 70)


final_records = load_existing_jsonl(
    PER_TRACK_JSONL
)


# ---------------------------------------------------------------------------
# Track-level validation
# ---------------------------------------------------------------------------

validated_track_names = set(
    final_records.keys()
)


invalid_persisted_records = []


for track_name, record in (
    final_records.items()
):

    if record.get(
        "status"
    ) != "success":

        invalid_persisted_records.append(
            track_name
        )

        continue


    if set(
        record["sources"].keys()
    ) != set(
        SOURCE_NAMES
    ):

        invalid_persisted_records.append(
            track_name
        )

        continue


    if set(
        record["track_average"].keys()
    ) != set(
        APPROVED_METRICS
    ):

        invalid_persisted_records.append(
            track_name
        )

        continue


    for source_name in (
        SOURCE_NAMES
    ):

        source_metrics = record[
            "sources"
        ][source_name]


        for key in (
            "SDR_dB",
            "SIR_dB",
            "SAR_dB",
            "SI_SDR_dB",
        ):

            value = float(
                source_metrics[key]
            )


            if not math.isfinite(
                value
            ):

                invalid_persisted_records.append(
                    track_name
                )


if invalid_persisted_records:

    raise RuntimeError(
        "Invalid persisted metric records detected:\n"
        + "\n".join(
            f"  - {track}"
            for track in sorted(
                set(
                    invalid_persisted_records
                ),
                key=str.casefold,
            )
        )
    )


print(
    f"✓ {len(final_records)} persisted "
    "track records are structurally valid"
)


# ============================================================================
# 18. RECHECK CURRENT PREDICTION COMPLETENESS
# ============================================================================

current_complete_prediction_tracks = []
current_incomplete_prediction_tracks = []


for ground_truth_dir in (
    ground_truth_tracks
):

    track_name = (
        ground_truth_dir.name
    )


    prediction_directory = (
        PREDICTIONS_ROOT
        / track_name
    )


    inspection = inspect_prediction_set(
        prediction_directory
    )


    if inspection["complete"]:

        current_complete_prediction_tracks.append(
            track_name
        )

    else:

        current_incomplete_prediction_tracks.append(
            {
                "track":
                    track_name,

                "missing":
                    inspection["missing"],
            }
        )


# ============================================================================
# 19. BUILD FINAL AUDIT
# ============================================================================

complete_evaluated_tracks = (
    len(
        final_records
    )
)


complete_source_evaluations = (
    complete_evaluated_tracks
    * EXPECTED_SOURCES_PER_TRACK
)


expected_evaluation_tracks = (
    len(
        current_complete_prediction_tracks
    )
)


pending_complete_tracks = [
    track_name
    for track_name in (
        current_complete_prediction_tracks
    )
    if track_name not in final_records
]


metric_audit = {
    "experiment":
        EXPERIMENT_NAME,

    "timestamp_utc":
        datetime.utcnow().isoformat()
        + "Z",

    "expected_musdb18_tracks":
        EXPECTED_TRACK_COUNT,

    "discovered_musdb18_tracks":
        len(
            ground_truth_tracks
        ),

    "sources":
        list(
            SOURCE_NAMES
        ),

    "expected_sources_per_track":
        EXPECTED_SOURCES_PER_TRACK,

    "approved_metrics":
        list(
            APPROVED_METRICS
        ),

    "excluded_metrics":
        list(
            EXCLUDED_METRICS
        ),

    "bsseval": {
        "version":
            "BSSEval v4",

        "window":
            "whole_track",

        "window_value":
            "np.inf",

        "hop_value":
            "np.inf",

        "filters_len":
            BSSEVAL_FILTERS_LEN,

        "framewise_filters":
            False,

        "compute_permutation":
            False,
    },

    "si_sdr": {
        "implementation":
            "torchmetrics.audio.ScaleInvariantSignalDistortionRatio",

        "scale_invariant":
            True,
    },

    "prediction_root":
        str(
            PREDICTIONS_ROOT
        ),

    "ground_truth_root":
        str(
            TEST_ROOT
        ),

    "metrics_root":
        str(
            METRICS_ROOT
        ),

    "complete_prediction_track_count":
        expected_evaluation_tracks,

    "incomplete_prediction_track_count":
        len(
            current_incomplete_prediction_tracks
        ),

    "incomplete_prediction_tracks":
        current_incomplete_prediction_tracks,

    "evaluated_track_count":
        complete_evaluated_tracks,

    "pending_complete_track_count":
        len(
            pending_complete_tracks
        ),

    "pending_complete_tracks":
        sorted(
            pending_complete_tracks,
            key=str.casefold,
        ),

    "source_evaluation_count":
        complete_source_evaluations,

    "expected_source_evaluation_count_if_all_50_complete":
        EXPECTED_SOURCE_EVALUATIONS,

    "expected_source_evaluation_count_currently_complete":
        (
            expected_evaluation_tracks
            * EXPECTED_SOURCES_PER_TRACK
        ),

    "csv_source_rows":
        actual_source_rows,

    "evaluation_failures_this_run":
        evaluation_failures,

    "total_evaluation_seconds_this_run":
        overall_elapsed,

    "status":
        (
            "COMPLETE"
            if (
                complete_evaluated_tracks
                == EXPECTED_TRACK_COUNT
                and not current_incomplete_prediction_tracks
                and not pending_complete_tracks
                and not evaluation_failures
            )
            else "INCOMPLETE"
        ),

    "resumable":
        True,

    "ht_demucs_inference_performed":
        False,
}


atomic_write_text(
    EVALUATION_AUDIT_JSON,
    json.dumps(
        metric_audit,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
)


# ============================================================================
# 20. FINAL RESULT
# ============================================================================

print("\n")
print("=" * 70)
print("EXP004 OBJECTIVE METRIC EVALUATION RESULT")
print("=" * 70)


print(
    f"Ground-truth tracks discovered : "
    f"{len(ground_truth_tracks)}"
)


print(
    f"Complete prediction tracks     : "
    f"{expected_evaluation_tracks}"
)


print(
    f"Incomplete prediction tracks   : "
    f"{len(current_incomplete_prediction_tracks)}"
)


print(
    f"Metric tracks completed        : "
    f"{complete_evaluated_tracks}"
)


print(
    f"Pending complete tracks         : "
    f"{len(pending_complete_tracks)}"
)


print(
    f"Source evaluations completed   : "
    f"{complete_source_evaluations}"
)


print(
    f"Expected at 50 tracks          : "
    f"{EXPECTED_SOURCE_EVALUATIONS}"
)


print(
    f"Per-source CSV rows            : "
    f"{actual_source_rows}"
)


print(
    f"Metric failures this run      : "
    f"{len(evaluation_failures)}"
)


print(
    "\nApproved metrics:"
)

for metric_name in APPROVED_METRICS:

    print(
        f"  ✓ {metric_name}"
    )


print(
    "\nExcluded:"
)

for metric_name in EXCLUDED_METRICS:

    print(
        f"  ✗ {metric_name}"
    )


print(
    "\nBSSEval:"
)

print(
    "  ✓ BSSEval v4"
)

print(
    "  ✓ Whole-track"
)

print(
    "  ✓ 512-tap filters"
)

print(
    "  ✓ Time-invariant filters"
)

print(
    "  ✓ Fixed source ordering"
)

print(
    "  ✓ Permutation search disabled"
)


print(
    "\nSI-SDR:"
)

print(
    "  ✓ TorchMetrics"
)

print(
    "  ✓ Scale-invariant"
)


print(
    "\nPersistence:"
)

print(
    f"  ✓ {PER_TRACK_JSONL}"
)

print(
    f"  ✓ {PER_SOURCE_CSV}"
)

print(
    f"  ✓ {EVALUATION_AUDIT_JSON}"
)


# ---------------------------------------------------------------------------
# Explicit incomplete-track report.
# ---------------------------------------------------------------------------

if current_incomplete_prediction_tracks:

    print(
        "\nIncomplete prediction sets "
        "(automatically skipped):"
    )


    for item in (
        current_incomplete_prediction_tracks
    ):

        print(
            f"  - {item['track']}"
            f" | missing: "
            f"{', '.join(item['missing'])}"
        )


# ---------------------------------------------------------------------------
# Explicit pending-track report.
# ---------------------------------------------------------------------------

if pending_complete_tracks:

    print(
        "\nComplete prediction sets "
        "awaiting metric evaluation:"
    )


    for track_name in sorted(
        pending_complete_tracks,
        key=str.casefold,
    ):

        print(
            f"  - {track_name}"
        )


# ---------------------------------------------------------------------------
# Final status.
# ---------------------------------------------------------------------------

if metric_audit["status"] == "COMPLETE":

    print(
        "\n✓ ALL 50 TRACKS HAVE COMPLETE "
        "OBJECTIVE METRICS"
    )

    print(
        "✓ ALL 200 SOURCE EVALUATIONS COMPLETE"
    )

    print(
        "✓ EXP004 OBJECTIVE EVALUATION COMPLETE"
    )


else:

    print(
        "\n⚠ EXP004 objective evaluation "
        "is currently incomplete."
    )

    print(
        "This is expected when prediction sets "
        "are incomplete."
    )

    print(
        "Run Cell 15 again after missing "
        "prediction WAVs become available."
    )


print(
    "\n✓ Results are persisted after every "
    "successful track."
)

print(
    "✓ Cell 15 is resumable."
)

print(
    "✓ No HT-Demucs inference is performed "
    "by this cell."
)

print(
    "✓ No objective metric is recomputed for "
    "already persisted successful tracks."
)


print(
    "\n"
    + "=" * 70
)

print(
    "✓ CELL 15 COMPLETE"
)

print(
    "=" * 70
)

EXP004 RESUMABLE OBJECTIVE METRIC EVALUATION

----------------------------------------------------------------------
PERSISTENT REGISTRY VALIDATION
----------------------------------------------------------------------
✓ EXP004_PATHS registry available
✓ MUSDB18_TRACKS registry available
✓ MUSDB18_AUDIO_METADATA registry available
✓ EXP004_MODEL_CONFIG registry available
✓ PREDICTION_AUDIT registry available

----------------------------------------------------------------------
EXP004 EVALUATION CONFIGURATION
----------------------------------------------------------------------
Experiment                  : EXP004
Expected tracks             : 50
Sources                     : ['vocals', 'drums', 'bass', 'other']
Expected source evaluations : 200
Sample rate                 : 44100
Channels                    : 2
BSSEval window              : whole_track
BSSEval filter length      : 512
Framewise filters           : False
Permutation search          : False
Approved metrics           

C:\Users\SARANGA PANI DUTTA\AppData\Local\Temp\ipykernel_26432\3572981128.py:2815: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat()


In [ ]:
# ============================================================================
# EXP004 — PREDICTION AUDIO DIAGNOSTIC
# ============================================================================
#
# Purpose
# -------
# Diagnose prediction WAV integrity BEFORE resuming objective metrics.
#
# IMPORTANT
# ---------
# 1. NO HT-Demucs inference.
# 2. NO SDR/SIR/SAR/SI-SDR computation.
# 3. NO audio arrays are loaded into memory.
# 4. Only SoundFile metadata is inspected.
# 5. Existing persisted metric results are not modified.
# 6. The diagnostic checks ALL 50 prediction directories.
#
# This specifically detects the problem encountered with:
#
#     Skelpolu - Resurrection / drums.wav
#
# where the WAV file exists but contains zero samples.
#
# ============================================================================

from pathlib import Path
import json
import math
import soundfile as sf


print("=" * 70)
print("EXP004 PREDICTION AUDIO INTEGRITY DIAGNOSTIC")
print("=" * 70)


# ============================================================================
# 1. REQUIRED REGISTRY VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("PERSISTENT REGISTRY VALIDATION")
print("-" * 70)


required_registries = [
    "EXP004_PATHS",
    "MUSDB18_TRACKS",
    "MUSDB18_AUDIO_METADATA",
    "EXP004_MODEL_CONFIG",
    "PREDICTION_AUDIT",
]


missing_registries = [
    name
    for name in required_registries
    if name not in globals()
]


if missing_registries:

    raise RuntimeError(
        "Required persistent EXP004 registries are unavailable:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_registries
        )
    )


for name in required_registries:

    print(f"✓ {name} registry available")


# ============================================================================
# 2. CONFIGURATION
# ============================================================================

print("\n" + "-" * 70)
print("DIAGNOSTIC CONFIGURATION")
print("-" * 70)


EXPECTED_SOURCES = [
    "vocals",
    "drums",
    "bass",
    "other",
]


EXPECTED_TRACKS = int(
    EXP004_MODEL_CONFIG.get(
        "expected_tracks",
        50,
    )
)


EXPECTED_SAMPLE_RATE = int(
    EXP004_MODEL_CONFIG.get(
        "sample_rate",
        44100,
    )
)


EXPECTED_CHANNELS = int(
    EXP004_MODEL_CONFIG.get(
        "audio_channels",
        2,
    )
)


TEST_ROOT = Path(
    EXP004_PATHS[
        "musdb18_test_root"
    ]
)


PREDICTIONS_ROOT = Path(
    EXP004_PATHS[
        "objective_predictions"
    ]
)


print(
    f"Expected tracks       : {EXPECTED_TRACKS}"
)

print(
    f"Expected sources      : {EXPECTED_SOURCES}"
)

print(
    f"Expected sample rate  : {EXPECTED_SAMPLE_RATE}"
)

print(
    f"Expected channels     : {EXPECTED_CHANNELS}"
)

print(
    f"Ground-truth root     : {TEST_ROOT}"
)

print(
    f"Prediction root       : {PREDICTIONS_ROOT}"
)


if not TEST_ROOT.is_dir():

    raise RuntimeError(
        f"Ground-truth root does not exist:\n{TEST_ROOT}"
    )


if not PREDICTIONS_ROOT.is_dir():

    raise RuntimeError(
        f"Prediction root does not exist:\n{PREDICTIONS_ROOT}"
    )


# ============================================================================
# 3. AUTHORITATIVE TRACK DISCOVERY
# ============================================================================

print("\n" + "-" * 70)
print("AUTHORITATIVE TRACK DISCOVERY")
print("-" * 70)


ground_truth_tracks = sorted(
    [
        path
        for path in TEST_ROOT.iterdir()
        if path.is_dir()
        and (path / "mixture.wav").is_file()
    ],
    key=lambda path: path.name.casefold(),
)


if len(ground_truth_tracks) != EXPECTED_TRACKS:

    raise RuntimeError(
        "Unexpected MUSDB18 ground-truth track count.\n"
        f"Expected: {EXPECTED_TRACKS}\n"
        f"Found:    {len(ground_truth_tracks)}"
    )


print(
    f"✓ {len(ground_truth_tracks)} ground-truth tracks discovered"
)


# ============================================================================
# 4. DIAGNOSTIC RESULT CONTAINERS
# ============================================================================

diagnostic_records = []

complete_prediction_tracks = []

invalid_prediction_tracks = []

missing_prediction_tracks = []

zero_sample_files = []

metadata_mismatch_files = []

prediction_open_failures = []


# ============================================================================
# 5. INSPECT ALL 50 PREDICTION SETS
# ============================================================================

print("\n" + "-" * 70)
print("PREDICTION WAV METADATA INSPECTION")
print("-" * 70)


for track_index, ground_truth_dir in enumerate(
    ground_truth_tracks,
    start=1,
):

    track_name = ground_truth_dir.name

    prediction_dir = (
        PREDICTIONS_ROOT
        / track_name
    )


    print(
        f"\n[{track_index:02d}/{EXPECTED_TRACKS:02d}] "
        f"{track_name}"
    )


    track_record = {
        "track_name": track_name,
        "ground_truth_directory": str(
            ground_truth_dir
        ),
        "prediction_directory": str(
            prediction_dir
        ),
        "sources": {},
        "complete": True,
        "problems": [],
    }


    if not prediction_dir.is_dir():

        track_record["complete"] = False

        track_record["problems"].append(
            "MISSING_PREDICTION_DIRECTORY"
        )

        missing_prediction_tracks.append(
            track_name
        )

        diagnostic_records.append(
            track_record
        )

        print(
            "  ✗ prediction directory missing"
        )

        continue


    # ------------------------------------------------------------------------
    # Inspect each required source.
    # ------------------------------------------------------------------------

    for source_name in EXPECTED_SOURCES:

        reference_path = (
            ground_truth_dir
            / f"{source_name}.wav"
        )

        prediction_path = (
            prediction_dir
            / f"{source_name}.wav"
        )


        source_record = {
            "reference_path": str(
                reference_path
            ),
            "prediction_path": str(
                prediction_path
            ),
            "reference": {},
            "prediction": {},
            "valid": True,
        }


        # ====================================================================
        # Reference metadata
        # ====================================================================

        try:

            with sf.SoundFile(
                reference_path,
                mode="r",
            ) as reference_info:

                reference_metadata = {
                    "exists": True,
                    "frames": int(
                        reference_info.frames
                    ),
                    "sample_rate": int(
                        reference_info.samplerate
                    ),
                    "channels": int(
                        reference_info.channels
                    ),
                    "duration_seconds": (
                        float(
                            reference_info.frames
                        )
                        / float(
                            reference_info.samplerate
                        )
                        if reference_info.samplerate > 0
                        else 0.0
                    ),
                    "file_size_bytes": int(
                        reference_path.stat().st_size
                    ),
                }


            source_record[
                "reference"
            ] = reference_metadata


        except Exception as exc:

            source_record["reference"] = {
                "exists": reference_path.is_file(),
                "error": (
                    f"{type(exc).__name__}: {exc}"
                ),
            }

            source_record["valid"] = False

            track_record["complete"] = False

            track_record["problems"].append(
                f"{source_name}:REFERENCE_OPEN_FAILURE"
            )


        # ====================================================================
        # Prediction metadata
        # ====================================================================

        if not prediction_path.is_file():

            source_record["prediction"] = {
                "exists": False
            }

            source_record["valid"] = False

            track_record["complete"] = False

            track_record["problems"].append(
                f"{source_name}:MISSING_PREDICTION"
            )

            print(
                f"  {source_name:<7} "
                f"✗ MISSING"
            )

            continue


        try:

            with sf.SoundFile(
                prediction_path,
                mode="r",
            ) as prediction_info:

                prediction_metadata = {
                    "exists": True,
                    "frames": int(
                        prediction_info.frames
                    ),
                    "sample_rate": int(
                        prediction_info.samplerate
                    ),
                    "channels": int(
                        prediction_info.channels
                    ),
                    "duration_seconds": (
                        float(
                            prediction_info.frames
                        )
                        / float(
                            prediction_info.samplerate
                        )
                        if prediction_info.samplerate > 0
                        else 0.0
                    ),
                    "file_size_bytes": int(
                        prediction_path.stat().st_size
                    ),
                }


            source_record[
                "prediction"
            ] = prediction_metadata


        except Exception as exc:

            source_record["prediction"] = {
                "exists": True,
                "error": (
                    f"{type(exc).__name__}: {exc}"
                ),
                "file_size_bytes": int(
                    prediction_path.stat().st_size
                ),
            }

            source_record["valid"] = False

            track_record["complete"] = False

            track_record["problems"].append(
                f"{source_name}:PREDICTION_OPEN_FAILURE"
            )

            prediction_open_failures.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": str(
                        prediction_path
                    ),
                    "error": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }
            )

            print(
                f"  {source_name:<7} "
                f"✗ OPEN FAILURE"
            )

            continue


        # ====================================================================
        # Prediction zero-frame validation
        # ====================================================================

        prediction_frames = int(
            source_record[
                "prediction"
            ]["frames"]
        )


        if prediction_frames <= 0:

            source_record["valid"] = False

            track_record["complete"] = False

            problem = (
                f"{source_name}:ZERO_SAMPLES"
            )

            track_record[
                "problems"
            ].append(problem)


            zero_sample_files.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "path": str(
                        prediction_path
                    ),
                    "frames": prediction_frames,
                    "file_size_bytes":
                        source_record[
                            "prediction"
                        ]["file_size_bytes"],
                }
            )


        # ====================================================================
        # Prediction metadata validation
        # ====================================================================

        prediction_rate = int(
            source_record[
                "prediction"
            ]["sample_rate"]
        )

        prediction_channels = int(
            source_record[
                "prediction"
            ]["channels"]
        )


        if (
            prediction_rate
            != EXPECTED_SAMPLE_RATE
        ):

            source_record["valid"] = False

            track_record["complete"] = False

            track_record[
                "problems"
            ].append(
                f"{source_name}:"
                "PREDICTION_SAMPLE_RATE_MISMATCH"
            )

            metadata_mismatch_files.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "type":
                        "PREDICTION_SAMPLE_RATE",
                    "expected":
                        EXPECTED_SAMPLE_RATE,
                    "actual":
                        prediction_rate,
                }
            )


        if (
            prediction_channels
            != EXPECTED_CHANNELS
        ):

            source_record["valid"] = False

            track_record["complete"] = False

            track_record[
                "problems"
            ].append(
                f"{source_name}:"
                "PREDICTION_CHANNEL_MISMATCH"
            )

            metadata_mismatch_files.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "type":
                        "PREDICTION_CHANNELS",
                    "expected":
                        EXPECTED_CHANNELS,
                    "actual":
                        prediction_channels,
                }
            )


        # ====================================================================
        # Reference/prediction frame-count validation
        # ====================================================================

        if (
            source_record["reference"].get(
                "frames"
            )
            is not None
            and prediction_frames
            != source_record[
                "reference"
            ]["frames"]
        ):

            source_record["valid"] = False

            track_record["complete"] = False

            track_record[
                "problems"
            ].append(
                f"{source_name}:"
                "FRAME_COUNT_MISMATCH"
            )

            metadata_mismatch_files.append(
                {
                    "track": track_name,
                    "source": source_name,
                    "type":
                        "FRAME_COUNT",
                    "expected":
                        source_record[
                            "reference"
                        ]["frames"],
                    "actual":
                        prediction_frames,
                }
            )


        # ====================================================================
        # Human-readable source status
        # ====================================================================

        if source_record["valid"]:

            status = "✓ VALID"

        elif prediction_frames <= 0:

            status = "✗ ZERO SAMPLES"

        else:

            status = "✗ INVALID"


        print(
            f"  {source_name:<7} "
            f"{status:<15} "
            f"frames={prediction_frames:,}"
        )


        track_record[
            "sources"
        ][source_name] = source_record


    # ========================================================================
    # Track-level result
    # ========================================================================

    if track_record["complete"]:

        complete_prediction_tracks.append(
            track_name
        )

        print(
            "  → ✓ COMPLETE VALID PREDICTION SET"
        )

    else:

        invalid_prediction_tracks.append(
            track_name
        )

        print(
            "  → ✗ INVALID / INCOMPLETE "
            "PREDICTION SET"
        )


    diagnostic_records.append(
        track_record
    )


# ============================================================================
# 6. SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("EXP004 PREDICTION DIAGNOSTIC RESULT")
print("=" * 70)


print(
    f"Ground-truth tracks           : "
    f"{len(ground_truth_tracks)}"
)

print(
    f"Valid complete predictions    : "
    f"{len(complete_prediction_tracks)}"
)

print(
    f"Invalid/incomplete predictions: "
    f"{len(invalid_prediction_tracks)}"
)

print(
    f"Missing prediction dirs       : "
    f"{len(missing_prediction_tracks)}"
)

print(
    f"Zero-sample prediction files  : "
    f"{len(zero_sample_files)}"
)

print(
    f"Metadata mismatches           : "
    f"{len(metadata_mismatch_files)}"
)

print(
    f"Prediction open failures      : "
    f"{len(prediction_open_failures)}"
)


# ============================================================================
# 7. EXPLICIT PROBLEM REPORT
# ============================================================================

if invalid_prediction_tracks:

    print(
        "\n" + "-" * 70
    )

    print(
        "INVALID / INCOMPLETE TRACKS"
    )

    print(
        "-" * 70
    )

    for track_name in invalid_prediction_tracks:

        record = next(
            item
            for item in diagnostic_records
            if item["track_name"]
            == track_name
        )

        print(
            f"\n{track_name}"
        )

        for problem in record[
            "problems"
        ]:

            print(
                f"  - {problem}"
            )


if zero_sample_files:

    print(
        "\n" + "-" * 70
    )

    print(
        "ZERO-SAMPLE PREDICTION FILES"
    )

    print(
        "-" * 70
    )

    for item in zero_sample_files:

        print(
            f"\nTrack : {item['track']}"
        )

        print(
            f"Source: {item['source']}"
        )

        print(
            f"Frames: {item['frames']}"
        )

        print(
            f"Size  : "
            f"{item['file_size_bytes']:,} bytes"
        )

        print(
            f"Path  : {item['path']}"
        )


# ============================================================================
# 8. SAVE DIAGNOSTIC REPORT
# ============================================================================

diagnostic_output_path = (
    PREDICTIONS_ROOT.parent
    / "metrics"
    / "EXP004_prediction_audio_integrity_diagnostic.json"
)


diagnostic_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


diagnostic_report = {
    "experiment": "EXP004",

    "diagnostic": (
        "PREDICTION_AUDIO_INTEGRITY"
    ),

    "ground_truth_root":
        str(TEST_ROOT),

    "prediction_root":
        str(PREDICTIONS_ROOT),

    "expected_tracks":
        EXPECTED_TRACKS,

    "expected_sources":
        list(EXPECTED_SOURCES),

    "expected_sample_rate":
        EXPECTED_SAMPLE_RATE,

    "expected_channels":
        EXPECTED_CHANNELS,

    "ground_truth_track_count":
        len(ground_truth_tracks),

    "valid_complete_prediction_track_count":
        len(complete_prediction_tracks),

    "invalid_prediction_track_count":
        len(invalid_prediction_tracks),

    "missing_prediction_directory_count":
        len(missing_prediction_tracks),

    "zero_sample_prediction_file_count":
        len(zero_sample_files),

    "metadata_mismatch_count":
        len(metadata_mismatch_files),

    "prediction_open_failure_count":
        len(prediction_open_failures),

    "valid_complete_tracks":
        complete_prediction_tracks,

    "invalid_tracks":
        invalid_prediction_tracks,

    "missing_prediction_tracks":
        missing_prediction_tracks,

    "zero_sample_files":
        zero_sample_files,

    "metadata_mismatches":
        metadata_mismatch_files,

    "prediction_open_failures":
        prediction_open_failures,

    "track_records":
        diagnostic_records,
}


with open(
    diagnostic_output_path,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        diagnostic_report,
        handle,
        indent=2,
        ensure_ascii=False,
    )


print(
    "\n✓ Diagnostic report written:"
)

print(
    f"  {diagnostic_output_path}"
)


# ============================================================================
# 9. FINAL DECISION
# ============================================================================

print("\n" + "=" * 70)
print("DIAGNOSTIC DECISION")
print("=" * 70)


if zero_sample_files:

    print(
        "⚠ ZERO-SAMPLE PREDICTION FILES DETECTED."
    )

    print(
        "⚠ DO NOT resume objective metric evaluation yet."
    )

    print(
        "⚠ Repair/regenerate the affected prediction "
        "file(s) first."
    )

elif metadata_mismatch_files:

    print(
        "⚠ PREDICTION METADATA MISMATCHES DETECTED."
    )

    print(
        "⚠ DO NOT resume objective metric evaluation yet."
    )

else:

    print(
        "✓ All prediction WAVs passed metadata "
        "and non-empty validation."
    )

    print(
        "✓ Objective metric evaluation may resume."
    )


print(
    "\nNo HT-Demucs inference was performed."
)

print(
    "No objective metric was computed."
)

print(
    "No existing metric result was modified."
)

print("=" * 70)
print("✓ EXP004 PREDICTION DIAGNOSTIC COMPLETE")
print("=" * 70)

EXP004 PREDICTION AUDIO INTEGRITY DIAGNOSTIC

----------------------------------------------------------------------
PERSISTENT REGISTRY VALIDATION
----------------------------------------------------------------------
✓ EXP004_PATHS registry available
✓ MUSDB18_TRACKS registry available
✓ MUSDB18_AUDIO_METADATA registry available
✓ EXP004_MODEL_CONFIG registry available
✓ PREDICTION_AUDIT registry available

----------------------------------------------------------------------
DIAGNOSTIC CONFIGURATION
----------------------------------------------------------------------
Expected tracks       : 50
Expected sources      : ['vocals', 'drums', 'bass', 'other']
Expected sample rate  : 44100
Expected channels     : 2
Ground-truth root     : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\03_Datasets\Benchmarks\MUSDB18\test
Prediction root       : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\predictions

------

# EXP004 Targeted Skelpolu Prediction Repair

In [ ]:
# ============================================================================
# EXP004 TARGETED PREDICTION REPAIR — SKELPOLU - RESURRECTION / DRUMS
# ============================================================================
#
# Purpose
# -------
# Repair ONLY the invalid prediction identified by the EXP004 prediction
# audio integrity diagnostic:
#
#     Track  : Skelpolu - Resurrection
#     Source : drums
#
# The diagnostic established that:
#
#     vocals.wav -> VALID
#     drums.wav  -> ZERO SAMPLES / INVALID
#     bass.wav   -> VALID
#     other.wav  -> VALID
#
# Therefore this cell MUST NOT rewrite the three already-valid prediction
# files.
#
# Exact EXP004 inference configuration:
#
#     Model              : HT-Demucs (htdemucs)
#     Sample rate        : 44100 Hz
#     Channels           : 2
#     Physical chunk     : 60 s
#     Physical overlap   : 5 s
#     Physical step      : 55 s
#     Demucs API         : demucs.apply.apply_model
#     shifts             : 1
#     split              : True
#     Demucs overlap     : 0.25
#     Reconstruction     : linear crossfade
#
# No objective metrics are computed here.
#
# IMPORTANT
# ---------
# This cell intentionally does NOT require:
#
#     EXP004_INFERENCE_RESULTS
#
# That registry is produced only after the full inference cell completes
# and is not required to repair an individual prediction.
#
# ============================================================================


from pathlib import Path
import gc
import time

import numpy as np
import soundfile as sf
import torch

from demucs.apply import apply_model


print("=" * 70)
print("EXP004 TARGETED PREDICTION REPAIR")
print("=" * 70)


# ============================================================================
# 1. REQUIRED EXP004 STATE VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("REQUIRED EXP004 STATE VALIDATION")
print("-" * 70)


required_registries = [
    "EXP004_PATHS",
    "MUSDB18_TRACKS",
    "MUSDB18_AUDIO_METADATA",
    "EXP004_MODEL_CONFIG",
    "PREDICTION_AUDIT",
]


missing_registries = [
    name
    for name in required_registries
    if name not in globals()
]


if missing_registries:

    raise RuntimeError(
        "Required EXP004 registries are unavailable:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_registries
        )
    )


for name in required_registries:

    print(
        f"✓ {name} registry available"
    )


# ============================================================================
# 2. VALIDATED MODEL STATE
# ============================================================================

print("\n" + "-" * 70)
print("MODEL STATE VALIDATION")
print("-" * 70)


if "exp004_model" not in globals():

    raise RuntimeError(
        "Validated EXP004 HT-Demucs model `exp004_model` "
        "is unavailable."
    )


if "EXP004_DEVICE" not in globals():

    raise RuntimeError(
        "Validated EXP004_DEVICE is unavailable."
    )


if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA is unavailable. "
        "EXP004 targeted repair requires CUDA."
    )


MODEL_DEVICE = EXP004_DEVICE


MODEL_SAMPLE_RATE = int(
    EXP004_MODEL_CONFIG["sample_rate"]
)


MODEL_CHANNELS = int(
    EXP004_MODEL_CONFIG["audio_channels"]
)


EXPECTED_SOURCES = list(
    EXP004_MODEL_CONFIG["expected_sources"]
)


MODEL_SOURCES = list(
    exp004_model.sources
)


SOURCE_INDEX = {
    source_name: index
    for index, source_name in enumerate(
        MODEL_SOURCES
    )
}


# --------------------------------------------------------------------------
# Validate source configuration.
# --------------------------------------------------------------------------

if set(MODEL_SOURCES) != set(
    EXP004_MODEL_CONFIG["model_sources"]
):

    raise RuntimeError(
        "Loaded HT-Demucs source set does not match "
        "EXP004_MODEL_CONFIG.\n"
        f"Configured model sources: "
        f"{EXP004_MODEL_CONFIG['model_sources']}\n"
        f"Loaded model sources:     {MODEL_SOURCES}"
    )


if set(MODEL_SOURCES) != set(
    EXPECTED_SOURCES
):

    raise RuntimeError(
        "Loaded HT-Demucs source set does not match "
        "the expected EXP004 source set.\n"
        f"Expected: {EXPECTED_SOURCES}\n"
        f"Model:    {MODEL_SOURCES}"
    )


if int(
    exp004_model.samplerate
) != MODEL_SAMPLE_RATE:

    raise RuntimeError(
        "HT-Demucs sample-rate mismatch.\n"
        f"Expected: {MODEL_SAMPLE_RATE}\n"
        f"Model:    {exp004_model.samplerate}"
    )


if int(
    exp004_model.audio_channels
) != MODEL_CHANNELS:

    raise RuntimeError(
        "HT-Demucs channel-count mismatch.\n"
        f"Expected: {MODEL_CHANNELS}\n"
        f"Model:    {exp004_model.audio_channels}"
    )


if MODEL_DEVICE.type != "cuda":

    raise RuntimeError(
        "EXP004 targeted repair requires CUDA.\n"
        f"Configured device: {MODEL_DEVICE}"
    )


exp004_model.to(
    MODEL_DEVICE
)

exp004_model.eval()


print(
    f"Model             : "
    f"{EXP004_MODEL_CONFIG['model_name']}"
)

print(
    f"Device            : "
    f"{MODEL_DEVICE}"
)

print(
    f"Sample rate       : "
    f"{MODEL_SAMPLE_RATE} Hz"
)

print(
    f"Channels          : "
    f"{MODEL_CHANNELS}"
)

print(
    f"Model sources     : "
    f"{MODEL_SOURCES}"
)

print(
    f"Expected sources  : "
    f"{EXPECTED_SOURCES}"
)

print(
    "✓ Validated EXP004 HT-Demucs model state"
)


# ============================================================================
# 3. EXACT EXP004 PHYSICAL CHUNKING VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("PHYSICAL CHUNKING VALIDATION")
print("-" * 70)


CHUNK_DURATION_SECONDS = float(
    EXP004_MODEL_CONFIG[
        "chunk_duration_seconds"
    ]
)


CHUNK_OVERLAP_SECONDS = float(
    EXP004_MODEL_CONFIG[
        "chunk_overlap_seconds"
    ]
)


CHUNK_STEP_SECONDS = float(
    EXP004_MODEL_CONFIG[
        "chunk_step_seconds"
    ]
)


if (
    CHUNK_DURATION_SECONDS != 60.0
    or CHUNK_OVERLAP_SECONDS != 5.0
    or CHUNK_STEP_SECONDS != 55.0
):

    raise RuntimeError(
        "EXP004 physical chunking configuration is not "
        "the established 60 s / 5 s / 55 s configuration.\n"
        f"Duration: {CHUNK_DURATION_SECONDS}\n"
        f"Overlap : {CHUNK_OVERLAP_SECONDS}\n"
        f"Step    : {CHUNK_STEP_SECONDS}"
    )


CHUNK_SIZE = int(
    round(
        CHUNK_DURATION_SECONDS
        * MODEL_SAMPLE_RATE
    )
)


OVERLAP_SIZE = int(
    round(
        CHUNK_OVERLAP_SECONDS
        * MODEL_SAMPLE_RATE
    )
)


STEP_SIZE = int(
    round(
        CHUNK_STEP_SECONDS
        * MODEL_SAMPLE_RATE
    )
)


if STEP_SIZE != (
    CHUNK_SIZE - OVERLAP_SIZE
):

    raise RuntimeError(
        "Physical chunk sample configuration is inconsistent."
    )


print(
    f"Chunk duration : "
    f"{CHUNK_DURATION_SECONDS:g} s"
)

print(
    f"Chunk overlap  : "
    f"{CHUNK_OVERLAP_SECONDS:g} s"
)

print(
    f"Chunk step     : "
    f"{CHUNK_STEP_SECONDS:g} s"
)

print(
    f"Chunk samples  : "
    f"{CHUNK_SIZE:,}"
)

print(
    f"Overlap samples: "
    f"{OVERLAP_SIZE:,}"
)

print(
    f"Step samples   : "
    f"{STEP_SIZE:,}"
)

print(
    "✓ Exact EXP004 physical chunking validated"
)


# ============================================================================
# 4. TARGET TRACK AND TARGET SOURCE
# ============================================================================

print("\n" + "-" * 70)
print("TARGET TRACK / SOURCE VALIDATION")
print("-" * 70)


TARGET_TRACK_NAME = (
    "Skelpolu - Resurrection"
)


TARGET_SOURCE_NAME = "drums"


if TARGET_SOURCE_NAME not in EXPECTED_SOURCES:

    raise RuntimeError(
        f"Target source '{TARGET_SOURCE_NAME}' is not present "
        "in the configured EXP004 source set.\n"
        f"Expected sources: {EXPECTED_SOURCES}"
    )


# --------------------------------------------------------------------------
# Locate the target using the authoritative MUSDB18_TRACKS registry.
# --------------------------------------------------------------------------

target_records = [
    record
    for record in MUSDB18_TRACKS
    if record.get("track_name")
    == TARGET_TRACK_NAME
]


if len(target_records) != 1:

    raise RuntimeError(
        "Expected exactly one target record in MUSDB18_TRACKS.\n"
        f"Target: {TARGET_TRACK_NAME}\n"
        f"Matches found: {len(target_records)}"
    )


target_record = target_records[0]


if "track_path" not in target_record:

    raise RuntimeError(
        "Target MUSDB18_TRACKS record does not contain "
        "the established 'track_path' field."
    )


target_track_path = Path(
    target_record["track_path"]
)


mixture_path = (
    target_track_path
    / "mixture.wav"
)


PREDICTIONS_ROOT = Path(
    EXP004_PATHS[
        "objective_predictions"
    ]
)


target_output_dir = (
    PREDICTIONS_ROOT
    / TARGET_TRACK_NAME
)


target_prediction_path = (
    target_output_dir
    / f"{TARGET_SOURCE_NAME}.wav"
)


if not target_track_path.is_dir():

    raise RuntimeError(
        "Target track directory does not exist:\n"
        f"{target_track_path}"
    )


if not mixture_path.is_file():

    raise RuntimeError(
        "Target mixture.wav does not exist:\n"
        f"{mixture_path}"
    )


print(
    f"Target track       : "
    f"{TARGET_TRACK_NAME}"
)

print(
    f"Target source      : "
    f"{TARGET_SOURCE_NAME}"
)

print(
    f"Track directory    : "
    f"{target_track_path}"
)

print(
    f"Mixture            : "
    f"{mixture_path}"
)

print(
    f"Prediction to repair:\n"
    f"  {target_prediction_path}"
)

print(
    "✓ Target track located through "
    "authoritative MUSDB18_TRACKS registry"
)


# ============================================================================
# 5. PRESERVE VALID PREDICTIONS
# ============================================================================

print("\n" + "-" * 70)
print("EXISTING PREDICTION PRESERVATION CHECK")
print("-" * 70)


for source_name in EXPECTED_SOURCES:

    source_path = (
        target_output_dir
        / f"{source_name}.wav"
    )

    if not source_path.is_file():

        print(
            f"⚠ {source_name}.wav is missing"
        )

        continue


    with sf.SoundFile(
        source_path,
        mode="r",
    ) as info:

        print(
            f"{source_name:<8} "
            f"frames={info.frames:,} "
            f"sr={info.samplerate} "
            f"channels={info.channels}"
        )


print(
    "\nOnly the invalid drums.wav will be replaced."
)

print(
    "✓ Existing valid prediction files will not be modified"
)


# ============================================================================
# 6. LOAD AND VALIDATE MIXTURE
# ============================================================================

print("\n" + "-" * 70)
print("TARGET MIXTURE VALIDATION")
print("-" * 70)


mixture_audio, sample_rate = sf.read(
    mixture_path,
    dtype="float32",
    always_2d=True,
)


if sample_rate != MODEL_SAMPLE_RATE:

    raise RuntimeError(
        "Target mixture sample-rate mismatch.\n"
        f"Expected: {MODEL_SAMPLE_RATE}\n"
        f"Actual:   {sample_rate}"
    )


if mixture_audio.shape[1] != MODEL_CHANNELS:

    raise RuntimeError(
        "Target mixture channel-count mismatch.\n"
        f"Expected: {MODEL_CHANNELS}\n"
        f"Actual:   {mixture_audio.shape[1]}"
    )


if not np.all(
    np.isfinite(mixture_audio)
):

    raise RuntimeError(
        "Target mixture contains non-finite values."
    )


# soundfile:
#
#     [samples, channels]
#
# Demucs:
#
#     [channels, samples]

mixture_audio = np.ascontiguousarray(
    mixture_audio.T,
    dtype=np.float32,
)


total_samples = (
    mixture_audio.shape[1]
)


if total_samples <= 0:

    raise RuntimeError(
        "Target mixture contains zero samples."
    )


duration_seconds = (
    total_samples
    / MODEL_SAMPLE_RATE
)


print(
    f"Sample rate : {sample_rate} Hz"
)

print(
    f"Channels    : {mixture_audio.shape[0]}"
)

print(
    f"Samples     : {total_samples:,}"
)

print(
    f"Duration    : {duration_seconds:.2f} s"
)

print(
    "✓ Target mixture validated"
)


# ============================================================================
# 7. PHYSICAL CHUNK BOUNDARIES
# ============================================================================

def generate_target_chunk_boundaries(
    total_samples,
):

    if total_samples <= 0:

        raise ValueError(
            "total_samples must be positive."
        )


    boundaries = []

    start = 0


    while start < total_samples:

        end = min(
            start + CHUNK_SIZE,
            total_samples,
        )


        boundaries.append(
            (start, end)
        )


        if end >= total_samples:

            break


        start += STEP_SIZE


    return boundaries


chunk_boundaries = (
    generate_target_chunk_boundaries(
        total_samples
    )
)


if not chunk_boundaries:

    raise RuntimeError(
        "No physical chunk boundaries were generated."
    )


print(
    f"Physical chunks : "
    f"{len(chunk_boundaries)}"
)

print(
    "✓ Physical chunk boundaries generated"
)


# ============================================================================
# 8. SINGLE-CHUNK HT-DEMUCS INFERENCE
# ============================================================================

def infer_target_chunk(
    chunk_audio,
):

    if not isinstance(
        chunk_audio,
        np.ndarray,
    ):

        raise TypeError(
            "chunk_audio must be a numpy array."
        )


    if chunk_audio.ndim != 2:

        raise ValueError(
            "Expected chunk audio shape "
            "[channels, samples].\n"
            f"Actual: {chunk_audio.shape}"
        )


    if chunk_audio.shape[0] != (
        MODEL_CHANNELS
    ):

        raise ValueError(
            "Unexpected chunk channel count.\n"
            f"Expected: {MODEL_CHANNELS}\n"
            f"Actual: {chunk_audio.shape[0]}"
        )


    if not np.all(
        np.isfinite(chunk_audio)
    ):

        raise RuntimeError(
            "Chunk contains non-finite values."
        )


    mixture_tensor = (
        torch.from_numpy(
            np.ascontiguousarray(
                chunk_audio,
                dtype=np.float32,
            )
        )
        .unsqueeze(0)
        .to(MODEL_DEVICE)
    )


    with torch.inference_mode():

        output = apply_model(
            exp004_model,
            mixture_tensor,
            shifts=1,
            split=True,
            overlap=0.25,
            progress=False,
            device=MODEL_DEVICE,
        )


    if not isinstance(
        output,
        torch.Tensor,
    ):

        raise RuntimeError(
            "HT-Demucs returned an unexpected object.\n"
            f"Type: {type(output)}"
        )


    if output.ndim != 4:

        raise RuntimeError(
            "Unexpected HT-Demucs output dimensions.\n"
            f"Expected: 4\n"
            f"Actual: {output.ndim}\n"
            f"Shape: {tuple(output.shape)}"
        )


    if output.shape[0] != 1:

        raise RuntimeError(
            "Unexpected HT-Demucs batch dimension.\n"
            f"Shape: {tuple(output.shape)}"
        )


    if output.shape[1] != len(
        MODEL_SOURCES
    ):

        raise RuntimeError(
            "Unexpected HT-Demucs source dimension.\n"
            f"Model sources: {MODEL_SOURCES}\n"
            f"Output shape:  {tuple(output.shape)}"
        )


    if output.shape[2] != MODEL_CHANNELS:

        raise RuntimeError(
            "Unexpected HT-Demucs channel dimension.\n"
            f"Expected: {MODEL_CHANNELS}\n"
            f"Output shape: {tuple(output.shape)}"
        )


    expected_sample_count = (
        chunk_audio.shape[1]
    )


    if output.shape[3] != (
        expected_sample_count
    ):

        raise RuntimeError(
            "HT-Demucs chunk sample count mismatch.\n"
            f"Expected: {expected_sample_count}\n"
            f"Actual: {output.shape[3]}"
        )


    output = (
        output
        .squeeze(0)
        .detach()
        .cpu()
        .numpy()
    )


    output = np.asarray(
        output,
        dtype=np.float32,
    )


    if not np.all(
        np.isfinite(output)
    ):

        raise RuntimeError(
            "HT-Demucs prediction contains "
            "non-finite values."
        )


    return output


# ============================================================================
# 9. LINEAR CROSSFADE RECONSTRUCTION
# ============================================================================

def reconstruct_target_chunks(
    chunk_predictions,
    chunk_boundaries,
    total_samples,
):

    source_count = len(
        EXPECTED_SOURCES
    )


    if len(chunk_predictions) != len(
        chunk_boundaries
    ):

        raise RuntimeError(
            "Chunk prediction/boundary count mismatch."
        )


    reconstructed = np.zeros(
        (
            source_count,
            MODEL_CHANNELS,
            total_samples,
        ),
        dtype=np.float32,
    )


    weights = np.zeros(
        total_samples,
        dtype=np.float32,
    )


    for chunk_index, (
        prediction,
        boundary,
    ) in enumerate(
        zip(
            chunk_predictions,
            chunk_boundaries,
        )
    ):

        start, end = boundary

        chunk_length = (
            end - start
        )


        expected_shape = (
            source_count,
            MODEL_CHANNELS,
            chunk_length,
        )


        if prediction.shape != (
            expected_shape
        ):

            raise RuntimeError(
                "Chunk reconstruction shape mismatch.\n"
                f"Chunk index: {chunk_index}\n"
                f"Expected: {expected_shape}\n"
                f"Actual:   {prediction.shape}"
            )


        left_overlap = 0
        right_overlap = 0


        if chunk_index > 0:

            previous_start, previous_end = (
                chunk_boundaries[
                    chunk_index - 1
                ]
            )

            left_overlap = max(
                0,
                previous_end - start,
            )


        if chunk_index < (
            len(chunk_boundaries) - 1
        ):

            next_start, next_end = (
                chunk_boundaries[
                    chunk_index + 1
                ]
            )

            right_overlap = max(
                0,
                end - next_start,
            )


        crossfade = np.ones(
            chunk_length,
            dtype=np.float32,
        )


        if left_overlap > 0:

            left_overlap = min(
                left_overlap,
                chunk_length,
            )


            crossfade[
                :left_overlap
            ] = np.linspace(
                0.0,
                1.0,
                left_overlap,
                endpoint=False,
                dtype=np.float32,
            )


        if right_overlap > 0:

            right_overlap = min(
                right_overlap,
                chunk_length,
            )


            crossfade[
                -right_overlap:
            ] = np.linspace(
                1.0,
                0.0,
                right_overlap,
                endpoint=False,
                dtype=np.float32,
            )


        reconstructed[
            :,
            :,
            start:end,
        ] += (
            prediction
            * crossfade[
                None,
                None,
                :,
            ]
        )


        weights[
            start:end
        ] += crossfade


    if np.any(
        weights <= 0
    ):

        raise RuntimeError(
            "Chunk reconstruction left uncovered "
            "samples."
        )


    reconstructed /= (
        weights[
            None,
            None,
            :,
        ]
    )


    if not np.all(
        np.isfinite(reconstructed)
    ):

        raise RuntimeError(
            "Reconstructed prediction contains "
            "non-finite values."
        )


    return reconstructed


# ============================================================================
# 10. RUN TARGETED INFERENCE
# ============================================================================

print("\n" + "-" * 70)
print("REGENERATING SKELPOLU PREDICTION")
print("-" * 70)


repair_start_time = time.time()


chunk_predictions = []


for chunk_number, (
    start,
    end,
) in enumerate(
    chunk_boundaries,
    start=1,
):

    chunk_audio = (
        mixture_audio[
            :,
            start:end,
        ]
    )


    print(
        f"Chunk "
        f"{chunk_number:03d}/"
        f"{len(chunk_boundaries):03d}"
        f" | {start:,} -> {end:,}",
        end="",
    )


    try:

        prediction = (
            infer_target_chunk(
                chunk_audio
            )
        )


        chunk_predictions.append(
            prediction
        )


        print(" ✓")


    except Exception:

        print(" ✗ FAILED")

        raise


    finally:

        del chunk_audio

        gc.collect()

        if MODEL_DEVICE.type == "cuda":

            torch.cuda.empty_cache()


print(
    "✓ All target chunks inferred"
)


# ============================================================================
# 11. RECONSTRUCT FULL TARGET PREDICTION
# ============================================================================

print("\nReconstructing target prediction...")


reconstructed = (
    reconstruct_target_chunks(
        chunk_predictions,
        chunk_boundaries,
        total_samples,
    )
)


expected_reconstruction_shape = (
    len(EXPECTED_SOURCES),
    MODEL_CHANNELS,
    total_samples,
)


if reconstructed.shape != (
    expected_reconstruction_shape
):

    raise RuntimeError(
        "Full-track reconstruction shape mismatch.\n"
        f"Expected: {expected_reconstruction_shape}\n"
        f"Actual:   {reconstructed.shape}"
    )


print(
    "✓ Linear crossfade reconstruction complete"
)


# ============================================================================
# 12. EXTRACT ONLY THE DRUMS PREDICTION
# ============================================================================

print("\n" + "-" * 70)
print("TARGET SOURCE EXTRACTION")
print("-" * 70)


drums_index = SOURCE_INDEX[
    TARGET_SOURCE_NAME
]


drums_audio = (
    reconstructed[
        drums_index
    ]
)


drums_audio = np.ascontiguousarray(
    drums_audio.T,
    dtype=np.float32,
)


expected_drums_shape = (
    total_samples,
    MODEL_CHANNELS,
)


if drums_audio.shape != (
    expected_drums_shape
):

    raise RuntimeError(
        "Reconstructed drums shape mismatch.\n"
        f"Expected: {expected_drums_shape}\n"
        f"Actual:   {drums_audio.shape}"
    )


if not np.all(
    np.isfinite(drums_audio)
):

    raise RuntimeError(
        "Reconstructed drums prediction "
        "contains non-finite values."
    )


print(
    f"Source             : "
    f"{TARGET_SOURCE_NAME}"
)

print(
    f"Model output index : "
    f"{drums_index}"
)

print(
    f"Output shape       : "
    f"{drums_audio.shape}"
)

print(
    "✓ Target drums prediction extracted"
)


# ============================================================================
# 13. REPLACE ONLY INVALID DRUMS.WAV
# ============================================================================

print("\n" + "-" * 70)
print("REPLACING ONLY INVALID DRUMS PREDICTION")
print("-" * 70)


target_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


if target_prediction_path.exists():

    print(
        f"Removing invalid prediction:\n"
        f"  {target_prediction_path}"
    )

    target_prediction_path.unlink()


sf.write(
    target_prediction_path,
    drums_audio,
    MODEL_SAMPLE_RATE,
    subtype="FLOAT",
)


print(
    f"✓ Wrote repaired file:\n"
    f"  {target_prediction_path}"
)


# ============================================================================
# 14. IMMEDIATE REPAIRED FILE VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("REPAIRED DRUMS.WAV VALIDATION")
print("-" * 70)


if not target_prediction_path.is_file():

    raise RuntimeError(
        "Repaired drums.wav was not created."
    )


with sf.SoundFile(
    target_prediction_path,
    mode="r",
) as repaired_info:

    print(
        f"Frames     : "
        f"{repaired_info.frames:,}"
    )

    print(
        f"Sample rate: "
        f"{repaired_info.samplerate}"
    )

    print(
        f"Channels   : "
        f"{repaired_info.channels}"
    )

    print(
        f"File size  : "
        f"{target_prediction_path.stat().st_size:,} bytes"
    )


    if repaired_info.frames != (
        total_samples
    ):

        raise RuntimeError(
            "Repaired drums.wav frame count mismatch.\n"
            f"Expected: {total_samples:,}\n"
            f"Actual:   {repaired_info.frames:,}"
        )


    if repaired_info.samplerate != (
        MODEL_SAMPLE_RATE
    ):

        raise RuntimeError(
            "Repaired drums.wav sample-rate mismatch."
        )


    if repaired_info.channels != (
        MODEL_CHANNELS
    ):

        raise RuntimeError(
            "Repaired drums.wav channel-count mismatch."
        )


    if repaired_info.frames <= 0:

        raise RuntimeError(
            "Repaired drums.wav still contains zero samples."
        )


print(
    "✓ Repaired drums.wav passes immediate "
    "audio-integrity validation"
)


# ============================================================================
# 15. FINAL REPAIR RESULT
# ============================================================================

repair_elapsed = (
    time.time()
    - repair_start_time
)


# Cleanup temporary inference state.

del chunk_predictions
del reconstructed
del drums_audio
del mixture_audio

gc.collect()

if MODEL_DEVICE.type == "cuda":

    torch.cuda.empty_cache()


print("\n")
print("=" * 70)
print("EXP004 TARGETED REPAIR RESULT")
print("=" * 70)


print(
    f"Target track          : "
    f"{TARGET_TRACK_NAME}"
)

print(
    f"Repaired source       : "
    f"{TARGET_SOURCE_NAME}"
)

print(
    f"Physical chunks       : "
    f"{len(chunk_boundaries)}"
)

print(
    f"Expected frames      : "
    f"{total_samples:,}"
)

print(
    f"Duration              : "
    f"{duration_seconds:.2f} s"
)

print(
    f"Repair time           : "
    f"{repair_elapsed / 60:.2f} min"
)

print(
    "\n✓ drums.wav REPAIRED"
)

print(
    "✓ Existing vocals.wav was not modified"
)

print(
    "✓ Existing bass.wav was not modified"
)

print(
    "✓ Existing other.wav was not modified"
)

print(
    "\n✓ No objective metrics were computed."
)

print(
    "\nNEXT STEP:"
)

print(
    "Run the EXP004 PREDICTION AUDIO INTEGRITY "
    "DIAGNOSTIC again."
)

print(
    "Do NOT run the objective metric evaluation "
    "until the diagnostic reports 50/50 complete "
    "valid prediction sets."
)

print("=" * 70)
print("✓ EXP004 TARGETED REPAIR COMPLETE")
print("=" * 70)

EXP004 TARGETED PREDICTION REPAIR

----------------------------------------------------------------------
REQUIRED EXP004 STATE VALIDATION
----------------------------------------------------------------------
✓ EXP004_PATHS registry available
✓ MUSDB18_TRACKS registry available
✓ MUSDB18_AUDIO_METADATA registry available
✓ EXP004_MODEL_CONFIG registry available
✓ PREDICTION_AUDIT registry available

----------------------------------------------------------------------
MODEL STATE VALIDATION
----------------------------------------------------------------------
Model             : htdemucs
Device            : cuda
Sample rate       : 44100 Hz
Channels          : 2
Model sources     : ['drums', 'bass', 'other', 'vocals']
Expected sources  : ['vocals', 'drums', 'bass', 'other']
✓ Validated EXP004 HT-Demucs model state

----------------------------------------------------------------------
PHYSICAL CHUNKING VALIDATION
----------------------------------------------------------------------

# EXP004 OBJECTIVE METRICS INSPECTION & INTEGRITY AUDIT

In [ ]:
# ============================================================================
# CELL 16 — EXP004 OBJECTIVE METRICS INSPECTION & INTEGRITY AUDIT
# ============================================================================

from pathlib import Path
from datetime import datetime
import csv
import json
import math


print("=" * 70)
print("EXP004 OBJECTIVE METRICS INSPECTION & INTEGRITY AUDIT")
print("=" * 70)


# ============================================================================
# 1. REQUIRED EXP004 REGISTRY VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("REQUIRED EXP004 REGISTRY VALIDATION")
print("-" * 70)

_required_registries = [
    "EXP004_PATHS",
    "MUSDB18_TRACKS",
    "MUSDB18_AUDIO_METADATA",
    "EXP004_MODEL_CONFIG",
    "PREDICTION_AUDIT",
]

_missing_registries = [
    name
    for name in _required_registries
    if name not in globals()
]

if _missing_registries:
    raise RuntimeError(
        "Required EXP004 registries are unavailable:\n"
        + "\n".join(
            f"  - {name}"
            for name in _missing_registries
        )
    )

for _name in _required_registries:
    print(f"✓ {_name} registry available")


# ============================================================================
# 2. AUTHORITATIVE EXP004 ANALYSIS CONFIGURATION
# ============================================================================

EXP004_ANALYSIS_EXPERIMENT = "EXP004"

EXP004_ANALYSIS_TRACKS = int(
    EXP004_MODEL_CONFIG.get(
        "expected_tracks",
        50,
    )
)

EXP004_ANALYSIS_SOURCES = [
    "vocals",
    "drums",
    "bass",
    "other",
]

EXP004_ANALYSIS_APPROVED_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]

EXP004_ANALYSIS_EXCLUDED_METRICS = [
    "ISR",
]

EXP004_ANALYSIS_SOURCE_EVALUATIONS = (
    EXP004_ANALYSIS_TRACKS
    * len(EXP004_ANALYSIS_SOURCES)
)


# ============================================================================
# 3. AUTHORITATIVE PATH CONFIGURATION
# ============================================================================

print("\n" + "-" * 70)
print("EXP004 ANALYSIS PATH CONFIGURATION")
print("-" * 70)

EXP004_ANALYSIS_METRICS_DIR = Path(
    EXP004_PATHS["objective_metrics"]
)

EXP004_ANALYSIS_REPORTS_DIR = Path(
    EXP004_PATHS["objective_reports"]
)

EXP004_ANALYSIS_PLOTS_DIR = Path(
    EXP004_PATHS["objective_plots"]
)

EXP004_ANALYSIS_SOURCE_CSV = (
    EXP004_ANALYSIS_METRICS_DIR
    / "EXP004_per_source_metrics.csv"
)

EXP004_ANALYSIS_TRACK_JSONL = (
    EXP004_ANALYSIS_METRICS_DIR
    / "EXP004_per_track_metrics.jsonl"
)

EXP004_ANALYSIS_SOURCE_AUDIT = (
    EXP004_ANALYSIS_METRICS_DIR
    / "EXP004_metric_integrity_audit.json"
)

EXP004_ANALYSIS_REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXP004_ANALYSIS_PLOTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Persistent registry for all downstream analysis cells.
EXP004_ANALYSIS_PATHS = {
    "metrics": EXP004_ANALYSIS_METRICS_DIR,
    "reports": EXP004_ANALYSIS_REPORTS_DIR,
    "plots": EXP004_ANALYSIS_PLOTS_DIR,
    "source_csv": EXP004_ANALYSIS_SOURCE_CSV,
    "track_jsonl": EXP004_ANALYSIS_TRACK_JSONL,
    "cell15_metric_audit": EXP004_ANALYSIS_SOURCE_AUDIT,
}


for _key, _path in EXP004_ANALYSIS_PATHS.items():
    print(f"{_key:<22}: {_path}")


# ============================================================================
# 4. REQUIRED FILE VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("PERSISTED OBJECTIVE OUTPUT VALIDATION")
print("-" * 70)

for _label, _path in [
    ("Per-source CSV", EXP004_ANALYSIS_SOURCE_CSV),
    ("Per-track JSONL", EXP004_ANALYSIS_TRACK_JSONL),
    ("Cell 15 metric audit", EXP004_ANALYSIS_SOURCE_AUDIT),
]:
    if not _path.is_file():
        raise FileNotFoundError(
            f"{_label} was not found:\n{_path}"
        )

    if _path.stat().st_size == 0:
        raise ValueError(
            f"{_label} is empty:\n{_path}"
        )

    print(f"✓ {_label}: {_path.name}")


# ============================================================================
# 5. SOURCE CSV SCHEMA INSPECTION
# ============================================================================

print("\n" + "-" * 70)
print("ACTUAL EXP004 SOURCE CSV SCHEMA")
print("-" * 70)

with EXP004_ANALYSIS_SOURCE_CSV.open(
    "r",
    encoding="utf-8",
    newline="",
) as _f:

    _reader = csv.DictReader(_f)

    EXP004_ANALYSIS_CSV_COLUMNS = (
        _reader.fieldnames or []
    )

    if not EXP004_ANALYSIS_CSV_COLUMNS:
        raise ValueError(
            "EXP004 source-level CSV contains no header."
        )

    for _index, _column in enumerate(
        EXP004_ANALYSIS_CSV_COLUMNS,
        start=1,
    ):
        print(f"{_index:2d}. {_column}")


# ============================================================================
# 6. ACTUAL METRIC COLUMN DETECTION
# ============================================================================

def _normalize_metric_column_name(name):
    return (
        str(name)
        .strip()
        .upper()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


EXP004_ANALYSIS_METRIC_COLUMN_MAP = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _normalized_target = _normalize_metric_column_name(
        _metric
    )

    _matches = [
        _column
        for _column in EXP004_ANALYSIS_CSV_COLUMNS
        if _normalize_metric_column_name(_column)
        == _normalized_target
    ]

    if len(_matches) != 1:
        raise ValueError(
            f"Could not uniquely identify CSV column for "
            f"metric '{_metric}'. Matches: {_matches}"
        )

    EXP004_ANALYSIS_METRIC_COLUMN_MAP[
        _metric
    ] = _matches[0]


print("\nDetected approved metric columns:")

for _metric, _column in (
    EXP004_ANALYSIS_METRIC_COLUMN_MAP.items()
):
    print(
        f"  {_metric:<8} -> {_column}"
    )


# ============================================================================
# 7. TRACK / SOURCE COLUMN DETECTION
# ============================================================================

EXP004_ANALYSIS_LOWER_COLUMNS = {
    str(_column).strip().lower(): _column
    for _column in EXP004_ANALYSIS_CSV_COLUMNS
}

EXP004_ANALYSIS_TRACK_COLUMN = (
    EXP004_ANALYSIS_LOWER_COLUMNS.get("track")
)

EXP004_ANALYSIS_SOURCE_COLUMN = (
    EXP004_ANALYSIS_LOWER_COLUMNS.get("source")
)

if EXP004_ANALYSIS_TRACK_COLUMN is None:
    raise ValueError(
        "The actual EXP004 source CSV does not contain "
        "a 'track' column."
    )

if EXP004_ANALYSIS_SOURCE_COLUMN is None:
    raise ValueError(
        "The actual EXP004 source CSV does not contain "
        "a 'source' column."
    )

print(
    f"\nTrack column : "
    f"{EXP004_ANALYSIS_TRACK_COLUMN}"
)

print(
    f"Source column: "
    f"{EXP004_ANALYSIS_SOURCE_COLUMN}"
)


# ============================================================================
# 8. LOAD AND VALIDATE SOURCE-LEVEL DATA
# ============================================================================

print("\n" + "-" * 70)
print("SOURCE-LEVEL DATA VALIDATION")
print("-" * 70)

EXP004_ANALYSIS_SOURCE_ROWS = []

with EXP004_ANALYSIS_SOURCE_CSV.open(
    "r",
    encoding="utf-8",
    newline="",
) as _f:

    _reader = csv.DictReader(_f)

    for _row_number, _row in enumerate(
        _reader,
        start=2,
    ):

        _track = (
            _row.get(
                EXP004_ANALYSIS_TRACK_COLUMN,
                "",
            )
            .strip()
        )

        _source = (
            _row.get(
                EXP004_ANALYSIS_SOURCE_COLUMN,
                "",
            )
            .strip()
            .lower()
        )

        if not _track:
            raise ValueError(
                f"CSV row {_row_number}: empty track identifier."
            )

        if _source not in EXP004_ANALYSIS_SOURCES:
            raise ValueError(
                f"CSV row {_row_number}: unexpected source "
                f"'{_source}'."
            )

        _parsed = {
            "track": _track,
            "source": _source,
        }

        for _metric, _column in (
            EXP004_ANALYSIS_METRIC_COLUMN_MAP.items()
        ):

            _raw = _row.get(_column)

            try:
                _value = float(_raw)
            except (TypeError, ValueError):
                raise ValueError(
                    f"CSV row {_row_number}: invalid "
                    f"{_metric} value '{_raw}'."
                )

            if not math.isfinite(_value):
                raise ValueError(
                    f"CSV row {_row_number}: non-finite "
                    f"{_metric} value."
                )

            _parsed[_metric] = _value

        EXP004_ANALYSIS_SOURCE_ROWS.append(
            _parsed
        )


print(
    f"Source-level rows discovered: "
    f"{len(EXP004_ANALYSIS_SOURCE_ROWS)}"
)

if (
    len(EXP004_ANALYSIS_SOURCE_ROWS)
    != EXP004_ANALYSIS_SOURCE_EVALUATIONS
):
    raise ValueError(
        f"Expected "
        f"{EXP004_ANALYSIS_SOURCE_EVALUATIONS} "
        f"source-level rows, found "
        f"{len(EXP004_ANALYSIS_SOURCE_ROWS)}."
    )

print(
    f"✓ {EXP004_ANALYSIS_TRACKS} tracks × "
    f"{len(EXP004_ANALYSIS_SOURCES)} sources = "
    f"{EXP004_ANALYSIS_SOURCE_EVALUATIONS} rows"
)


# ============================================================================
# 9. SOURCE COVERAGE AND DUPLICATE VALIDATION
# ============================================================================

EXP004_ANALYSIS_TRACKS_FOUND = sorted(
    {
        _row["track"]
        for _row in EXP004_ANALYSIS_SOURCE_ROWS
    },
    key=str.casefold,
)

if (
    len(EXP004_ANALYSIS_TRACKS_FOUND)
    != EXP004_ANALYSIS_TRACKS
):
    raise ValueError(
        f"Expected {EXP004_ANALYSIS_TRACKS} unique tracks, "
        f"found {len(EXP004_ANALYSIS_TRACKS_FOUND)}."
    )

EXP004_ANALYSIS_TRACK_SOURCE_PAIRS = [
    (
        _row["track"],
        _row["source"],
    )
    for _row in EXP004_ANALYSIS_SOURCE_ROWS
]

if (
    len(set(EXP004_ANALYSIS_TRACK_SOURCE_PAIRS))
    != EXP004_ANALYSIS_SOURCE_EVALUATIONS
):
    raise ValueError(
        "Duplicate or missing track/source combinations "
        "were detected."
    )

for _source in EXP004_ANALYSIS_SOURCES:

    _source_tracks = {
        _row["track"]
        for _row in EXP004_ANALYSIS_SOURCE_ROWS
        if _row["source"] == _source
    }

    if (
        _source_tracks
        != set(EXP004_ANALYSIS_TRACKS_FOUND)
    ):
        raise ValueError(
            f"Source coverage mismatch for '{_source}'."
        )

    print(
        f"✓ {_source:<7}: "
        f"{len(_source_tracks)}/"
        f"{EXP004_ANALYSIS_TRACKS}"
    )


# ============================================================================
# 10. TRACK JSONL STRUCTURE VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("TRACK-LEVEL JSONL STRUCTURE VALIDATION")
print("-" * 70)

EXP004_ANALYSIS_JSONL_RECORDS = []

with EXP004_ANALYSIS_TRACK_JSONL.open(
    "r",
    encoding="utf-8",
) as _f:

    for _line_number, _line in enumerate(
        _f,
        start=1,
    ):

        _line = _line.strip()

        if not _line:
            continue

        try:
            _record = json.loads(_line)
        except json.JSONDecodeError as _exc:
            raise ValueError(
                f"Invalid JSON on line {_line_number}: {_exc}"
            )

        EXP004_ANALYSIS_JSONL_RECORDS.append(
            _record
        )


if (
    len(EXP004_ANALYSIS_JSONL_RECORDS)
    != EXP004_ANALYSIS_TRACKS
):
    raise ValueError(
        f"Expected {EXP004_ANALYSIS_TRACKS} JSONL "
        f"records, found "
        f"{len(EXP004_ANALYSIS_JSONL_RECORDS)}."
    )

EXP004_ANALYSIS_JSONL_TRACKS = set()

for _record in EXP004_ANALYSIS_JSONL_RECORDS:

    if _record.get("status") != "success":
        raise ValueError(
            "A persisted EXP004 metric record does not "
            "have status='success'."
        )

    _track = _record.get("track")

    if not _track:
        raise ValueError(
            "A persisted EXP004 metric record has no track."
        )

    EXP004_ANALYSIS_JSONL_TRACKS.add(
        _track
    )

    _sources = _record.get("sources")

    if not isinstance(_sources, dict):
        raise ValueError(
            f"Track '{_track}' has no valid sources object."
        )

    if set(_sources.keys()) != set(
        EXP004_ANALYSIS_SOURCES
    ):
        raise ValueError(
            f"Track '{_track}' source set mismatch.\n"
            f"Found: {list(_sources.keys())}\n"
            f"Expected: {EXP004_ANALYSIS_SOURCES}"
        )

    _track_average = _record.get(
        "track_average"
    )

    if not isinstance(
        _track_average,
        dict,
    ):
        raise ValueError(
            f"Track '{_track}' has no valid track_average."
        )

    if set(_track_average.keys()) != set(
        EXP004_ANALYSIS_APPROVED_METRICS
    ):
        raise ValueError(
            f"Track '{_track}' track_average keys mismatch.\n"
            f"Found: {list(_track_average.keys())}\n"
            f"Expected: {EXP004_ANALYSIS_APPROVED_METRICS}"
        )

    for _source in EXP004_ANALYSIS_SOURCES:

        _source_metrics = _sources[_source]

        if not isinstance(
            _source_metrics,
            dict,
        ):
            raise ValueError(
                f"Invalid source structure for "
                f"{_track} / {_source}."
            )

        _source_metric_keys = set(
            _source_metrics.keys()
        )

        if _source_metric_keys != {
            "SDR_dB",
            "SIR_dB",
            "SAR_dB",
            "SI_SDR_dB",
        }:
            raise ValueError(
                f"Unexpected metric keys for "
                f"{_track} / {_source}.\n"
                f"Found: {sorted(_source_metric_keys)}"
            )

        for _key, _value in _source_metrics.items():

            if not isinstance(
                _value,
                (int, float),
            ):
                raise ValueError(
                    f"Non-numeric persisted value "
                    f"{_track} / {_source} / {_key}"
                )

            if not math.isfinite(
                float(_value)
            ):
                raise ValueError(
                    f"Non-finite persisted value "
                    f"{_track} / {_source} / {_key}"
                )

        for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

            _value = _track_average[_metric]

            if not isinstance(
                _value,
                (int, float),
            ):
                raise ValueError(
                    f"Non-numeric track-average value "
                    f"{_track} / {_metric}"
                )

            if not math.isfinite(
                float(_value)
            ):
                raise ValueError(
                    f"Non-finite track-average value "
                    f"{_track} / {_metric}"
                )


if (
    EXP004_ANALYSIS_JSONL_TRACKS
    != set(EXP004_ANALYSIS_TRACKS_FOUND)
):
    raise ValueError(
        "JSONL track set does not match source CSV track set."
    )

print(
    f"✓ {len(EXP004_ANALYSIS_JSONL_RECORDS)} "
    "track records structurally valid"
)


# ============================================================================
# 11. VALIDATE CELL 15 AUDIT ARTIFACT
# ============================================================================

print("\n" + "-" * 70)
print("CELL 15 METRIC AUDIT ARTIFACT VALIDATION")
print("-" * 70)

with EXP004_ANALYSIS_SOURCE_AUDIT.open(
    "r",
    encoding="utf-8",
) as _f:

    EXP004_ANALYSIS_CELL15_AUDIT = json.load(
        _f
    )

if not isinstance(
    EXP004_ANALYSIS_CELL15_AUDIT,
    dict,
):
    raise ValueError(
        "Cell 15 metric audit is not a JSON object."
    )

print(
    "✓ Cell 15 metric audit JSON loaded"
)

print(
    "Top-level keys:"
)

for _key in EXP004_ANALYSIS_CELL15_AUDIT.keys():
    print(f"  - {_key}")


# ============================================================================
# 12. WRITE CANONICAL CELL 16 REPORT
# ============================================================================

EXP004_ANALYSIS_INTEGRITY_REPORT = (
    EXP004_ANALYSIS_REPORTS_DIR
    / "EXP004_metric_integrity_audit.json"
)

EXP004_ANALYSIS_INTEGRITY_REPORT_DATA = {
    "experiment": EXP004_ANALYSIS_EXPERIMENT,
    "audit_type": "objective_metric_integrity_audit",
    "generated_at": datetime.now().isoformat(),

    "configuration": {
        "tracks": EXP004_ANALYSIS_TRACKS,
        "sources_per_track": len(
            EXP004_ANALYSIS_SOURCES
        ),
        "source_evaluations": (
            EXP004_ANALYSIS_SOURCE_EVALUATIONS
        ),
        "sources": EXP004_ANALYSIS_SOURCES,
        "approved_metrics": (
            EXP004_ANALYSIS_APPROVED_METRICS
        ),
        "excluded_metrics": (
            EXP004_ANALYSIS_EXCLUDED_METRICS
        ),
    },

    "source_csv": {
        "path": str(
            EXP004_ANALYSIS_SOURCE_CSV
        ),
        "rows": len(
            EXP004_ANALYSIS_SOURCE_ROWS
        ),
        "columns": EXP004_ANALYSIS_CSV_COLUMNS,
        "metric_column_map": (
            EXP004_ANALYSIS_METRIC_COLUMN_MAP
        ),
    },

    "track_jsonl": {
        "path": str(
            EXP004_ANALYSIS_TRACK_JSONL
        ),
        "records": len(
            EXP004_ANALYSIS_JSONL_RECORDS
        ),
    },

    "cell15_audit_artifact": {
        "path": str(
            EXP004_ANALYSIS_SOURCE_AUDIT
        ),
        "top_level_keys": list(
            EXP004_ANALYSIS_CELL15_AUDIT.keys()
        ),
    },

    "validation": {
        "source_row_count_valid": True,
        "track_count_valid": True,
        "source_coverage_valid": True,
        "duplicate_track_source_pairs": False,
        "jsonl_structure_valid": True,
        "track_average_structure_valid": True,
        "cell15_audit_artifact_readable": True,
    },

    "policy": {
        "wav_files_loaded": False,
        "metrics_recomputed": False,
        "existing_metric_outputs_only": True,
        "excluded_metric": "ISR",
    },

    "status": "PASS",
}


with EXP004_ANALYSIS_INTEGRITY_REPORT.open(
    "w",
    encoding="utf-8",
) as _f:

    json.dump(
        EXP004_ANALYSIS_INTEGRITY_REPORT_DATA,
        _f,
        indent=2,
        ensure_ascii=False,
    )


print("\n" + "=" * 70)
print("CELL 16 RESULT")
print("=" * 70)

print(
    f"\nTracks validated:       "
    f"{EXP004_ANALYSIS_TRACKS}"
)

print(
    f"Source evaluations:    "
    f"{EXP004_ANALYSIS_SOURCE_EVALUATIONS}"
)

print(
    f"Approved metrics:      "
    f"{len(EXP004_ANALYSIS_APPROVED_METRICS)}"
)

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:
    print(f"  ✓ {_metric}")

print("\nExcluded:")
print("  ✗ ISR")

print("\nValidation:")
print("  ✓ Source-level CSV structure")
print("  ✓ Source coverage")
print("  ✓ Duplicate track/source combinations")
print("  ✓ Track-level JSONL structure")
print("  ✓ Stored track-average structure")
print("  ✓ Cell 15 metric audit artifact")

print("\nIntegrity report:")
print(
    f"  {EXP004_ANALYSIS_INTEGRITY_REPORT}"
)

print("\n" + "=" * 70)
print("✓ CELL 16 COMPLETE")
print("=" * 70)

print(
    "\nEXP004 objective metric integrity audit completed "
    "from the existing Cell 15 outputs."
)

print(
    "No WAV files were loaded and no objective metrics "
    "were recomputed."
)

EXP004 OBJECTIVE METRICS INSPECTION & INTEGRITY AUDIT

----------------------------------------------------------------------
REQUIRED EXP004 REGISTRY VALIDATION
----------------------------------------------------------------------
✓ EXP004_PATHS registry available
✓ MUSDB18_TRACKS registry available
✓ MUSDB18_AUDIO_METADATA registry available
✓ EXP004_MODEL_CONFIG registry available
✓ PREDICTION_AUDIT registry available

----------------------------------------------------------------------
EXP004 ANALYSIS PATH CONFIGURATION
----------------------------------------------------------------------
metrics               : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\metrics
reports               : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\reports
plots                 : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\O

# EXP004 Metric Descriptive Statistics

In [ ]:
# ============================================================================
# CELL 17 — EXP004 METRIC DESCRIPTIVE STATISTICS
# ============================================================================

import csv
import json
import math
import numpy as np


print("=" * 70)
print("EXP004 METRIC DESCRIPTIVE STATISTICS")
print("=" * 70)


# ============================================================================
# 1. USE CELL 16 CONFIGURATION
# ============================================================================

if "EXP004_ANALYSIS_PATHS" not in globals():
    raise RuntimeError(
        "EXP004_ANALYSIS_PATHS is unavailable. "
        "Run Cell 16 first."
    )

if "EXP004_ANALYSIS_APPROVED_METRICS" not in globals():
    raise RuntimeError(
        "EXP004 analysis configuration is unavailable. "
        "Run Cell 16 first."
    )


# ============================================================================
# 2. LOAD SOURCE-LEVEL CSV
# ============================================================================

EXP004_STATISTICS_SOURCE_ROWS = []

with EXP004_ANALYSIS_SOURCE_CSV.open(
    "r",
    encoding="utf-8",
    newline="",
) as _f:

    _reader = csv.DictReader(_f)

    for _row_number, _row in enumerate(
        _reader,
        start=2,
    ):

        _track = (
            _row[
                EXP004_ANALYSIS_TRACK_COLUMN
            ]
            .strip()
        )

        _source = (
            _row[
                EXP004_ANALYSIS_SOURCE_COLUMN
            ]
            .strip()
            .lower()
        )

        _parsed = {
            "track": _track,
            "source": _source,
        }

        for _metric, _column in (
            EXP004_ANALYSIS_METRIC_COLUMN_MAP.items()
        ):

            try:
                _value = float(
                    _row[_column]
                )
            except (TypeError, ValueError):
                raise ValueError(
                    f"Invalid {_metric} value on "
                    f"CSV row {_row_number}."
                )

            if not math.isfinite(_value):
                raise ValueError(
                    f"Non-finite {_metric} value on "
                    f"CSV row {_row_number}."
                )

            _parsed[_metric] = _value

        EXP004_STATISTICS_SOURCE_ROWS.append(
            _parsed
        )


if (
    len(EXP004_STATISTICS_SOURCE_ROWS)
    != EXP004_ANALYSIS_SOURCE_EVALUATIONS
):
    raise ValueError(
        "Unexpected source-level row count."
    )


# ============================================================================
# 3. DESCRIPTIVE STATISTICS FUNCTION
# ============================================================================

def EXP004_descriptive_statistics(values):

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    if values.size == 0:
        return {
            "count": 0,
            "mean_dB": None,
            "median_dB": None,
            "std_dB": None,
            "min_dB": None,
            "q1_dB": None,
            "q3_dB": None,
            "max_dB": None,
            "iqr_dB": None,
        }

    _q1 = float(
        np.percentile(
            values,
            25,
        )
    )

    _q3 = float(
        np.percentile(
            values,
            75,
        )
    )

    return {
        "count": int(values.size),
        "mean_dB": float(
            np.mean(values)
        ),
        "median_dB": float(
            np.median(values)
        ),
        "std_dB": (
            float(
                np.std(
                    values,
                    ddof=1,
                )
            )
            if values.size > 1
            else 0.0
        ),
        "min_dB": float(
            np.min(values)
        ),
        "q1_dB": _q1,
        "q3_dB": _q3,
        "max_dB": float(
            np.max(values)
        ),
        "iqr_dB": _q3 - _q1,
    }


# ============================================================================
# 4. OVERALL SOURCE-LEVEL STATISTICS
# ============================================================================

print("\n" + "=" * 70)
print("OVERALL METRIC STATISTICS")
print("=" * 70)

EXP004_OVERALL_SOURCE_STATISTICS = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _values = [
        _row[_metric]
        for _row in EXP004_STATISTICS_SOURCE_ROWS
    ]

    _stats = EXP004_descriptive_statistics(
        _values
    )

    EXP004_OVERALL_SOURCE_STATISTICS[
        _metric
    ] = _stats

    print(f"\n{_metric}")
    print("-" * 40)
    print(f"  Count:   {_stats['count']}")
    print(f"  Mean:    {_stats['mean_dB']:.3f} dB")
    print(f"  Median:  {_stats['median_dB']:.3f} dB")
    print(f"  Std:     {_stats['std_dB']:.3f} dB")
    print(f"  Min:     {_stats['min_dB']:.3f} dB")
    print(f"  Q1:      {_stats['q1_dB']:.3f} dB")
    print(f"  Q3:      {_stats['q3_dB']:.3f} dB")
    print(f"  Max:     {_stats['max_dB']:.3f} dB")
    print(f"  IQR:     {_stats['iqr_dB']:.3f} dB")


# ============================================================================
# 5. SOURCE-BY-SOURCE STATISTICS
# ============================================================================

print("\n" + "=" * 70)
print("SOURCE-BY-SOURCE METRIC STATISTICS")
print("=" * 70)

EXP004_SOURCE_STATISTICS = {}

for _source in EXP004_ANALYSIS_SOURCES:

    EXP004_SOURCE_STATISTICS[_source] = {}

    print(f"\n{_source.upper()}")
    print("-" * 70)

    _source_rows = [
        _row
        for _row in EXP004_STATISTICS_SOURCE_ROWS
        if _row["source"] == _source
    ]

    if len(_source_rows) != EXP004_ANALYSIS_TRACKS:
        raise ValueError(
            f"Expected {EXP004_ANALYSIS_TRACKS} rows "
            f"for source '{_source}', found "
            f"{len(_source_rows)}."
        )

    for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

        _values = [
            _row[_metric]
            for _row in _source_rows
        ]

        EXP004_SOURCE_STATISTICS[
            _source
        ][_metric] = (
            EXP004_descriptive_statistics(
                _values
            )
        )

        _stats = (
            EXP004_SOURCE_STATISTICS[
                _source
            ][_metric]
        )

        print(
            f"{_metric:<8}"
            f"{_stats['mean_dB']:>12.3f}"
            f"{_stats['median_dB']:>12.3f}"
            f"{_stats['std_dB']:>12.3f}"
            f"{_stats['min_dB']:>12.3f}"
            f"{_stats['q1_dB']:>12.3f}"
            f"{_stats['q3_dB']:>12.3f}"
            f"{_stats['max_dB']:>12.3f}"
        )


# ============================================================================
# 6. SOURCE MEAN SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("SOURCE MEAN SUMMARY")
print("=" * 70)

print(
    f"\n{'Source':<12}"
    f"{'SDR':>12}"
    f"{'SIR':>12}"
    f"{'SAR':>12}"
    f"{'SI-SDR':>12}"
)

print("-" * 60)

for _source in EXP004_ANALYSIS_SOURCES:

    print(
        f"{_source:<12}"
        f"{EXP004_SOURCE_STATISTICS[_source]['SDR']['mean_dB']:>12.3f}"
        f"{EXP004_SOURCE_STATISTICS[_source]['SIR']['mean_dB']:>12.3f}"
        f"{EXP004_SOURCE_STATISTICS[_source]['SAR']['mean_dB']:>12.3f}"
        f"{EXP004_SOURCE_STATISTICS[_source]['SI-SDR']['mean_dB']:>12.3f}"
    )


# ============================================================================
# 7. LOAD STORED TRACK AVERAGES FROM JSONL
# ============================================================================

print("\n" + "-" * 70)
print("TRACK-AVERAGE DISCOVERY")
print("-" * 70)

EXP004_TRACK_AVERAGE_RECORDS = {}

with EXP004_ANALYSIS_TRACK_JSONL.open(
    "r",
    encoding="utf-8",
) as _f:

    for _line_number, _line in enumerate(
        _f,
        start=1,
    ):

        _line = _line.strip()

        if not _line:
            continue

        _record = json.loads(_line)

        if _record.get("status") != "success":
            continue

        _track = _record.get("track")
        _track_average = _record.get(
            "track_average"
        )

        if not _track:
            continue

        if not isinstance(
            _track_average,
            dict,
        ):
            raise ValueError(
                f"Track '{_track}' has no valid "
                "track_average object."
            )

        _parsed_average = {}

        for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

            if _metric not in _track_average:
                raise ValueError(
                    f"Track '{_track}' is missing "
                    f"track-average metric '{_metric}'."
                )

            _value = float(
                _track_average[_metric]
            )

            if not math.isfinite(_value):
                raise ValueError(
                    f"Track '{_track}' contains non-finite "
                    f"track-average '{_metric}'."
                )

            _parsed_average[_metric] = _value

        EXP004_TRACK_AVERAGE_RECORDS[
            _track
        ] = _parsed_average


if (
    len(EXP004_TRACK_AVERAGE_RECORDS)
    != EXP004_ANALYSIS_TRACKS
):
    raise ValueError(
        f"Expected {EXP004_ANALYSIS_TRACKS} "
        f"track-average records, found "
        f"{len(EXP004_TRACK_AVERAGE_RECORDS)}."
    )

print(
    f"Track-average records discovered: "
    f"{len(EXP004_TRACK_AVERAGE_RECORDS)}"
)

print(
    "✓ Stored track-average records validated"
)


# ============================================================================
# 8. TRACK-AVERAGE STATISTICS
# ============================================================================

print("\n" + "=" * 70)
print("TRACK-AVERAGE METRIC STATISTICS")
print("=" * 70)

EXP004_TRACK_AVERAGE_STATISTICS = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _values = [
        _record[_metric]
        for _record in (
            EXP004_TRACK_AVERAGE_RECORDS.values()
        )
    ]

    _stats = EXP004_descriptive_statistics(
        _values
    )

    EXP004_TRACK_AVERAGE_STATISTICS[
        _metric
    ] = _stats

    print(f"\n{_metric}")
    print("-" * 40)
    print(f"  Count:   {_stats['count']}")
    print(f"  Mean:    {_stats['mean_dB']:.3f} dB")
    print(f"  Median:  {_stats['median_dB']:.3f} dB")
    print(f"  Std:     {_stats['std_dB']:.3f} dB")
    print(f"  Min:     {_stats['min_dB']:.3f} dB")
    print(f"  Q1:      {_stats['q1_dB']:.3f} dB")
    print(f"  Q3:      {_stats['q3_dB']:.3f} dB")
    print(f"  Max:     {_stats['max_dB']:.3f} dB")
    print(f"  IQR:     {_stats['iqr_dB']:.3f} dB")


# ============================================================================
# 9. SAVE DESCRIPTIVE STATISTICS REPORT
# ============================================================================

EXP004_DESCRIPTIVE_REPORT = (
    EXP004_ANALYSIS_REPORTS_DIR
    / "EXP004_metric_descriptive_statistics.json"
)

EXP004_DESCRIPTIVE_REPORT_DATA = {
    "experiment": EXP004_ANALYSIS_EXPERIMENT,
    "approved_metrics": (
        EXP004_ANALYSIS_APPROVED_METRICS
    ),
    "excluded_metrics": (
        EXP004_ANALYSIS_EXCLUDED_METRICS
    ),
    "dataset": {
        "tracks": EXP004_ANALYSIS_TRACKS,
        "sources_per_track": len(
            EXP004_ANALYSIS_SOURCES
        ),
        "source_evaluations": (
            EXP004_ANALYSIS_SOURCE_EVALUATIONS
        ),
    },
    "source_level_statistics": (
        EXP004_SOURCE_STATISTICS
    ),
    "overall_source_level_statistics": (
        EXP004_OVERALL_SOURCE_STATISTICS
    ),
    "track_average_statistics": (
        EXP004_TRACK_AVERAGE_STATISTICS
    ),
}


with EXP004_DESCRIPTIVE_REPORT.open(
    "w",
    encoding="utf-8",
) as _f:

    json.dump(
        EXP004_DESCRIPTIVE_REPORT_DATA,
        _f,
        indent=2,
        allow_nan=False,
    )


print("\n" + "=" * 70)
print("CELL 17 RESULT")
print("=" * 70)

print(
    f"\nTracks analysed:          "
    f"{EXP004_ANALYSIS_TRACKS}"
)

print(
    f"Source evaluations:      "
    f"{EXP004_ANALYSIS_SOURCE_EVALUATIONS}"
)

print(
    f"Approved metrics:        "
    f"{len(EXP004_ANALYSIS_APPROVED_METRICS)}"
)

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:
    print(f"  ✓ {_metric}")

print("\nExcluded:")
print("  ✗ ISR")

print("\nStatistics report:")
print(
    f"  {EXP004_DESCRIPTIVE_REPORT}"
)

print("\n" + "=" * 70)
print("✓ CELL 17 COMPLETE")
print("=" * 70)

print(
    "\nEXP004 descriptive statistics were calculated "
    "from the existing objective metric outputs."
)

print(
    "No WAV files were loaded and no objective metrics "
    "were recomputed."
)

EXP004 METRIC DESCRIPTIVE STATISTICS

OVERALL METRIC STATISTICS

SDR
----------------------------------------
  Count:   200
  Mean:    8.351 dB
  Median:  8.462 dB
  Std:     3.703 dB
  Min:     -3.361 dB
  Q1:      6.380 dB
  Q3:      10.644 dB
  Max:     22.163 dB
  IQR:     4.264 dB

SIR
----------------------------------------
  Count:   200
  Mean:    17.239 dB
  Median:  17.811 dB
  Std:     6.398 dB
  Min:     -2.599 dB
  Q1:      13.236 dB
  Q3:      21.009 dB
  Max:     38.809 dB
  IQR:     7.773 dB

SAR
----------------------------------------
  Count:   200
  Mean:    9.229 dB
  Median:  9.304 dB
  Std:     3.852 dB
  Min:     -4.288 dB
  Q1:      7.203 dB
  Q3:      11.403 dB
  Max:     23.533 dB
  IQR:     4.201 dB

SI-SDR
----------------------------------------
  Count:   200
  Mean:    7.442 dB
  Median:  7.833 dB
  Std:     4.522 dB
  Min:     -9.244 dB
  Q1:      5.283 dB
  Q3:      10.263 dB
  Max:     22.214 dB
  IQR:     4.980 dB

SOURCE-BY-SOURCE METRIC STATISTIC

# EXP004 Source-Level Comparative Analysis

In [ ]:
# ============================================================================
# CELL 18 — EXP004 SOURCE-LEVEL COMPARATIVE ANALYSIS
# ============================================================================

from datetime import datetime
import csv
import json
import math


print("=" * 70)
print("EXP004 SOURCE-LEVEL COMPARATIVE ANALYSIS")
print("=" * 70)


if "EXP004_ANALYSIS_PATHS" not in globals():
    raise RuntimeError(
        "EXP004 analysis configuration is unavailable. "
        "Run Cell 16 first."
    )


# ============================================================================
# 1. LOAD AUTHORITATIVE SOURCE CSV
# ============================================================================

EXP004_SOURCE_COMPARISON_ROWS = []

with EXP004_ANALYSIS_SOURCE_CSV.open(
    "r",
    encoding="utf-8",
    newline="",
) as _f:

    _reader = csv.DictReader(_f)

    for _row_number, _row in enumerate(
        _reader,
        start=2,
    ):

        _track = (
            _row[
                EXP004_ANALYSIS_TRACK_COLUMN
            ]
            .strip()
        )

        _source = (
            _row[
                EXP004_ANALYSIS_SOURCE_COLUMN
            ]
            .strip()
            .lower()
        )

        if _source not in EXP004_ANALYSIS_SOURCES:
            raise ValueError(
                f"Unexpected source '{_source}' "
                f"on row {_row_number}."
            )

        _normalized = {
            "track": _track,
            "source": _source,
        }

        for _metric, _column in (
            EXP004_ANALYSIS_METRIC_COLUMN_MAP.items()
        ):

            _value = float(
                _row[_column]
            )

            if not math.isfinite(_value):
                raise ValueError(
                    f"Non-finite {_metric} value for "
                    f"{_track} / {_source}."
                )

            _normalized[_metric] = _value

        EXP004_SOURCE_COMPARISON_ROWS.append(
            _normalized
        )


if (
    len(EXP004_SOURCE_COMPARISON_ROWS)
    != EXP004_ANALYSIS_SOURCE_EVALUATIONS
):
    raise ValueError(
        "Unexpected source-level row count."
    )


# ============================================================================
# 2. GROUP VALUES BY SOURCE
# ============================================================================

EXP004_SOURCE_COMPARISON_VALUES = {
    _source: {
        _metric: []
        for _metric in EXP004_ANALYSIS_APPROVED_METRICS
    }
    for _source in EXP004_ANALYSIS_SOURCES
}

for _row in EXP004_SOURCE_COMPARISON_ROWS:

    for _metric in EXP004_ANALYSIS_APPROVED_METRICS:
        EXP004_SOURCE_COMPARISON_VALUES[
            _row["source"]
        ][_metric].append(
            _row[_metric]
        )


# ============================================================================
# 3. SOURCE MEANS
# ============================================================================

print("\n" + "=" * 70)
print("SOURCE MEAN COMPARISON")
print("=" * 70)

EXP004_SOURCE_MEANS_D_B = {
    _source: {}
    for _source in EXP004_ANALYSIS_SOURCES
}

for _source in EXP004_ANALYSIS_SOURCES:

    for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

        _values = (
            EXP004_SOURCE_COMPARISON_VALUES[
                _source
            ][_metric]
        )

        EXP004_SOURCE_MEANS_D_B[
            _source
        ][_metric] = (
            sum(_values)
            / len(_values)
        )


print(
    f"\n{'Source':<10}"
    + "".join(
        f"{_metric:>12}"
        for _metric in EXP004_ANALYSIS_APPROVED_METRICS
    )
)

print("-" * 60)

for _source in EXP004_ANALYSIS_SOURCES:

    print(
        f"{_source:<10}"
        + "".join(
            f"{EXP004_SOURCE_MEANS_D_B[_source][_metric]:>12.3f}"
            for _metric in EXP004_ANALYSIS_APPROVED_METRICS
        )
    )


# ============================================================================
# 4. SOURCE RANKING BY METRIC
# ============================================================================

print("\n" + "=" * 70)
print("SOURCE RANKING BY METRIC")
print("=" * 70)

EXP004_SOURCE_RANKINGS = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _ranking = sorted(
        EXP004_ANALYSIS_SOURCES,
        key=lambda _source:
            EXP004_SOURCE_MEANS_D_B[
                _source
            ][_metric],
        reverse=True,
    )

    EXP004_SOURCE_RANKINGS[
        _metric
    ] = _ranking

    print(f"\n{_metric}")
    print("-" * 40)

    for _rank, _source in enumerate(
        _ranking,
        start=1,
    ):

        print(
            f"  {_rank}. {_source:<7} "
            f"{EXP004_SOURCE_MEANS_D_B[_source][_metric]:.3f} dB"
        )


# ============================================================================
# 5. BEST / WORST SOURCE COMPARISON
# ============================================================================

print("\n" + "=" * 70)
print("BEST / WORST SOURCE ANALYSIS")
print("=" * 70)

EXP004_BEST_WORST_COMPARISON = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _ranking = EXP004_SOURCE_RANKINGS[
        _metric
    ]

    _best_source = _ranking[0]
    _worst_source = _ranking[-1]

    _best_value = (
        EXP004_SOURCE_MEANS_D_B[
            _best_source
        ][_metric]
    )

    _worst_value = (
        EXP004_SOURCE_MEANS_D_B[
            _worst_source
        ][_metric]
    )

    _gap = (
        _best_value
        - _worst_value
    )

    EXP004_BEST_WORST_COMPARISON[
        _metric
    ] = {
        "best_source": _best_source,
        "best_mean_dB": _best_value,
        "worst_source": _worst_source,
        "worst_mean_dB": _worst_value,
        "best_worst_gap_dB": _gap,
    }

    print(f"\n{_metric}")
    print(
        f"  Best:  {_best_source} "
        f"({_best_value:.3f} dB)"
    )
    print(
        f"  Worst: {_worst_source} "
        f"({_worst_value:.3f} dB)"
    )
    print(
        f"  Gap:   {_gap:.3f} dB"
    )


# ============================================================================
# 6. SOURCE PERFORMANCE PROFILES
# ============================================================================

print("\n" + "=" * 70)
print("SOURCE PERFORMANCE PROFILE")
print("=" * 70)

EXP004_SOURCE_PROFILES = {}

for _source in EXP004_ANALYSIS_SOURCES:

    _ranking_positions = {}

    for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

        _ranking_positions[
            _metric
        ] = (
            EXP004_SOURCE_RANKINGS[
                _metric
            ].index(_source)
            + 1
        )

    _average_rank = (
        sum(
            _ranking_positions.values()
        )
        / len(_ranking_positions)
    )

    EXP004_SOURCE_PROFILES[
        _source
    ] = {
        "means_dB": (
            EXP004_SOURCE_MEANS_D_B[
                _source
            ]
        ),
        "rankings": _ranking_positions,
        "average_rank": _average_rank,
    }

    print(f"\n{_source.upper()}")

    for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

        print(
            f"  {_metric:<6}: "
            f"{EXP004_SOURCE_MEANS_D_B[_source][_metric]:.3f} dB "
            f"(rank "
            f"{_ranking_positions[_metric]}/"
            f"{len(EXP004_ANALYSIS_SOURCES)})"
        )

    print(
        f"  Average rank: "
        f"{_average_rank:.2f}"
    )


# ============================================================================
# 7. OVERALL SOURCE RANKING
# ============================================================================

EXP004_OVERALL_SOURCE_RANKING = sorted(
    EXP004_ANALYSIS_SOURCES,
    key=lambda _source:
        EXP004_SOURCE_PROFILES[
            _source
        ]["average_rank"],
)


print("\n" + "=" * 70)
print("OVERALL SOURCE RANKING")
print("=" * 70)

for _rank, _source in enumerate(
    EXP004_OVERALL_SOURCE_RANKING,
    start=1,
):

    print(
        f"  {_rank}. {_source:<7} "
        f"(average rank: "
        f"{EXP004_SOURCE_PROFILES[_source]['average_rank']:.2f})"
    )


# ============================================================================
# 8. TRACK-LEVEL SOURCE COMPARISON
# ============================================================================

EXP004_TRACK_SOURCE_VALUES = {}

for _row in EXP004_SOURCE_COMPARISON_ROWS:

    EXP004_TRACK_SOURCE_VALUES.setdefault(
        _row["track"],
        {}
    )[_row["source"]] = {
        _metric: _row[_metric]
        for _metric in EXP004_ANALYSIS_APPROVED_METRICS
    }


EXP004_TRACK_LEVEL_SOURCE_COMPARISON = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _best_counts = {
        _source: 0
        for _source in EXP004_ANALYSIS_SOURCES
    }

    _worst_counts = {
        _source: 0
        for _source in EXP004_ANALYSIS_SOURCES
    }

    for _track in EXP004_ANALYSIS_TRACKS_FOUND:

        _values = {
            _source:
                EXP004_TRACK_SOURCE_VALUES[
                    _track
                ][_source][_metric]
            for _source in EXP004_ANALYSIS_SOURCES
        }

        _best_source = max(
            _values,
            key=_values.get,
        )

        _worst_source = min(
            _values,
            key=_values.get,
        )

        _best_counts[
            _best_source
        ] += 1

        _worst_counts[
            _worst_source
        ] += 1

    EXP004_TRACK_LEVEL_SOURCE_COMPARISON[
        _metric
    ] = {
        "best_source_count": _best_counts,
        "worst_source_count": _worst_counts,
    }


# ============================================================================
# 9. SAVE REPORT
# ============================================================================

EXP004_SOURCE_COMPARISON_REPORT = (
    EXP004_ANALYSIS_REPORTS_DIR
    / "EXP004_source_level_comparative_analysis.json"
)

EXP004_SOURCE_COMPARISON_REPORT_DATA = {
    "experiment": EXP004_ANALYSIS_EXPERIMENT,
    "generated_at": datetime.now().isoformat(),

    "scope": {
        "tracks": EXP004_ANALYSIS_TRACKS,
        "sources": EXP004_ANALYSIS_SOURCES,
        "approved_metrics": (
            EXP004_ANALYSIS_APPROVED_METRICS
        ),
        "excluded_metrics": (
            EXP004_ANALYSIS_EXCLUDED_METRICS
        ),
    },

    "input": {
        "source_level_csv": str(
            EXP004_ANALYSIS_SOURCE_CSV
        ),
        "source_level_rows": len(
            EXP004_SOURCE_COMPARISON_ROWS
        ),
    },

    "source_means_dB": (
        EXP004_SOURCE_MEANS_D_B
    ),

    "source_rankings": (
        EXP004_SOURCE_RANKINGS
    ),

    "best_worst_comparison": (
        EXP004_BEST_WORST_COMPARISON
    ),

    "source_profiles": (
        EXP004_SOURCE_PROFILES
    ),

    "overall_source_ranking": (
        EXP004_OVERALL_SOURCE_RANKING
    ),

    "track_level_source_comparison": (
        EXP004_TRACK_LEVEL_SOURCE_COMPARISON
    ),

    "policy": {
        "wav_files_loaded": False,
        "metrics_recomputed": False,
        "isr_used_downstream": False,
        "authoritative_source": (
            "EXP004_per_source_metrics.csv"
        ),
    },
}

with EXP004_SOURCE_COMPARISON_REPORT.open(
    "w",
    encoding="utf-8",
) as _f:

    json.dump(
        EXP004_SOURCE_COMPARISON_REPORT_DATA,
        _f,
        indent=2,
        ensure_ascii=False,
    )


print("\n" + "=" * 70)
print("CELL 18 RESULT")
print("=" * 70)

print(
    f"\nTracks analysed:       "
    f"{EXP004_ANALYSIS_TRACKS}"
)

print(
    f"Source evaluations:   "
    f"{EXP004_ANALYSIS_SOURCE_EVALUATIONS}"
)

print(
    f"Approved metrics:     "
    f"{len(EXP004_ANALYSIS_APPROVED_METRICS)}"
)

print("\nApproved metrics:")
for _metric in EXP004_ANALYSIS_APPROVED_METRICS:
    print(f"  ✓ {_metric}")

print("\nExcluded:")
print("  ✗ ISR")

print("\nAnalyses completed:")
print("  ✓ Source mean comparison")
print("  ✓ Source ranking by metric")
print("  ✓ Best/worst source analysis")
print("  ✓ Source performance profiles")
print("  ✓ Overall source ranking")
print("  ✓ Track-level source comparison")

print("\nReport:")
print(
    f"  {EXP004_SOURCE_COMPARISON_REPORT}"
)

print("\n" + "=" * 70)
print("✓ CELL 18 COMPLETE")
print("=" * 70)

print(
    "\nEXP004 source-level comparative analysis was completed "
    "from the existing metric outputs."
)

print(
    "No WAV files were loaded and no objective metrics were recomputed."
)

EXP004 SOURCE-LEVEL COMPARATIVE ANALYSIS

SOURCE MEAN COMPARISON

Source             SDR         SIR         SAR      SI-SDR
------------------------------------------------------------
vocals           8.674      18.305       9.060       7.796
drums            9.947      20.787      11.119       9.507
bass             8.616      18.001       9.592       7.758
other            6.167      11.862       7.146       4.706

SOURCE RANKING BY METRIC

SDR
----------------------------------------
  1. drums   9.947 dB
  2. vocals  8.674 dB
  3. bass    8.616 dB
  4. other   6.167 dB

SIR
----------------------------------------
  1. drums   20.787 dB
  2. vocals  18.305 dB
  3. bass    18.001 dB
  4. other   11.862 dB

SAR
----------------------------------------
  1. drums   11.119 dB
  2. bass    9.592 dB
  3. vocals  9.060 dB
  4. other   7.146 dB

SI-SDR
----------------------------------------
  1. drums   9.507 dB
  2. vocals  7.796 dB
  3. bass    7.758 dB
  4. other   4.706 dB

BEST / 

# EXP004 Track-Level Aggregate Analysis

In [ ]:
# ============================================================================
# CELL 19 — EXP004 TRACK-LEVEL AGGREGATE ANALYSIS
# ============================================================================

import json
import math
import statistics
from datetime import datetime


print("=" * 70)
print("EXP004 TRACK-LEVEL AGGREGATE ANALYSIS")
print("=" * 70)


if "EXP004_ANALYSIS_PATHS" not in globals():
    raise RuntimeError(
        "EXP004 analysis configuration is unavailable. "
        "Run Cell 16 first."
    )


# ============================================================================
# 1. LOAD JSONL
# ============================================================================

EXP004_TRACK_AGGREGATE_RECORDS = []

with EXP004_ANALYSIS_TRACK_JSONL.open(
    "r",
    encoding="utf-8",
) as _f:

    for _line_number, _line in enumerate(
        _f,
        start=1,
    ):

        _line = _line.strip()

        if not _line:
            continue

        try:
            _record = json.loads(_line)
        except json.JSONDecodeError as _exc:
            raise ValueError(
                f"Invalid JSON on line "
                f"{_line_number}: {_exc}"
            )

        EXP004_TRACK_AGGREGATE_RECORDS.append(
            _record
        )


if (
    len(EXP004_TRACK_AGGREGATE_RECORDS)
    != EXP004_ANALYSIS_TRACKS
):
    raise ValueError(
        f"Expected {EXP004_ANALYSIS_TRACKS} "
        f"track records, found "
        f"{len(EXP004_TRACK_AGGREGATE_RECORDS)}."
    )


# ============================================================================
# 2. ACTUAL JSONL METRIC KEY MAP
# ============================================================================

EXP004_TRACK_SOURCE_METRIC_KEYS = {
    "SDR": "SDR_dB",
    "SIR": "SIR_dB",
    "SAR": "SAR_dB",
    "SI-SDR": "SI_SDR_dB",
}


# ============================================================================
# 3. VALIDATE RECORDS AND COLLECT TRACK AVERAGES
# ============================================================================

EXP004_TRACK_IDS = []

EXP004_TRACK_AVERAGE_VALUES = {}

for _record_index, _record in enumerate(
    EXP004_TRACK_AGGREGATE_RECORDS,
    start=1,
):

    if _record.get("status") != "success":
        raise ValueError(
            f"Record {_record_index} does not have "
            "status='success'."
        )

    _track = _record.get("track")

    if not _track:
        raise ValueError(
            f"Record {_record_index} has no track identifier."
        )

    EXP004_TRACK_IDS.append(
        _track
    )

    _sources = _record.get("sources")

    if not isinstance(
        _sources,
        dict,
    ):
        raise ValueError(
            f"Missing sources structure for {_track}."
        )

    if set(_sources.keys()) != set(
        EXP004_ANALYSIS_SOURCES
    ):
        raise ValueError(
            f"Source mismatch for {_track}."
        )

    for _source in EXP004_ANALYSIS_SOURCES:

        _source_data = _sources[_source]

        if not isinstance(
            _source_data,
            dict,
        ):
            raise ValueError(
                f"Invalid source data for "
                f"{_track} / {_source}."
            )

        for _metric, _key in (
            EXP004_TRACK_SOURCE_METRIC_KEYS.items()
        ):

            if _key not in _source_data:
                raise ValueError(
                    f"Missing {_key} for "
                    f"{_track} / {_source}."
                )

            _value = float(
                _source_data[_key]
            )

            if not math.isfinite(_value):
                raise ValueError(
                    f"Non-finite {_metric} for "
                    f"{_track} / {_source}."
                )

    _track_average = _record.get(
        "track_average"
    )

    if not isinstance(
        _track_average,
        dict,
    ):
        raise ValueError(
            f"Missing track_average for {_track}."
        )

    EXP004_TRACK_AVERAGE_VALUES[
        _track
    ] = {}

    for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

        if _metric not in _track_average:
            raise ValueError(
                f"Missing track-average {_metric} "
                f"for {_track}."
            )

        _value = float(
            _track_average[_metric]
        )

        if not math.isfinite(_value):
            raise ValueError(
                f"Non-finite track-average {_metric} "
                f"for {_track}."
            )

        EXP004_TRACK_AVERAGE_VALUES[
            _track
        ][_metric] = _value


if (
    len(set(EXP004_TRACK_IDS))
    != len(EXP004_TRACK_IDS)
):
    raise ValueError(
        "Duplicate track identifiers detected."
    )


print(
    f"Tracks validated: "
    f"{len(EXP004_TRACK_IDS)}"
)

print(
    "✓ All source-level structures are valid"
)

print(
    "✓ All stored track-average metrics are valid"
)


# ============================================================================
# 4. TRACK-AVERAGE DESCRIPTIVE STATISTICS
# ============================================================================

print("\n" + "=" * 70)
print("TRACK-AVERAGE METRIC STATISTICS")
print("=" * 70)


def EXP004_track_percentile(
    _data,
    _p,
):

    _sorted = sorted(_data)

    if len(_sorted) == 1:
        return _sorted[0]

    _position = (
        len(_sorted) - 1
    ) * _p

    _lower = int(
        math.floor(_position)
    )

    _upper = int(
        math.ceil(_position)
    )

    if _lower == _upper:
        return _sorted[_lower]

    _fraction = (
        _position
        - _lower
    )

    return (
        _sorted[_lower]
        + _fraction
        * (
            _sorted[_upper]
            - _sorted[_lower]
        )
    )


EXP004_TRACK_AGGREGATE_STATISTICS = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _values = [
        EXP004_TRACK_AVERAGE_VALUES[
            _track
        ][_metric]
        for _track in EXP004_TRACK_IDS
    ]

    _q1 = EXP004_track_percentile(
        _values,
        0.25,
    )

    _q3 = EXP004_track_percentile(
        _values,
        0.75,
    )

    _iqr = _q3 - _q1

    _lower_fence = (
        _q1
        - 1.5 * _iqr
    )

    _upper_fence = (
        _q3
        + 1.5 * _iqr
    )

    _outlier_count = sum(
        _value < _lower_fence
        or _value > _upper_fence
        for _value in _values
    )

    EXP004_TRACK_AGGREGATE_STATISTICS[
        _metric
    ] = {
        "count": len(_values),
        "mean_dB": statistics.mean(
            _values
        ),
        "median_dB": statistics.median(
            _values
        ),
        "std_dB": (
            statistics.stdev(
                _values
            )
            if len(_values) > 1
            else 0.0
        ),
        "min_dB": min(_values),
        "q1_dB": _q1,
        "q3_dB": _q3,
        "max_dB": max(_values),
        "iqr_dB": _iqr,
        "lower_fence_dB": _lower_fence,
        "upper_fence_dB": _upper_fence,
        "iqr_outlier_count": _outlier_count,
    }

    _stats = (
        EXP004_TRACK_AGGREGATE_STATISTICS[
            _metric
        ]
    )

    print(f"\n{_metric}")
    print("-" * 40)
    print(f"  Count:        {_stats['count']}")
    print(f"  Mean:         {_stats['mean_dB']:.3f} dB")
    print(f"  Median:       {_stats['median_dB']:.3f} dB")
    print(f"  Std:          {_stats['std_dB']:.3f} dB")
    print(f"  Min:          {_stats['min_dB']:.3f} dB")
    print(f"  Q1:           {_stats['q1_dB']:.3f} dB")
    print(f"  Q3:           {_stats['q3_dB']:.3f} dB")
    print(f"  Max:          {_stats['max_dB']:.3f} dB")
    print(f"  IQR:          {_stats['iqr_dB']:.3f} dB")
    print(
        f"  IQR outliers: "
        f"{_stats['iqr_outlier_count']}"
    )


# ============================================================================
# 5. TRACK-LEVEL RANKINGS
# ============================================================================

print("\n" + "=" * 70)
print("TRACK-LEVEL PERFORMANCE RANKING")
print("=" * 70)

EXP004_TRACK_RANKINGS = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _ranking = sorted(
        [
            {
                "track": _track,
                "value_dB":
                    EXP004_TRACK_AVERAGE_VALUES[
                        _track
                    ][_metric],
            }
            for _track in EXP004_TRACK_IDS
        ],
        key=lambda _item:
            _item["value_dB"],
        reverse=True,
    )

    EXP004_TRACK_RANKINGS[
        _metric
    ] = _ranking

    print(f"\n{_metric}")
    print("-" * 40)

    print("  Highest:")

    for _item in _ranking[:5]:
        print(
            f"    {_item['value_dB']:8.3f} dB | "
            f"{_item['track']}"
        )

    print("  Lowest:")

    for _item in _ranking[-5:][::-1]:
        print(
            f"    {_item['value_dB']:8.3f} dB | "
            f"{_item['track']}"
        )


# ============================================================================
# 6. BEST / WORST METRIC PER TRACK
# ============================================================================

EXP004_TRACK_SUMMARY = {}

for _track in EXP004_TRACK_IDS:

    _values = {
        _metric:
            EXP004_TRACK_AVERAGE_VALUES[
                _track
            ][_metric]
        for _metric in EXP004_ANALYSIS_APPROVED_METRICS
    }

    _mean_across_metrics = statistics.mean(
        _values.values()
    )

    _best_metric = max(
        _values,
        key=_values.get,
    )

    _worst_metric = min(
        _values,
        key=_values.get,
    )

    EXP004_TRACK_SUMMARY[
        _track
    ] = {
        "metrics": _values,
        "mean_across_approved_metrics_dB":
            _mean_across_metrics,
        "best_metric": _best_metric,
        "worst_metric": _worst_metric,
    }


EXP004_OVERALL_TRACK_RANKING = sorted(
    [
        {
            "track": _track,
            "mean_across_approved_metrics_dB":
                EXP004_TRACK_SUMMARY[
                    _track
                ][
                    "mean_across_approved_metrics_dB"
                ],
        }
        for _track in EXP004_TRACK_IDS
    ],
    key=lambda _item:
        _item[
            "mean_across_approved_metrics_dB"
        ],
    reverse=True,
)


print(
    "\nOverall track ranking using mean "
    "of approved metrics:"
)

print("\n  Highest-performing tracks:")

for _item in EXP004_OVERALL_TRACK_RANKING[:5]:

    print(
        f"    "
        f"{_item['mean_across_approved_metrics_dB']:.3f} dB | "
        f"{_item['track']}"
    )

print("\n  Lowest-performing tracks:")

for _item in EXP004_OVERALL_TRACK_RANKING[-5:][::-1]:

    print(
        f"    "
        f"{_item['mean_across_approved_metrics_dB']:.3f} dB | "
        f"{_item['track']}"
    )


# ============================================================================
# 7. STORED TRACK-AVERAGE CONSISTENCY
# ============================================================================

print("\n" + "=" * 70)
print("TRACK-AVERAGE CONSISTENCY CHECK")
print("=" * 70)

EXP004_TRACK_AVERAGE_CONSISTENCY = []

_EXP004_TRACK_AVERAGE_TOLERANCE = 1e-9

for _record in EXP004_TRACK_AGGREGATE_RECORDS:

    _track = _record["track"]

    _sources = _record["sources"]
    _stored_average = _record["track_average"]

    for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

        _key = EXP004_TRACK_SOURCE_METRIC_KEYS[
            _metric
        ]

        _source_values = [
            float(
                _sources[_source][_key]
            )
            for _source in EXP004_ANALYSIS_SOURCES
        ]

        _calculated_average = statistics.mean(
            _source_values
        )

        _stored_value = float(
            _stored_average[_metric]
        )

        _difference = abs(
            _calculated_average
            - _stored_value
        )

        _consistent = (
            _difference
            <= _EXP004_TRACK_AVERAGE_TOLERANCE
        )

        EXP004_TRACK_AVERAGE_CONSISTENCY.append(
            {
                "track": _track,
                "metric": _metric,
                "stored_track_average_dB":
                    _stored_value,
                "calculated_source_mean_dB":
                    _calculated_average,
                "absolute_difference_dB":
                    _difference,
                "consistent": _consistent,
            }
        )

        if not _consistent:
            raise ValueError(
                f"Track-average inconsistency detected "
                f"for {_track} / {_metric}: "
                f"stored={_stored_value}, "
                f"calculated={_calculated_average}"
            )


print(
    f"Consistency checks performed: "
    f"{len(EXP004_TRACK_AVERAGE_CONSISTENCY)}"
)

print(
    "✓ All stored track averages equal the mean "
    "of the four source values"
)


# ============================================================================
# 8. SAVE REPORT
# ============================================================================

EXP004_TRACK_AGGREGATE_REPORT = (
    EXP004_ANALYSIS_REPORTS_DIR
    / "EXP004_track_level_aggregate_analysis.json"
)

EXP004_TRACK_AGGREGATE_REPORT_DATA = {
    "experiment": EXP004_ANALYSIS_EXPERIMENT,
    "analysis": "track_level_aggregate_analysis",

    "data_source": {
        "jsonl": str(
            EXP004_ANALYSIS_TRACK_JSONL
        ),
        "tracks": len(
            EXP004_TRACK_IDS
        ),
        "sources_per_track": len(
            EXP004_ANALYSIS_SOURCES
        ),
        "source_evaluations": (
            len(EXP004_TRACK_IDS)
            * len(EXP004_ANALYSIS_SOURCES)
        ),
    },

    "approved_metrics": (
        EXP004_ANALYSIS_APPROVED_METRICS
    ),

    "excluded_metrics": (
        EXP004_ANALYSIS_EXCLUDED_METRICS
    ),

    "track_average_statistics": (
        EXP004_TRACK_AGGREGATE_STATISTICS
    ),

    "track_rankings": (
        EXP004_TRACK_RANKINGS
    ),

    "track_summary": (
        EXP004_TRACK_SUMMARY
    ),

    "overall_track_ranking": (
        EXP004_OVERALL_TRACK_RANKING
    ),

    "track_average_consistency": {
        "checks_performed":
            len(
                EXP004_TRACK_AVERAGE_CONSISTENCY
            ),
        "all_consistent":
            all(
                _item["consistent"]
                for _item in (
                    EXP004_TRACK_AVERAGE_CONSISTENCY
                )
            ),
        "tolerance_dB":
            _EXP004_TRACK_AVERAGE_TOLERANCE,
    },
}


with EXP004_TRACK_AGGREGATE_REPORT.open(
    "w",
    encoding="utf-8",
) as _f:

    json.dump(
        EXP004_TRACK_AGGREGATE_REPORT_DATA,
        _f,
        indent=2,
        ensure_ascii=False,
    )


print("\n" + "=" * 70)
print("CELL 19 RESULT")
print("=" * 70)

print(
    f"\nTracks analysed:       "
    f"{len(EXP004_TRACK_IDS)}"
)

print(
    f"Source evaluations:   "
    f"{len(EXP004_TRACK_IDS) * len(EXP004_ANALYSIS_SOURCES)}"
)

print(
    f"Approved metrics:     "
    f"{len(EXP004_ANALYSIS_APPROVED_METRICS)}"
)

print("\nApproved metrics:")
for _metric in EXP004_ANALYSIS_APPROVED_METRICS:
    print(f"  ✓ {_metric}")

print("\nExcluded:")
print("  ✗ ISR")

print("\nAnalyses completed:")
print("  ✓ Track-average descriptive statistics")
print("  ✓ Track-level metric rankings")
print("  ✓ Highest/lowest performing tracks")
print("  ✓ Cross-metric track performance summary")
print("  ✓ Stored track-average consistency validation")

print("\nReport:")
print(
    f"  {EXP004_TRACK_AGGREGATE_REPORT}"
)

print("\n" + "=" * 70)
print("✓ CELL 19 COMPLETE")
print("=" * 70)

print(
    "\nEXP004 track-level aggregate analysis was completed "
    "from the existing metric outputs."
)

print(
    "No WAV files were loaded and no objective metrics were recomputed."
)

EXP004 TRACK-LEVEL AGGREGATE ANALYSIS
Tracks validated: 50
✓ All source-level structures are valid
✓ All stored track-average metrics are valid

TRACK-AVERAGE METRIC STATISTICS

SDR
----------------------------------------
  Count:        50
  Mean:         8.351 dB
  Median:       8.804 dB
  Std:          2.032 dB
  Min:          3.302 dB
  Q1:           6.984 dB
  Q3:           9.657 dB
  Max:          11.910 dB
  IQR:          2.673 dB
  IQR outliers: 0

SIR
----------------------------------------
  Count:        50
  Mean:         17.239 dB
  Median:       17.367 dB
  Std:          3.098 dB
  Min:          9.158 dB
  Q1:           14.494 dB
  Q3:           19.616 dB
  Max:          23.113 dB
  IQR:          5.122 dB
  IQR outliers: 0

SAR
----------------------------------------
  Count:        50
  Mean:         9.229 dB
  Median:       9.504 dB
  Std:          1.889 dB
  Min:          4.702 dB
  Q1:           7.935 dB
  Q3:           10.517 dB
  Max:          12.960 dB
  IQR:   

# EXP004 Distribution & Outlier Analysis

In [ ]:
# ============================================================================
# CELL 20 — EXP004 DISTRIBUTION & OUTLIER ANALYSIS
# ============================================================================

import csv
import json
import math
from datetime import datetime

import matplotlib.pyplot as plt


print("=" * 70)
print("EXP004 DISTRIBUTION & OUTLIER ANALYSIS")
print("=" * 70)


if "EXP004_ANALYSIS_PATHS" not in globals():
    raise RuntimeError(
        "EXP004 analysis configuration is unavailable. "
        "Run Cell 16 first."
    )


# ============================================================================
# 1. LOAD SOURCE-LEVEL METRIC DATA
# ============================================================================

EXP004_DISTRIBUTION_ROWS = []

with EXP004_ANALYSIS_SOURCE_CSV.open(
    "r",
    encoding="utf-8",
    newline="",
) as _f:

    _reader = csv.DictReader(_f)

    for _row in _reader:

        _parsed = {
            "track":
                _row[
                    EXP004_ANALYSIS_TRACK_COLUMN
                ].strip(),

            "source":
                _row[
                    EXP004_ANALYSIS_SOURCE_COLUMN
                ].strip().lower(),
        }

        for _metric, _column in (
            EXP004_ANALYSIS_METRIC_COLUMN_MAP.items()
        ):

            _value = float(
                _row[_column]
            )

            if not math.isfinite(_value):
                raise ValueError(
                    f"Non-finite {_metric} value "
                    f"for {_parsed['track']} / "
                    f"{_parsed['source']}."
                )

            _parsed[_metric] = _value

        EXP004_DISTRIBUTION_ROWS.append(
            _parsed
        )


if (
    len(EXP004_DISTRIBUTION_ROWS)
    != EXP004_ANALYSIS_SOURCE_EVALUATIONS
):
    raise ValueError(
        "Unexpected source-level row count."
    )


# ============================================================================
# 2. NUMERIC GROUPS
# ============================================================================

EXP004_DISTRIBUTION_METRIC_VALUES = {
    _metric: [
        _row[_metric]
        for _row in EXP004_DISTRIBUTION_ROWS
    ]
    for _metric in EXP004_ANALYSIS_APPROVED_METRICS
}

EXP004_DISTRIBUTION_SOURCE_METRIC_VALUES = {
    _source: {
        _metric: [
            _row[_metric]
            for _row in EXP004_DISTRIBUTION_ROWS
            if _row["source"] == _source
        ]
        for _metric in EXP004_ANALYSIS_APPROVED_METRICS
    }
    for _source in EXP004_ANALYSIS_SOURCES
}


# ============================================================================
# 3. DESCRIPTIVE / IQR FUNCTION
# ============================================================================

def EXP004_distribution_statistics(
    _values,
):

    _values = sorted(
        float(_value)
        for _value in _values
    )

    if not _values:
        raise ValueError(
            "Cannot analyse an empty metric distribution."
        )

    _count = len(_values)

    _mean = (
        sum(_values)
        / _count
    )

    if _count % 2 == 1:
        _median = _values[
            _count // 2
        ]
    else:
        _median = (
            _values[
                _count // 2 - 1
            ]
            + _values[
                _count // 2
            ]
        ) / 2.0

    _variance = (
        sum(
            (
                _value
                - _mean
            ) ** 2
            for _value in _values
        )
        / _count
    )

    _std = math.sqrt(
        _variance
    )

    def _percentile(_data, _p):

        if len(_data) == 1:
            return _data[0]

        _position = (
            len(_data) - 1
        ) * _p

        _lower = int(
            math.floor(_position)
        )

        _upper = int(
            math.ceil(_position)
        )

        if _lower == _upper:
            return _data[_lower]

        _fraction = (
            _position
            - _lower
        )

        return (
            _data[_lower]
            + _fraction
            * (
                _data[_upper]
                - _data[_lower]
            )
        )

    _q1 = _percentile(
        _values,
        0.25,
    )

    _q3 = _percentile(
        _values,
        0.75,
    )

    _iqr = (
        _q3
        - _q1
    )

    _lower_bound = (
        _q1
        - 1.5 * _iqr
    )

    _upper_bound = (
        _q3
        + 1.5 * _iqr
    )

    _outliers = [
        _value
        for _value in _values
        if (
            _value < _lower_bound
            or _value > _upper_bound
        )
    ]

    return {
        "count": _count,
        "mean": _mean,
        "median": _median,
        "std": _std,
        "min": _values[0],
        "q1": _q1,
        "q3": _q3,
        "max": _values[-1],
        "iqr": _iqr,
        "lower_bound": _lower_bound,
        "upper_bound": _upper_bound,
        "iqr_outlier_count": len(
            _outliers
        ),
    }


# ============================================================================
# 4. OVERALL DISTRIBUTIONS
# ============================================================================

print("\n" + "=" * 70)
print("OVERALL METRIC DISTRIBUTIONS")
print("=" * 70)

EXP004_DISTRIBUTION_OVERALL_STATS = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _stats = EXP004_distribution_statistics(
        EXP004_DISTRIBUTION_METRIC_VALUES[
            _metric
        ]
    )

    EXP004_DISTRIBUTION_OVERALL_STATS[
        _metric
    ] = _stats

    print(f"\n{_metric}")
    print("-" * 40)
    print(f"  Count:   {_stats['count']}")
    print(f"  Mean:    {_stats['mean']:.3f} dB")
    print(f"  Median:  {_stats['median']:.3f} dB")
    print(f"  Std:     {_stats['std']:.3f} dB")
    print(f"  Min:     {_stats['min']:.3f} dB")
    print(f"  Q1:      {_stats['q1']:.3f} dB")
    print(f"  Q3:      {_stats['q3']:.3f} dB")
    print(f"  Max:     {_stats['max']:.3f} dB")
    print(f"  IQR:     {_stats['iqr']:.3f} dB")
    print(
        f"  IQR outliers: "
        f"{_stats['iqr_outlier_count']}"
    )


# ============================================================================
# 5. SOURCE-BY-SOURCE DISTRIBUTIONS
# ============================================================================

print("\n" + "=" * 70)
print("SOURCE-BY-SOURCE DISTRIBUTIONS")
print("=" * 70)

EXP004_DISTRIBUTION_SOURCE_STATS = {}

for _source in EXP004_ANALYSIS_SOURCES:

    EXP004_DISTRIBUTION_SOURCE_STATS[
        _source
    ] = {}

    print(f"\n{_source.upper()}")
    print("-" * 70)

    for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

        _stats = EXP004_distribution_statistics(
            EXP004_DISTRIBUTION_SOURCE_METRIC_VALUES[
                _source
            ][_metric]
        )

        EXP004_DISTRIBUTION_SOURCE_STATS[
            _source
        ][_metric] = _stats

        print(
            f"{_metric:<8}"
            f"{_stats['mean']:>9.3f}"
            f"{_stats['median']:>9.3f}"
            f"{_stats['std']:>9.3f}"
            f"{_stats['min']:>9.3f}"
            f"{_stats['q1']:>9.3f}"
            f"{_stats['q3']:>9.3f}"
            f"{_stats['max']:>9.3f}"
            f"{_stats['iqr_outlier_count']:>10}"
        )


# ============================================================================
# 6. LOWEST / HIGHEST SOURCE EVALUATIONS
# ============================================================================

print("\n" + "=" * 70)
print("LOWEST / HIGHEST SOURCE EVALUATIONS")
print("=" * 70)

EXP004_EXTREME_EVALUATIONS = {}

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _metric_rows = [
        {
            "track": _row["track"],
            "source": _row["source"],
            "value": float(
                _row[_metric]
            ),
        }
        for _row in EXP004_DISTRIBUTION_ROWS
    ]

    _metric_rows.sort(
        key=lambda _item:
            _item["value"]
    )

    _lowest = _metric_rows[:5]

    _highest = list(
        reversed(
            _metric_rows[-5:]
        )
    )

    EXP004_EXTREME_EVALUATIONS[
        _metric
    ] = {
        "lowest": _lowest,
        "highest": _highest,
    }

    print(f"\n{_metric}")
    print("-" * 40)

    print("Lowest:")

    for _item in _lowest:
        print(
            f"  {_item['value']:.3f} dB | "
            f"{_item['track']} | "
            f"{_item['source']}"
        )

    print("Highest:")

    for _item in _highest:
        print(
            f"  {_item['value']:.3f} dB | "
            f"{_item['track']} | "
            f"{_item['source']}"
        )


# ============================================================================
# 7. EXACTLY EIGHT REQUIRED PLOTS
# ============================================================================

print("\n" + "=" * 70)
print("GENERATING EXACTLY 8 REQUIRED PLOTS")
print("=" * 70)

EXP004_DISTRIBUTION_PLOT_NAMES = {
    "SDR": "EXP004_sdr_distribution.png",
    "SIR": "EXP004_sir_distribution.png",
    "SAR": "EXP004_sar_distribution.png",
    "SI-SDR": "EXP004_si_sdr_distribution.png",
}

EXP004_BOXPLOT_NAMES = {
    "SDR": "EXP004_sdr_by_source_boxplot.png",
    "SIR": "EXP004_sir_by_source_boxplot.png",
    "SAR": "EXP004_sar_by_source_boxplot.png",
    "SI-SDR": "EXP004_si_sdr_by_source_boxplot.png",
}

EXP004_REQUIRED_PLOT_PATHS = []


# ----------------------------------------------------------------
# 7A. Four distribution histograms
# ----------------------------------------------------------------

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    plt.figure(
        figsize=(8, 5)
    )

    plt.hist(
        EXP004_DISTRIBUTION_METRIC_VALUES[
            _metric
        ],
        bins=20,
    )

    plt.xlabel(
        f"{_metric} (dB)"
    )

    plt.ylabel(
        "Frequency"
    )

    plt.title(
        f"EXP004 {_metric} Distribution"
    )

    plt.tight_layout()

    _output_path = (
        EXP004_ANALYSIS_PLOTS_DIR
        / EXP004_DISTRIBUTION_PLOT_NAMES[
            _metric
        ]
    )

    plt.savefig(
        _output_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close()

    EXP004_REQUIRED_PLOT_PATHS.append(
        _output_path
    )

    print(
        f"✓ {_output_path}"
    )


# ----------------------------------------------------------------
# 7B. Four source comparison boxplots
# ----------------------------------------------------------------

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _source_values = [
        EXP004_DISTRIBUTION_SOURCE_METRIC_VALUES[
            _source
        ][_metric]
        for _source in EXP004_ANALYSIS_SOURCES
    ]

    plt.figure(
        figsize=(8, 5)
    )

    plt.boxplot(
        _source_values,
        tick_labels=EXP004_ANALYSIS_SOURCES,
    )

    plt.xlabel(
        "Source"
    )

    plt.ylabel(
        f"{_metric} (dB)"
    )

    plt.title(
        f"EXP004 {_metric} by Source"
    )

    plt.tight_layout()

    _output_path = (
        EXP004_ANALYSIS_PLOTS_DIR
        / EXP004_BOXPLOT_NAMES[
            _metric
        ]
    )

    plt.savefig(
        _output_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close()

    EXP004_REQUIRED_PLOT_PATHS.append(
        _output_path
    )

    print(
        f"✓ {_output_path}"
    )


if len(
    EXP004_REQUIRED_PLOT_PATHS
) != 8:
    raise RuntimeError(
        "EXP004 did not generate exactly "
        "8 required plot paths."
    )


# ============================================================================
# 8. BUILD REPORT
# ============================================================================

EXP004_DISTRIBUTION_REPORT = (
    EXP004_ANALYSIS_REPORTS_DIR
    / "EXP004_distribution_outlier_analysis.json"
)

EXP004_DISTRIBUTION_REPORT_DATA = {
    "experiment": EXP004_ANALYSIS_EXPERIMENT,
    "generated_at": datetime.now().isoformat(),

    "evaluation_scope": {
        "tracks": EXP004_ANALYSIS_TRACKS,
        "sources_per_track": len(
            EXP004_ANALYSIS_SOURCES
        ),
        "source_evaluations": (
            len(EXP004_DISTRIBUTION_ROWS)
        ),
        "sources": EXP004_ANALYSIS_SOURCES,
        "approved_metrics": (
            EXP004_ANALYSIS_APPROVED_METRICS
        ),
        "excluded_metrics": (
            EXP004_ANALYSIS_EXCLUDED_METRICS
        ),
    },

    "overall_statistics": (
        EXP004_DISTRIBUTION_OVERALL_STATS
    ),

    "source_statistics": (
        EXP004_DISTRIBUTION_SOURCE_STATS
    ),

    "extreme_evaluations": (
        EXP004_EXTREME_EVALUATIONS
    ),

    "plots": [
        str(_path)
        for _path in EXP004_REQUIRED_PLOT_PATHS
    ],

    "policy": {
        "audio_loaded": False,
        "metrics_recomputed": False,
        "source_level_values_used": True,
        "whole_track_objective_evaluation": False,
        "analysis_only": True,
        "outlier_method": "IQR",
    },
}


with EXP004_DISTRIBUTION_REPORT.open(
    "w",
    encoding="utf-8",
) as _f:

    json.dump(
        EXP004_DISTRIBUTION_REPORT_DATA,
        _f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================================
# 9. FINAL OUTPUT VALIDATION
# ============================================================================

print("\n" + "=" * 70)
print("FINAL OUTPUT VALIDATION")
print("=" * 70)

for _plot_path in EXP004_REQUIRED_PLOT_PATHS:

    if not _plot_path.exists():
        raise FileNotFoundError(
            f"Expected plot was not generated:\n"
            f"{_plot_path}"
        )

    if _plot_path.stat().st_size == 0:
        raise ValueError(
            f"Generated plot is empty:\n"
            f"{_plot_path}"
        )

print(
    "✓ Exactly 8 plot files validated"
)

if not EXP004_DISTRIBUTION_REPORT.exists():
    raise FileNotFoundError(
        "Distribution report was not created."
    )

with EXP004_DISTRIBUTION_REPORT.open(
    "r",
    encoding="utf-8",
) as _f:

    _validation_report = json.load(
        _f
    )

if (
    _validation_report.get(
        "experiment"
    )
    != EXP004_ANALYSIS_EXPERIMENT
):
    raise ValueError(
        "Distribution report experiment identifier "
        "is incorrect."
    )

print(
    "✓ Distribution report validated"
)


# ============================================================================
# 10. FINAL RESULT
# ============================================================================

print("\n" + "=" * 70)
print("CELL 20 RESULT")
print("=" * 70)

print(
    f"\nTracks analysed:       "
    f"{EXP004_ANALYSIS_TRACKS}"
)

print(
    f"Source evaluations:   "
    f"{EXP004_ANALYSIS_SOURCE_EVALUATIONS}"
)

print(
    "Approved metrics:     "
    f"{len(EXP004_ANALYSIS_APPROVED_METRICS)}"
)

print("\nApproved metrics:")

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:
    print(f"  ✓ {_metric}")

print("\nExcluded:")
print("  ✗ ISR")

print("\nAnalyses completed:")
print("  ✓ Overall distributions")
print("  ✓ Source-by-source distributions")
print("  ✓ IQR-based outlier detection")
print("  ✓ Lowest/highest source evaluations")
print("  ✓ Four distribution histograms")
print("  ✓ Four source comparison boxplots")

print("\nPlots:")
for _plot_path in EXP004_REQUIRED_PLOT_PATHS:
    print(f"  - {_plot_path}")

print("\nAnalysis report:")
print(
    f"  {EXP004_DISTRIBUTION_REPORT}"
)

print("\n" + "=" * 70)
print("✓ CELL 20 COMPLETE")
print("=" * 70)

print(
    "\nEXP004 distributions and outliers were analysed "
    "from the existing metric outputs."
)

print(
    "No WAV files were loaded and no objective metrics "
    "were recomputed."
)

EXP004 DISTRIBUTION & OUTLIER ANALYSIS

OVERALL METRIC DISTRIBUTIONS

SDR
----------------------------------------
  Count:   200
  Mean:    8.351 dB
  Median:  8.462 dB
  Std:     3.694 dB
  Min:     -3.361 dB
  Q1:      6.380 dB
  Q3:      10.644 dB
  Max:     22.163 dB
  IQR:     4.264 dB
  IQR outliers: 6

SIR
----------------------------------------
  Count:   200
  Mean:    17.239 dB
  Median:  17.811 dB
  Std:     6.382 dB
  Min:     -2.599 dB
  Q1:      13.236 dB
  Q3:      21.009 dB
  Max:     38.809 dB
  IQR:     7.773 dB
  IQR outliers: 5

SAR
----------------------------------------
  Count:   200
  Mean:    9.229 dB
  Median:  9.304 dB
  Std:     3.843 dB
  Min:     -4.288 dB
  Q1:      7.203 dB
  Q3:      11.403 dB
  Max:     23.533 dB
  IQR:     4.201 dB
  IQR outliers: 7

SI-SDR
----------------------------------------
  Count:   200
  Mean:    7.442 dB
  Median:  7.833 dB
  Std:     4.511 dB
  Min:     -9.244 dB
  Q1:      5.283 dB
  Q3:      10.263 dB
  Max:     22.21

# EXP004 Consolidated Objective Analysis


In [ ]:
# ============================================================================
# CELL 21 — EXP004 CONSOLIDATED OBJECTIVE ANALYSIS
# ============================================================================

import json
from datetime import datetime


print("=" * 70)
print("EXP004 CONSOLIDATED OBJECTIVE ANALYSIS")
print("=" * 70)


if "EXP004_ANALYSIS_PATHS" not in globals():
    raise RuntimeError(
        "EXP004 analysis configuration is unavailable. "
        "Run Cell 16 first."
    )


# ============================================================================
# 1. REPORT PATHS
# ============================================================================

EXP004_CONSOLIDATED_REPORT_PATHS = {
    "integrity":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_metric_integrity_audit.json",

    "descriptive":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_metric_descriptive_statistics.json",

    "distribution":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_distribution_outlier_analysis.json",

    "source_comparison":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_source_level_comparative_analysis.json",

    "track_aggregate":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_track_level_aggregate_analysis.json",
}


# ============================================================================
# 2. INPUT VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("INPUT VALIDATION")
print("-" * 70)

for _name, _path in (
    EXP004_CONSOLIDATED_REPORT_PATHS.items()
):

    if not _path.exists():
        raise FileNotFoundError(
            f"Required EXP004 report not found:\n{_path}"
        )

    if _path.stat().st_size == 0:
        raise ValueError(
            f"Required EXP004 report is empty:\n{_path}"
        )

    print(
        f"✓ {_name.replace('_', ' ').title()}: "
        f"{_path}"
    )


# ============================================================================
# 3. LOAD REPORTS
# ============================================================================

print("\n" + "-" * 70)
print("LOADING EXISTING EXP004 REPORTS")
print("-" * 70)

EXP004_CONSOLIDATED_LOADED_REPORTS = {}

for _name, _path in (
    EXP004_CONSOLIDATED_REPORT_PATHS.items()
):

    with _path.open(
        "r",
        encoding="utf-8",
    ) as _f:

        EXP004_CONSOLIDATED_LOADED_REPORTS[
            _name
        ] = json.load(_f)

    print(
        f"✓ {_name.replace('_', ' ').title()} loaded"
    )


# ============================================================================
# 4. BUILD CONSOLIDATED REPORT
# ============================================================================

EXP004_CONSOLIDATED_REPORT = (
    EXP004_ANALYSIS_REPORTS_DIR
    / "EXP004_consolidated_objective_analysis.json"
)

EXP004_CONSOLIDATED_REPORT_DATA = {
    "experiment": EXP004_ANALYSIS_EXPERIMENT,
    "timestamp": datetime.now().isoformat(),

    "configuration": {
        "tracks": EXP004_ANALYSIS_TRACKS,
        "sources_per_track": len(
            EXP004_ANALYSIS_SOURCES
        ),
        "source_evaluations": (
            EXP004_ANALYSIS_SOURCE_EVALUATIONS
        ),
        "sources": EXP004_ANALYSIS_SOURCES,
        "approved_metrics": (
            EXP004_ANALYSIS_APPROVED_METRICS
        ),
        "excluded_metrics": (
            EXP004_ANALYSIS_EXCLUDED_METRICS
        ),
    },

    "objective_evaluation_policy": {
        "metric_recomputation_in_analysis_cells": False,
        "wav_files_loaded_in_analysis_cells": False,
        "analysis_based_on_existing_metric_outputs": True,
        "excluded_metric": "ISR",
    },

    "analyses": {
        "metric_integrity":
            EXP004_CONSOLIDATED_LOADED_REPORTS[
                "integrity"
            ],

        "source_level":
            EXP004_CONSOLIDATED_LOADED_REPORTS[
                "source_comparison"
            ],

        "track_level":
            EXP004_CONSOLIDATED_LOADED_REPORTS[
                "track_aggregate"
            ],

        "distribution_outlier":
            EXP004_CONSOLIDATED_LOADED_REPORTS[
                "distribution"
            ],

        "descriptive_statistics":
            EXP004_CONSOLIDATED_LOADED_REPORTS[
                "descriptive"
            ],
    },

    "input_reports": {
        _name: str(_path)
        for _name, _path in (
            EXP004_CONSOLIDATED_REPORT_PATHS.items()
        )
    },

    "notes": [
        "Consolidated from existing EXP004 objective-evaluation reports.",
        "No WAV files were loaded.",
        "No objective metrics were recomputed.",
        "ISR remains excluded from approved analysis.",
        "Cell 15 persisted the objective metrics before downstream analysis.",
    ],
}


with EXP004_CONSOLIDATED_REPORT.open(
    "w",
    encoding="utf-8",
) as _f:

    json.dump(
        EXP004_CONSOLIDATED_REPORT_DATA,
        _f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================================
# 5. VALIDATE CONSOLIDATED REPORT
# ============================================================================

print("\n" + "-" * 70)
print("FINAL OUTPUT VALIDATION")
print("-" * 70)

if not EXP004_CONSOLIDATED_REPORT.exists():
    raise RuntimeError(
        "EXP004 consolidated report was not created."
    )

with EXP004_CONSOLIDATED_REPORT.open(
    "r",
    encoding="utf-8",
) as _f:

    _validation = json.load(
        _f
    )

if (
    _validation.get("experiment")
    != EXP004_ANALYSIS_EXPERIMENT
):
    raise ValueError(
        "Consolidated report experiment identifier "
        "is incorrect."
    )

if (
    _validation.get(
        "configuration",
        {},
    ).get("approved_metrics")
    != EXP004_ANALYSIS_APPROVED_METRICS
):
    raise ValueError(
        "Consolidated report approved metrics are incorrect."
    )

if (
    _validation.get(
        "configuration",
        {},
    ).get("excluded_metrics")
    != EXP004_ANALYSIS_EXCLUDED_METRICS
):
    raise ValueError(
        "Consolidated report excluded metrics are incorrect."
    )

print("✓ Consolidated report validated")
print("✓ Experiment identifier validated")
print("✓ Approved metrics validated")
print("✓ Excluded metrics validated")


# ============================================================================
# 6. RESULT
# ============================================================================

print("\n" + "=" * 70)
print("CELL 21 RESULT")
print("=" * 70)

print(
    f"\nTracks analysed:       "
    f"{EXP004_ANALYSIS_TRACKS}"
)

print(
    f"Source evaluations:   "
    f"{EXP004_ANALYSIS_SOURCE_EVALUATIONS}"
)

print(
    f"Approved metrics:     "
    f"{len(EXP004_ANALYSIS_APPROVED_METRICS)}"
)

print("\nApproved metrics:")

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:
    print(f"  ✓ {_metric}")

print("\nExcluded:")
print("  ✗ ISR")

print("\nConsolidated analyses:")
print("  ✓ Metric integrity audit")
print("  ✓ Overall source-level statistics")
print("  ✓ Track-average statistics")
print("  ✓ Source performance summary")
print("  ✓ Source rankings")
print("  ✓ Best/worst source analysis")
print("  ✓ Outlier analysis")
print("  ✓ Existing track-level aggregate results")

print("\nConsolidated report:")
print(
    f"  {EXP004_CONSOLIDATED_REPORT}"
)

print("\n" + "=" * 70)
print("✓ CELL 21 COMPLETE")
print("=" * 70)

print(
    "\nEXP004 consolidated objective analysis was generated "
    "from the existing evaluation reports."
)

print(
    "No WAV files were loaded and no objective metrics were recomputed."
)

EXP004 CONSOLIDATED OBJECTIVE ANALYSIS

----------------------------------------------------------------------
INPUT VALIDATION
----------------------------------------------------------------------
✓ Integrity: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\reports\EXP004_metric_integrity_audit.json
✓ Descriptive: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\reports\EXP004_metric_descriptive_statistics.json
✓ Distribution: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\reports\EXP004_distribution_outlier_analysis.json
✓ Source Comparison: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\reports\EXP004_source_level_comparative_analysis.json
✓ Track Aggregate: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluati

# EXP004 Final Report Consistency & Quality-Control Audit

In [ ]:
# ============================================================================
# CELL 22 — EXP004 FINAL REPORT CONSISTENCY & QUALITY-CONTROL AUDIT
# ============================================================================

import json
import math
from datetime import datetime


print("=" * 70)
print("EXP004 FINAL REPORT CONSISTENCY & QUALITY-CONTROL AUDIT")
print("=" * 70)


if "EXP004_ANALYSIS_PATHS" not in globals():
    raise RuntimeError(
        "EXP004 analysis configuration is unavailable. "
        "Run Cell 16 first."
    )


# ============================================================================
# 1. EXPECTED REPORTS
# ============================================================================

EXP004_QC_REPORT_PATHS = {
    "integrity":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_metric_integrity_audit.json",

    "descriptive":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_metric_descriptive_statistics.json",

    "distribution":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_distribution_outlier_analysis.json",

    "source_comparison":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_source_level_comparative_analysis.json",

    "track_aggregate":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_track_level_aggregate_analysis.json",

    "consolidated":
        EXP004_ANALYSIS_REPORTS_DIR
        / "EXP004_consolidated_objective_analysis.json",
}


# ============================================================================
# 2. LOAD REPORTS
# ============================================================================

print("\n" + "-" * 70)
print("REPORT LOADING")
print("-" * 70)

EXP004_QC_REPORTS = {}

for _name, _path in EXP004_QC_REPORT_PATHS.items():

    if not _path.exists():
        raise FileNotFoundError(
            f"Required EXP004 report is missing:\n{_path}"
        )

    with _path.open(
        "r",
        encoding="utf-8",
    ) as _f:

        EXP004_QC_REPORTS[
            _name
        ] = json.load(_f)

    print(
        f"✓ Loaded: {_path.name}"
    )


# ============================================================================
# 3. QC STATE
# ============================================================================

EXP004_QC_CHECKS = []
EXP004_QC_ERRORS = []
EXP004_QC_WARNINGS = []


def EXP004_qc_check(
    _name,
    _condition,
    _detail="",
):

    if _condition:

        EXP004_QC_CHECKS.append(
            _name
        )

        print(
            f"✓ {_name}"
        )

    else:

        EXP004_QC_ERRORS.append(
            _name
        )

        print(
            f"✗ {_name}"
        )

        if _detail:
            print(
                f"  {_detail}"
            )


def EXP004_qc_warning(
    _name,
    _detail="",
):

    EXP004_QC_WARNINGS.append(
        _name
    )

    print(
        f"⚠ {_name}"
    )

    if _detail:
        print(
            f"  {_detail}"
        )


# ============================================================================
# 4. EXPERIMENT IDENTIFICATION
# ============================================================================

print("\n" + "-" * 70)
print("EXPERIMENT IDENTIFICATION")
print("-" * 70)

for _name, _report in (
    EXP004_QC_REPORTS.items()
):

    if "experiment" in _report:

        EXP004_qc_check(
            f"{_name} report identifies EXP004",
            _report["experiment"]
            == EXP004_ANALYSIS_EXPERIMENT,
            f"Found: {_report['experiment']}",
        )


# ============================================================================
# 5. METRIC CONFIGURATION
# ============================================================================

print("\n" + "-" * 70)
print("METRIC CONFIGURATION")
print("-" * 70)

EXP004_QC_DESCRIPTIVE = (
    EXP004_QC_REPORTS[
        "descriptive"
    ]
)

EXP004_QC_DISTRIBUTION = (
    EXP004_QC_REPORTS[
        "distribution"
    ]
)

EXP004_QC_SOURCE_COMPARISON = (
    EXP004_QC_REPORTS[
        "source_comparison"
    ]
)

EXP004_QC_TRACK_AGGREGATE = (
    EXP004_QC_REPORTS[
        "track_aggregate"
    ]
)

EXP004_QC_CONSOLIDATED = (
    EXP004_QC_REPORTS[
        "consolidated"
    ]
)

EXP004_qc_check(
    "Approved metrics match expected set",
    EXP004_QC_DESCRIPTIVE.get(
        "approved_metrics"
    )
    == EXP004_ANALYSIS_APPROVED_METRICS,
    (
        f"Found: "
        f"{EXP004_QC_DESCRIPTIVE.get('approved_metrics')}"
    ),
)

EXP004_qc_check(
    "Excluded metrics match expected set",
    EXP004_QC_DESCRIPTIVE.get(
        "excluded_metrics"
    )
    == EXP004_ANALYSIS_EXCLUDED_METRICS,
    (
        f"Found: "
        f"{EXP004_QC_DESCRIPTIVE.get('excluded_metrics')}"
    ),
)


# ============================================================================
# 6. DATASET COUNT CONSISTENCY
# ============================================================================

print("\n" + "-" * 70)
print("DATASET COUNT CONSISTENCY")
print("-" * 70)

EXP004_QC_DATASET = (
    EXP004_QC_DESCRIPTIVE.get(
        "dataset",
        {},
    )
)

EXP004_qc_check(
    "Descriptive report track count = 50",
    EXP004_QC_DATASET.get("tracks")
    == EXP004_ANALYSIS_TRACKS,
    f"Found: {EXP004_QC_DATASET.get('tracks')}",
)

EXP004_qc_check(
    "Descriptive report sources per track = 4",
    EXP004_QC_DATASET.get("sources_per_track")
    == len(EXP004_ANALYSIS_SOURCES),
    (
        "Found: "
        f"{EXP004_QC_DATASET.get('sources_per_track')}"
    ),
)

EXP004_qc_check(
    "Descriptive report source evaluations = 200",
    EXP004_QC_DATASET.get("source_evaluations")
    == EXP004_ANALYSIS_SOURCE_EVALUATIONS,
    (
        "Found: "
        f"{EXP004_QC_DATASET.get('source_evaluations')}"
    ),
)


# ============================================================================
# 7. SOURCE COVERAGE
# ============================================================================

print("\n" + "-" * 70)
print("SOURCE COVERAGE CONSISTENCY")
print("-" * 70)

EXP004_QC_SOURCE_STATS = (
    EXP004_QC_DESCRIPTIVE.get(
        "source_level_statistics",
        {},
    )
)

EXP004_qc_check(
    "Source set matches expected four sources",
    set(
        EXP004_QC_SOURCE_STATS.keys()
    )
    == set(EXP004_ANALYSIS_SOURCES),
    (
        f"Found: "
        f"{list(EXP004_QC_SOURCE_STATS.keys())}"
    ),
)

for _source in EXP004_ANALYSIS_SOURCES:

    EXP004_qc_check(
        f"{_source} source statistics present",
        _source in EXP004_QC_SOURCE_STATS,
    )

    if _source in EXP004_QC_SOURCE_STATS:

        for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

            EXP004_qc_check(
                f"{_source} contains {_metric}",
                _metric
                in EXP004_QC_SOURCE_STATS[
                    _source
                ],
            )


# ============================================================================
# 8. OVERALL SOURCE-LEVEL STATISTICS
# ============================================================================

print("\n" + "-" * 70)
print("OVERALL SOURCE-LEVEL STATISTICS")
print("-" * 70)

EXP004_QC_OVERALL_STATS = (
    EXP004_QC_DESCRIPTIVE.get(
        "overall_source_level_statistics",
        {},
    )
)

EXP004_qc_check(
    "All approved metrics present in overall statistics",
    set(
        EXP004_QC_OVERALL_STATS.keys()
    )
    == set(EXP004_ANALYSIS_APPROVED_METRICS),
    (
        f"Found: "
        f"{list(EXP004_QC_OVERALL_STATS.keys())}"
    ),
)

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    if _metric not in EXP004_QC_OVERALL_STATS:
        continue

    _stats = (
        EXP004_QC_OVERALL_STATS[
            _metric
        ]
    )

    EXP004_qc_check(
        f"{_metric} source-level count = 200",
        _stats.get("count")
        == EXP004_ANALYSIS_SOURCE_EVALUATIONS,
        f"Found: {_stats.get('count')}",
    )


# ============================================================================
# 9. TRACK-AVERAGE STATISTICS
# ============================================================================

print("\n" + "-" * 70)
print("TRACK-AVERAGE STATISTICS")
print("-" * 70)

EXP004_QC_TRACK_STATS = (
    EXP004_QC_DESCRIPTIVE.get(
        "track_average_statistics",
        {},
    )
)

EXP004_qc_check(
    "All approved metrics present in track-average statistics",
    set(
        EXP004_QC_TRACK_STATS.keys()
    )
    == set(EXP004_ANALYSIS_APPROVED_METRICS),
    (
        f"Found: "
        f"{list(EXP004_QC_TRACK_STATS.keys())}"
    ),
)

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    if _metric not in EXP004_QC_TRACK_STATS:
        continue

    _stats = (
        EXP004_QC_TRACK_STATS[
            _metric
        ]
    )

    EXP004_qc_check(
        f"{_metric} track-average count = 50",
        _stats.get("count")
        == EXP004_ANALYSIS_TRACKS,
        f"Found: {_stats.get('count')}",
    )


# ============================================================================
# 10. SOURCE COMPARISON CONSISTENCY
# ============================================================================

print("\n" + "-" * 70)
print("SOURCE COMPARISON CONSISTENCY")
print("-" * 70)

EXP004_QC_SOURCE_MEANS = (
    EXP004_QC_SOURCE_COMPARISON.get(
        "source_means_dB",
        {},
    )
)

for _source in EXP004_ANALYSIS_SOURCES:

    if _source not in EXP004_QC_SOURCE_MEANS:

        EXP004_qc_warning(
            f"{_source} not directly exposed "
            "in source comparison means."
        )

        continue

    for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

        _comparison_value = (
            EXP004_QC_SOURCE_MEANS[
                _source
            ].get(_metric)
        )

        _descriptive_value = (
            EXP004_QC_SOURCE_STATS[
                _source
            ][_metric].get("mean_dB")
        )

        if (
            isinstance(
                _comparison_value,
                (int, float),
            )
            and isinstance(
                _descriptive_value,
                (int, float),
            )
        ):

            EXP004_qc_check(
                f"{_source} {_metric} mean agrees",
                abs(
                    _comparison_value
                    - _descriptive_value
                )
                <= 1e-9,
                (
                    f"Comparison={_comparison_value}, "
                    f"Descriptive={_descriptive_value}"
                ),
            )


# ============================================================================
# 11. TRACK-AVERAGE / SOURCE-LEVEL MEAN CONSISTENCY
# ============================================================================

print("\n" + "-" * 70)
print("TRACK-AVERAGE / SOURCE-LEVEL MEAN CONSISTENCY")
print("-" * 70)

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _source_mean = (
        EXP004_QC_OVERALL_STATS[
            _metric
        ]["mean_dB"]
    )

    _track_mean = (
        EXP004_QC_TRACK_STATS[
            _metric
        ]["mean_dB"]
    )

    EXP004_qc_check(
        f"{_metric}: source-level and "
        "track-average means agree",
        abs(
            _source_mean
            - _track_mean
        )
        <= 1e-9,
        (
            f"Source-level={_source_mean}, "
            f"Track-average={_track_mean}"
        ),
    )


# ============================================================================
# 12. TRACK AGGREGATE VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("TRACK AGGREGATE REPORT CONSISTENCY")
print("-" * 70)

EXP004_QC_TRACK_DATA = (
    EXP004_QC_TRACK_AGGREGATE.get(
        "data_source",
        {},
    )
)

EXP004_qc_check(
    "Track aggregate report track count = 50",
    EXP004_QC_TRACK_DATA.get("tracks")
    == EXP004_ANALYSIS_TRACKS,
    f"Found: {EXP004_QC_TRACK_DATA.get('tracks')}",
)

EXP004_qc_check(
    "Track aggregate source evaluations = 200",
    EXP004_QC_TRACK_DATA.get(
        "source_evaluations"
    )
    == EXP004_ANALYSIS_SOURCE_EVALUATIONS,
    (
        "Found: "
        f"{EXP004_QC_TRACK_DATA.get('source_evaluations')}"
    ),
)

EXP004_QC_TRACK_CONSISTENCY = (
    EXP004_QC_TRACK_AGGREGATE.get(
        "track_average_consistency",
        {},
    )
)

EXP004_qc_check(
    "All stored track averages are consistent",
    EXP004_QC_TRACK_CONSISTENCY.get(
        "all_consistent"
    )
    is True,
    (
        "Found: "
        f"{EXP004_QC_TRACK_CONSISTENCY.get('all_consistent')}"
    ),
)


# ============================================================================
# 13. CONSOLIDATED REPORT VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("CONSOLIDATED REPORT CONSISTENCY")
print("-" * 70)

EXP004_QC_CONSOLIDATED_CONFIG = (
    EXP004_QC_CONSOLIDATED.get(
        "configuration",
        {},
    )
)

EXP004_qc_check(
    "Consolidated experiment = EXP004",
    EXP004_QC_CONSOLIDATED.get(
        "experiment"
    )
    == EXP004_ANALYSIS_EXPERIMENT,
)

EXP004_qc_check(
    "Consolidated track count = 50",
    EXP004_QC_CONSOLIDATED_CONFIG.get(
        "tracks"
    )
    == EXP004_ANALYSIS_TRACKS,
)

EXP004_qc_check(
    "Consolidated source evaluations = 200",
    EXP004_QC_CONSOLIDATED_CONFIG.get(
        "source_evaluations"
    )
    == EXP004_ANALYSIS_SOURCE_EVALUATIONS,
)

EXP004_qc_check(
    "Consolidated approved metrics match",
    EXP004_QC_CONSOLIDATED_CONFIG.get(
        "approved_metrics"
    )
    == EXP004_ANALYSIS_APPROVED_METRICS,
)

EXP004_qc_check(
    "Consolidated excluded metrics match",
    EXP004_QC_CONSOLIDATED_CONFIG.get(
        "excluded_metrics"
    )
    == EXP004_ANALYSIS_EXCLUDED_METRICS,
)


# ============================================================================
# 14. NUMERICAL SANITY CHECKS
# ============================================================================

print("\n" + "-" * 70)
print("NUMERICAL SANITY CHECKS")
print("-" * 70)

for _metric in EXP004_ANALYSIS_APPROVED_METRICS:

    _stats = (
        EXP004_QC_OVERALL_STATS[
            _metric
        ]
    )

    _values = [
        _stats.get("mean_dB"),
        _stats.get("median_dB"),
        _stats.get("std_dB"),
        _stats.get("min_dB"),
        _stats.get("q1_dB"),
        _stats.get("q3_dB"),
        _stats.get("max_dB"),
        _stats.get("iqr_dB"),
    ]

    EXP004_qc_check(
        f"{_metric} statistics are numeric",
        all(
            isinstance(
                _value,
                (int, float),
            )
            for _value in _values
        ),
    )

    if all(
        isinstance(
            _value,
            (int, float),
        )
        for _value in _values
    ):

        EXP004_qc_check(
            f"{_metric}: min <= Q1 <= median <= Q3 <= max",
            (
                _stats["min_dB"]
                <= _stats["q1_dB"]
                <= _stats["median_dB"]
                <= _stats["q3_dB"]
                <= _stats["max_dB"]
            ),
        )

        EXP004_qc_check(
            f"{_metric}: standard deviation >= 0",
            _stats["std_dB"] >= 0,
        )

        EXP004_qc_check(
            f"{_metric}: IQR >= 0",
            _stats["iqr_dB"] >= 0,
        )


# ============================================================================
# 15. EXCLUDED METRIC CHECK
# ============================================================================

print("\n" + "-" * 70)
print("EXCLUDED METRIC CHECK")
print("-" * 70)

for _section_name, _section in [
    (
        "overall source-level statistics",
        EXP004_QC_OVERALL_STATS,
    ),
    (
        "track-average statistics",
        EXP004_QC_TRACK_STATS,
    ),
]:

    EXP004_qc_check(
        f"ISR absent from {_section_name}",
        "ISR" not in _section,
    )


# ============================================================================
# 16. EXACTLY 8 PLOT VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("REQUIRED PLOT VALIDATION")
print("-" * 70)

EXP004_QC_EXPECTED_PLOTS = [
    EXP004_ANALYSIS_PLOTS_DIR
    / "EXP004_sdr_distribution.png",

    EXP004_ANALYSIS_PLOTS_DIR
    / "EXP004_sir_distribution.png",

    EXP004_ANALYSIS_PLOTS_DIR
    / "EXP004_sar_distribution.png",

    EXP004_ANALYSIS_PLOTS_DIR
    / "EXP004_si_sdr_distribution.png",

    EXP004_ANALYSIS_PLOTS_DIR
    / "EXP004_sdr_by_source_boxplot.png",

    EXP004_ANALYSIS_PLOTS_DIR
    / "EXP004_sir_by_source_boxplot.png",

    EXP004_ANALYSIS_PLOTS_DIR
    / "EXP004_sar_by_source_boxplot.png",

    EXP004_ANALYSIS_PLOTS_DIR
    / "EXP004_si_sdr_by_source_boxplot.png",
]

EXP004_qc_check(
    "Exactly 8 required plot paths are defined",
    len(EXP004_QC_EXPECTED_PLOTS) == 8,
)

for _plot_path in EXP004_QC_EXPECTED_PLOTS:

    EXP004_qc_check(
        f"Plot exists: {_plot_path.name}",
        _plot_path.exists(),
    )

    if _plot_path.exists():

        EXP004_qc_check(
            f"Plot non-empty: {_plot_path.name}",
            _plot_path.stat().st_size > 0,
        )


# ============================================================================
# 17. WRITE FINAL QC REPORT
# ============================================================================

EXP004_FINAL_QC_REPORT = (
    EXP004_ANALYSIS_REPORTS_DIR
    / "EXP004_final_report_consistency_audit.json"
)

EXP004_FINAL_QC_REPORT_DATA = {
    "experiment": EXP004_ANALYSIS_EXPERIMENT,
    "audit_type":
        "final_report_consistency_and_quality_control",
    "timestamp":
        datetime.now().isoformat(),

    "configuration": {
        "tracks":
            EXP004_ANALYSIS_TRACKS,

        "sources_per_track":
            len(EXP004_ANALYSIS_SOURCES),

        "source_evaluations":
            EXP004_ANALYSIS_SOURCE_EVALUATIONS,

        "sources":
            EXP004_ANALYSIS_SOURCES,

        "approved_metrics":
            EXP004_ANALYSIS_APPROVED_METRICS,

        "excluded_metrics":
            EXP004_ANALYSIS_EXCLUDED_METRICS,
    },

    "checks": {
        "passed":
            len(EXP004_QC_CHECKS),

        "failed":
            len(EXP004_QC_ERRORS),

        "warnings":
            len(EXP004_QC_WARNINGS),
    },

    "status":
        "PASS"
        if not EXP004_QC_ERRORS
        else "FAIL",

    "reports_audited": {
        _name: str(_path)
        for _name, _path in (
            EXP004_QC_REPORT_PATHS.items()
        )
    },

    "plots_audited": [
        str(_path)
        for _path in EXP004_QC_EXPECTED_PLOTS
    ],

    "notes": [
        "Audit performed using existing EXP004 objective reports.",
        "No WAV files were loaded.",
        "No objective metrics were recomputed.",
        "ISR remains excluded from approved analysis.",
    ],
}


with EXP004_FINAL_QC_REPORT.open(
    "w",
    encoding="utf-8",
) as _f:

    json.dump(
        EXP004_FINAL_QC_REPORT_DATA,
        _f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================================
# 18. FINAL RESULT
# ============================================================================

print("\n" + "=" * 70)
print("CELL 22 RESULT")
print("=" * 70)

print(
    f"\nChecks passed:       "
    f"{len(EXP004_QC_CHECKS)}"
)

print(
    f"Checks failed:       "
    f"{len(EXP004_QC_ERRORS)}"
)

print(
    f"Warnings:            "
    f"{len(EXP004_QC_WARNINGS)}"
)

print("\nCore EXP004 configuration:")
print(
    f"  Tracks:             "
    f"{EXP004_ANALYSIS_TRACKS}"
)

print(
    f"  Sources per track:  "
    f"{len(EXP004_ANALYSIS_SOURCES)}"
)

print(
    f"  Source evaluations: "
    f"{EXP004_ANALYSIS_SOURCE_EVALUATIONS}"
)

print(
    f"  Approved metrics:   "
    f"{', '.join(EXP004_ANALYSIS_APPROVED_METRICS)}"
)

print(
    "  Excluded metric:    ISR"
)

if EXP004_QC_ERRORS:

    print("\n" + "-" * 70)
    print("FAILED CHECKS")
    print("-" * 70)

    for _item in EXP004_QC_ERRORS:
        print(
            f"  ✗ {_item}"
        )

    raise RuntimeError(
        "EXP004 final QC audit failed with "
        f"{len(EXP004_QC_ERRORS)} check(s). "
        "Review the failed checks above."
    )


print(
    "\n✓ All critical EXP004 consistency checks passed."
)

if EXP004_QC_WARNINGS:

    print("\nWarnings:")

    for _item in EXP004_QC_WARNINGS:
        print(
            f"  ⚠ {_item}"
        )

print("\nFinal QC report:")
print(
    f"  {EXP004_FINAL_QC_REPORT}"
)

print("\n" + "=" * 70)
print("✓ CELL 22 COMPLETE")
print("=" * 70)

print(
    "\nEXP004 final report consistency and "
    "quality-control audit completed successfully."
)

EXP004 FINAL REPORT CONSISTENCY & QUALITY-CONTROL AUDIT

----------------------------------------------------------------------
REPORT LOADING
----------------------------------------------------------------------
✓ Loaded: EXP004_metric_integrity_audit.json
✓ Loaded: EXP004_metric_descriptive_statistics.json
✓ Loaded: EXP004_distribution_outlier_analysis.json
✓ Loaded: EXP004_source_level_comparative_analysis.json
✓ Loaded: EXP004_track_level_aggregate_analysis.json
✓ Loaded: EXP004_consolidated_objective_analysis.json

----------------------------------------------------------------------
EXPERIMENT IDENTIFICATION
----------------------------------------------------------------------
✓ integrity report identifies EXP004
✓ descriptive report identifies EXP004
✓ distribution report identifies EXP004
✓ source_comparison report identifies EXP004
✓ track_aggregate report identifies EXP004
✓ consolidated report identifies EXP004

------------------------------------------------------------

# EXP004 Final Package & Completion Manifest

In [ ]:
# ============================================================================
# CELL 23 — EXP004 FINAL PACKAGE & COMPLETION MANIFEST
# ============================================================================

import json
from datetime import datetime


print("=" * 70)
print("EXP004 FINAL PACKAGE & COMPLETION MANIFEST")
print("=" * 70)


if "EXP004_ANALYSIS_PATHS" not in globals():
    raise RuntimeError(
        "EXP004 analysis configuration is unavailable. "
        "Run Cell 16 first."
    )


# ============================================================================
# 1. AUTHORITATIVE DIRECTORIES
# ============================================================================

EXP004_MANIFEST_METRICS_DIR = (
    EXP004_ANALYSIS_METRICS_DIR
)

EXP004_MANIFEST_REPORTS_DIR = (
    EXP004_ANALYSIS_REPORTS_DIR
)

EXP004_MANIFEST_PLOTS_DIR = (
    EXP004_ANALYSIS_PLOTS_DIR
)


# ============================================================================
# 2. REQUIRED METRIC FILES
# ============================================================================

EXP004_MANIFEST_METRIC_FILES = [
    EXP004_MANIFEST_METRICS_DIR
    / "EXP004_per_source_metrics.csv",

    EXP004_MANIFEST_METRICS_DIR
    / "EXP004_per_track_metrics.jsonl",
]


# ============================================================================
# 3. REQUIRED REPORT FILES
# ============================================================================

EXP004_MANIFEST_REPORT_FILES = [
    EXP004_MANIFEST_REPORTS_DIR
    / "EXP004_metric_integrity_audit.json",

    EXP004_MANIFEST_REPORTS_DIR
    / "EXP004_metric_descriptive_statistics.json",

    EXP004_MANIFEST_REPORTS_DIR
    / "EXP004_distribution_outlier_analysis.json",

    EXP004_MANIFEST_REPORTS_DIR
    / "EXP004_source_level_comparative_analysis.json",

    EXP004_MANIFEST_REPORTS_DIR
    / "EXP004_track_level_aggregate_analysis.json",

    EXP004_MANIFEST_REPORTS_DIR
    / "EXP004_consolidated_objective_analysis.json",

    EXP004_MANIFEST_REPORTS_DIR
    / "EXP004_final_report_consistency_audit.json",
]


# ============================================================================
# 4. EXACTLY EIGHT REQUIRED PLOTS
# ============================================================================

EXP004_MANIFEST_PLOT_FILES = [
    EXP004_MANIFEST_PLOTS_DIR
    / "EXP004_sdr_distribution.png",

    EXP004_MANIFEST_PLOTS_DIR
    / "EXP004_sir_distribution.png",

    EXP004_MANIFEST_PLOTS_DIR
    / "EXP004_sar_distribution.png",

    EXP004_MANIFEST_PLOTS_DIR
    / "EXP004_si_sdr_distribution.png",

    EXP004_MANIFEST_PLOTS_DIR
    / "EXP004_sdr_by_source_boxplot.png",

    EXP004_MANIFEST_PLOTS_DIR
    / "EXP004_sir_by_source_boxplot.png",

    EXP004_MANIFEST_PLOTS_DIR
    / "EXP004_sar_by_source_boxplot.png",

    EXP004_MANIFEST_PLOTS_DIR
    / "EXP004_si_sdr_by_source_boxplot.png",
]


# ============================================================================
# 5. DIRECTORY VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("DIRECTORY VALIDATION")
print("-" * 70)

for _directory in [
    EXP004_MANIFEST_METRICS_DIR,
    EXP004_MANIFEST_REPORTS_DIR,
    EXP004_MANIFEST_PLOTS_DIR,
]:

    if not _directory.exists():
        raise FileNotFoundError(
            f"Required directory does not exist:\n"
            f"{_directory}"
        )

    print(
        f"✓ {_directory}"
    )


# ============================================================================
# 6. FILE VALIDATION
# ============================================================================

print("\n" + "-" * 70)
print("METRIC FILE VALIDATION")
print("-" * 70)

EXP004_MANIFEST_MISSING_METRICS = []

for _path in EXP004_MANIFEST_METRIC_FILES:

    if _path.exists() and _path.stat().st_size > 0:

        print(
            f"✓ {_path.name}"
        )

    else:

        print(
            f"✗ MISSING: {_path.name}"
        )

        EXP004_MANIFEST_MISSING_METRICS.append(
            str(_path)
        )


print("\n" + "-" * 70)
print("REPORT FILE VALIDATION")
print("-" * 70)

EXP004_MANIFEST_MISSING_REPORTS = []

for _path in EXP004_MANIFEST_REPORT_FILES:

    if _path.exists() and _path.stat().st_size > 0:

        print(
            f"✓ {_path.name}"
        )

    else:

        print(
            f"✗ MISSING: {_path.name}"
        )

        EXP004_MANIFEST_MISSING_REPORTS.append(
            str(_path)
        )


print("\n" + "-" * 70)
print("PLOT FILE VALIDATION")
print("-" * 70)

EXP004_MANIFEST_MISSING_PLOTS = []

for _path in EXP004_MANIFEST_PLOT_FILES:

    if _path.exists() and _path.stat().st_size > 0:

        print(
            f"✓ {_path.name}"
        )

    else:

        print(
            f"✗ MISSING: {_path.name}"
        )

        EXP004_MANIFEST_MISSING_PLOTS.append(
            str(_path)
        )


# ============================================================================
# 7. LOAD FINAL QC REPORT
# ============================================================================

EXP004_MANIFEST_QC_PATH = (
    EXP004_MANIFEST_REPORTS_DIR
    / "EXP004_final_report_consistency_audit.json"
)

if not EXP004_MANIFEST_QC_PATH.exists():
    raise FileNotFoundError(
        "Final EXP004 QC report was not found:\n"
        f"{EXP004_MANIFEST_QC_PATH}"
    )

with EXP004_MANIFEST_QC_PATH.open(
    "r",
    encoding="utf-8",
) as _f:

    EXP004_MANIFEST_QC_REPORT = json.load(
        _f
    )


# ============================================================================
# 8. QC INFORMATION
# ============================================================================

EXP004_MANIFEST_QC_CHECKS = (
    EXP004_MANIFEST_QC_REPORT
    .get("checks", {})
)

EXP004_MANIFEST_CHECKS_PASSED = (
    EXP004_MANIFEST_QC_CHECKS
    .get("passed")
)

EXP004_MANIFEST_CHECKS_FAILED = (
    EXP004_MANIFEST_QC_CHECKS
    .get("failed")
)

EXP004_MANIFEST_WARNINGS = (
    EXP004_MANIFEST_QC_CHECKS
    .get("warnings")
)

EXP004_MANIFEST_QC_STATUS = (
    EXP004_MANIFEST_QC_REPORT
    .get("status")
)


# ============================================================================
# 9. FINAL CONFIGURATION
# ============================================================================

EXP004_MANIFEST_APPROVED_METRICS = (
    EXP004_ANALYSIS_APPROVED_METRICS
)

EXP004_MANIFEST_EXCLUDED_METRICS = (
    EXP004_ANALYSIS_EXCLUDED_METRICS
)

EXP004_MANIFEST_SOURCES = (
    EXP004_ANALYSIS_SOURCES
)


# ============================================================================
# 10. COMPLETION STATUS
# ============================================================================

EXP004_MANIFEST_ALL_FILES_PRESENT = (
    len(
        EXP004_MANIFEST_MISSING_METRICS
    ) == 0
    and len(
        EXP004_MANIFEST_MISSING_REPORTS
    ) == 0
    and len(
        EXP004_MANIFEST_MISSING_PLOTS
    ) == 0
)

EXP004_MANIFEST_QC_PASSED = (
    EXP004_MANIFEST_CHECKS_FAILED == 0
    and EXP004_MANIFEST_QC_STATUS == "PASS"
)

EXP004_MANIFEST_COMPLETE = (
    EXP004_MANIFEST_ALL_FILES_PRESENT
    and EXP004_MANIFEST_QC_PASSED
)


# ============================================================================
# 11. COMPLETION MANIFEST
# ============================================================================

EXP004_COMPLETION_MANIFEST = (
    EXP004_MANIFEST_REPORTS_DIR
    / "EXP004_completion_manifest.json"
)

EXP004_COMPLETION_MANIFEST_DATA = {
    "experiment": EXP004_ANALYSIS_EXPERIMENT,
    "experiment_name":
        "Objective Evaluation Analysis",
    "generated_at":
        datetime.now().isoformat(),

    "status":
        "COMPLETE"
        if EXP004_MANIFEST_COMPLETE
        else "INCOMPLETE",

    "dataset": {
        "tracks":
            EXP004_ANALYSIS_TRACKS,

        "sources_per_track":
            len(EXP004_MANIFEST_SOURCES),

        "source_evaluations":
            EXP004_ANALYSIS_SOURCE_EVALUATIONS,

        "sources":
            EXP004_MANIFEST_SOURCES,
    },

    "metrics": {
        "approved":
            EXP004_MANIFEST_APPROVED_METRICS,

        "excluded":
            EXP004_MANIFEST_EXCLUDED_METRICS,
    },

    "analysis_stages_completed": [
        "Metric integrity audit",
        "Descriptive statistics",
        "Source-level comparative analysis",
        "Track-level aggregate analysis",
        "Distribution analysis",
        "IQR-based outlier analysis",
        "Eight required plots",
        "Consolidated objective analysis",
        "Final report consistency and QC audit",
    ],

    "objective_evaluation_policy": {
        "metric_recomputation_in_analysis_cells":
            False,

        "wav_files_loaded_in_analysis_cells":
            False,

        "analysis_based_on_existing_metric_outputs":
            True,

        "excluded_metric":
            "ISR",
    },

    "quality_control": {
        "checks_passed":
            EXP004_MANIFEST_CHECKS_PASSED,

        "checks_failed":
            EXP004_MANIFEST_CHECKS_FAILED,

        "warnings":
            EXP004_MANIFEST_WARNINGS,

        "qc_status":
            (
                "PASSED"
                if EXP004_MANIFEST_QC_PASSED
                else "FAILED"
            ),
    },

    "files": {
        "metric_files": [
            str(_path)
            for _path in EXP004_MANIFEST_METRIC_FILES
            if _path.exists()
        ],

        "report_files": [
            str(_path)
            for _path in EXP004_MANIFEST_REPORT_FILES
            if _path.exists()
        ],

        "plot_files": [
            str(_path)
            for _path in EXP004_MANIFEST_PLOT_FILES
            if _path.exists()
        ],
    },

    "missing_files": {
        "metrics":
            EXP004_MANIFEST_MISSING_METRICS,

        "reports":
            EXP004_MANIFEST_MISSING_REPORTS,

        "plots":
            EXP004_MANIFEST_MISSING_PLOTS,
    },

    "completion_statement": (
        "EXP004 objective evaluation analysis and "
        "quality-control processing completed successfully "
        "using the existing objective metric outputs."
        if EXP004_MANIFEST_COMPLETE
        else
        "EXP004 package validation is incomplete. "
        "One or more required files or QC conditions "
        "are missing."
    ),
}


with EXP004_COMPLETION_MANIFEST.open(
    "w",
    encoding="utf-8",
) as _f:

    json.dump(
        EXP004_COMPLETION_MANIFEST_DATA,
        _f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================================
# 12. FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("CELL 23 RESULT")
print("=" * 70)

print(
    f"\nExperiment status:      "
    f"{EXP004_COMPLETION_MANIFEST_DATA['status']}"
)

print(
    f"Tracks analysed:       "
    f"{EXP004_ANALYSIS_TRACKS}"
)

print(
    f"Source evaluations:    "
    f"{EXP004_ANALYSIS_SOURCE_EVALUATIONS}"
)

print(
    f"Approved metrics:      "
    f"{len(EXP004_MANIFEST_APPROVED_METRICS)}"
)

print(
    "Excluded metric:       ISR"
)

print("\nRequired files:")

print(
    f"  Metric files:         "
    f"{len(EXP004_MANIFEST_METRIC_FILES) - len(EXP004_MANIFEST_MISSING_METRICS)}"
    f"/{len(EXP004_MANIFEST_METRIC_FILES)}"
)

print(
    f"  Report files:         "
    f"{len(EXP004_MANIFEST_REPORT_FILES) - len(EXP004_MANIFEST_MISSING_REPORTS)}"
    f"/{len(EXP004_MANIFEST_REPORT_FILES)}"
)

print(
    f"  Plot files:           "
    f"{len(EXP004_MANIFEST_PLOT_FILES) - len(EXP004_MANIFEST_MISSING_PLOTS)}"
    f"/{len(EXP004_MANIFEST_PLOT_FILES)}"
)

print("\nQuality control:")

if EXP004_MANIFEST_CHECKS_PASSED is not None:
    print(
        f"  Checks passed:        "
        f"{EXP004_MANIFEST_CHECKS_PASSED}"
    )

if EXP004_MANIFEST_CHECKS_FAILED is not None:
    print(
        f"  Checks failed:        "
        f"{EXP004_MANIFEST_CHECKS_FAILED}"
    )

if EXP004_MANIFEST_WARNINGS is not None:
    print(
        f"  Warnings:             "
        f"{EXP004_MANIFEST_WARNINGS}"
    )

print(
    f"  QC status:            "
    f"{EXP004_COMPLETION_MANIFEST_DATA['quality_control']['qc_status']}"
)

print("\nAnalysis stages:")

for _stage in (
    EXP004_COMPLETION_MANIFEST_DATA[
        "analysis_stages_completed"
    ]
):

    print(
        f"  ✓ {_stage}"
    )

print("\nFinal package:")

print(
    f"  ✓ Metrics:            "
    f"{EXP004_MANIFEST_METRICS_DIR}"
)

print(
    f"  ✓ Reports:            "
    f"{EXP004_MANIFEST_REPORTS_DIR}"
)

print(
    f"  ✓ Plots:              "
    f"{EXP004_MANIFEST_PLOTS_DIR}"
)

print("\nCompletion manifest:")

print(
    f"  {EXP004_COMPLETION_MANIFEST}"
)

print("\n" + "=" * 70)

if EXP004_MANIFEST_COMPLETE:

    print("✓ EXP004 COMPLETE")

    print("=" * 70)

    print(
        "All required EXP004 metric outputs, "
        "reports, plots, and QC artifacts are present."
    )

    print(
        "No WAV files were loaded and no objective "
        "metrics were recomputed during the analysis stages."
    )

else:

    print("✗ EXP004 INCOMPLETE")

    print("=" * 70)

    if EXP004_MANIFEST_MISSING_METRICS:

        print("\nMissing metric files:")

        for _item in (
            EXP004_MANIFEST_MISSING_METRICS
        ):
            print(
                f"  - {_item}"
            )

    if EXP004_MANIFEST_MISSING_REPORTS:

        print("\nMissing report files:")

        for _item in (
            EXP004_MANIFEST_MISSING_REPORTS
        ):
            print(
                f"  - {_item}"
            )

    if EXP004_MANIFEST_MISSING_PLOTS:

        print("\nMissing plot files:")

        for _item in (
            EXP004_MANIFEST_MISSING_PLOTS
        ):
            print(
                f"  - {_item}"
            )

print("=" * 70)

EXP004 FINAL PACKAGE & COMPLETION MANIFEST

----------------------------------------------------------------------
DIRECTORY VALIDATION
----------------------------------------------------------------------
✓ G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\metrics
✓ G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\reports
✓ G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP004\plots

----------------------------------------------------------------------
METRIC FILE VALIDATION
----------------------------------------------------------------------
✓ EXP004_per_source_metrics.csv
✓ EXP004_per_track_metrics.jsonl

----------------------------------------------------------------------
REPORT FILE VALIDATION
----------------------------------------------------------------------
✓ EXP004_metric_integrity_audit.json
✓ EXP00